In [ ]:
# Instala todas las librerías que usa el notebook (en Kaggle casi todas ya vienen; pip saltea las que están)
%pip install -q numpy pandas matplotlib pillow torch torchvision transformers pycocotools jsonschema kagglehub

# TPI IA 2026 – Grupo 5

| Versión | Cambios | PipelineScore (público) |
|---|---|---|
| v1 | Fases 1 a 4 con cajas en `[x, y, w, h]` | rechazada por formato |
| v2 | Cajas en `[xmin, ymin, xmax, ymax]` | 0,6229 |
| v3 | Atributos y proximidad (3.6), preguntas en modo determinista (Fase 5), validación de filas antes del envío | 0,6656 |
| v4 | Experimentos de detector (2.9: OWLv2-large, sinónimos, ventanas) y de SAM2 (3.3b), modo LLM con Qwen2.5-3B (Fase 6) | 0,7022 |
| v5 | Ensamble base + large (E8), vocabulario bilingüe para el intérprete, ejemplos resueltos en la redacción y control de conteos en la descripción | 0,7028 |
| v6 | Traza completa (RUN_ID, hash de configuración, linaje por objeto, manifiesto), alias y reintento por vocabulario en el intérprete, validación cruzada (Fase 7), propagación de errores (Fase 8) y resumen final (Fase 9) | 0,7049 |

## 0. Identificación de la corrida (agregado en la v6)

Cada ejecución recibe un **`RUN_ID`** que aparece en todas las trazas y en cada SceneJSON del envío (campo `trace`). Acá también se fijan las semillas y se registran las versiones del entorno. Al final (Fase 9), `manifiesto_corrida.json` junta el `RUN_ID`, la configuración completa con su hash (`config_hash`), los tiempos de cada fase y la lista de artefactos generados con su tamaño y su SHA-256. Con eso se puede reconstruir qué corrida produjo cada archivo y con qué parámetros.

In [ ]:
import hashlib
import json
import platform
import random
import time
import uuid

import numpy as np
import torch
import transformers

SEMILLA = 0
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
T_INICIO = time.time()
RUN_ID = time.strftime("%Y%m%d-%H%M%S", time.gmtime()) + "-" + uuid.uuid4().hex[:6]
TIEMPOS_FASE = {}            # fase -> segundos desde el inicio (se completa a lo largo de la notebook)

def huella(obj):
    """Hash corto y estable de una configuración: SHA-256 del JSON con las claves ordenadas."""
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()[:12]

def marcar_fase(nombre):
    TIEMPOS_FASE[nombre] = round(time.time() - T_INICIO, 1)

VERSIONES = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
                 numpy=np.__version__, cuda=torch.version.cuda,
                 gpu=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("RUN_ID:", RUN_ID, "| semilla:", SEMILLA)
print(json.dumps(VERSIONES, ensure_ascii=False))

## 1. Análisis exploratorio (EDA)

### 1.0 Inspección del formato de los datos
Antes de programar el EDA verificamos cómo vienen las anotaciones, el envío, las preguntas y el vocabulario, para no asumir formatos.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

# Ubicar la raíz sin adivinar la ruta de montaje (en Kaggle: /kaggle/input; en una PC: el caché de kagglehub)
BASES = [Path("/kaggle/input"), Path.home() / ".cache" / "kagglehub" / "competitions"]
ROOT = next(p.parent for b in BASES if b.exists() for p in b.rglob("scene_schema.json"))
DEV, TEST, QA = ROOT/"development", ROOT/"tpi_ia_2026_competition_public", ROOT/"tpi_2026_qa_public"

# 1) Anotaciones: formato de máscaras y cajas
ann = json.load(open(DEV/"annotations.json"))
a0 = ann["annotations"][0]
print("Imágenes / anotaciones / categorías:", len(ann["images"]), len(ann["annotations"]), len(ann["categories"]))
print("Claves de imagen:", list(ann["images"][0].keys()))
print("Claves de anotación:", list(a0.keys()))
print("bbox:", a0["bbox"], "| area:", a0["area"])
print("Tipos de máscara:", Counter(type(a["segmentation"]).__name__ for a in ann["annotations"]))

# 2) Envío: cómo es una fila de desarrollo completa
sub = pd.read_csv(ROOT/"sample_submission.csv")
dev = sub[~sub.image_id.str.startswith("test_")]
print("\nIDs de desarrollo:", dev.image_id.head(3).tolist())
print(json.dumps(json.loads(dev.iloc[0].SceneJSON), ensure_ascii=False)[:2000])

# 3) Preguntas: cantidad por tipo y un ejemplo de cada una
qs = pd.read_json(QA/"questions_dev.jsonl", lines=True)
print("\n", qs.type.value_counts())
print(qs.groupby("type").first()[["question", "answer_format", "class_en"]].to_string())

# 4) Vocabulario del test vs. categorías anotadas
cats = pd.read_csv(TEST/"categories.csv")
print("\ncategories.csv:", cats.shape, list(cats.columns)); print(cats.head(3))

### 1.1 Carga de datos
Armamos dos tablas: una fila por imagen y una fila por objeto anotado. Las anotaciones se usan sólo para EDA y evaluación, nunca dentro de la pipeline de predicción.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Rectangle, Polygon

id2name = dict(zip(cats.category_id, cats.category_name))

# Estilo común: un solo color de datos, gris para referencias, grilla suave
AZUL, GRIS = "#3b6ea8", "#8a9099"
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.titleweight": "bold"})

# Una fila por imagen
imgs = pd.DataFrame(ann["images"])
imgs["fname"] = imgs.file_name.map(lambda s: Path(s).name)

# Una fila por objeto (sin el polígono, que es pesado)
objs = pd.DataFrame(ann["annotations"]).drop(columns="segmentation")
objs[["x", "y", "w", "h"]] = pd.DataFrame(objs.bbox.tolist(), index=objs.index)
objs = objs.merge(imgs[["id", "width", "height"]].rename(columns={"id": "image_id"}), on="image_id")
objs["category_name"] = objs.category_id.map(id2name)
objs["area_ratio"] = objs.area / (objs.width * objs.height)
objs["cx"], objs["cy"] = objs.x + objs.w / 2, objs.y + objs.h / 2   # centro de la caja (aprox. del centroide)

# Chequeo de consistencia del vocabulario
ann_cats = {c["id"]: c["name"] for c in ann["categories"]}
print("Objetos:", len(objs))
print("¿Mismos IDs en categories.csv y annotations.json?", set(id2name) == set(ann_cats))
print("¿Mismos nombres?", all(id2name[k] == ann_cats[k] for k in id2name))
print("Ejemplo de file_name:", imgs.file_name.iloc[0])

### 1.2 ¿Cuántos objetos hay por imagen?
Motiva: el número máximo de detecciones por imagen y qué tan bajo puede ser el umbral de confianza.

In [ ]:
n_obj = objs.groupby("image_id").size().reindex(imgs.id, fill_value=0)
n_cls = objs.groupby("image_id").category_id.nunique().reindex(imgs.id, fill_value=0)
print(pd.DataFrame({"objetos": n_obj.describe(), "clases distintas": n_cls.describe()}).round(1))
print("Imagen más cargada:", n_obj.max(), "objetos | límite del schema: 300")

ax = n_obj.plot.hist(bins=30, color=AZUL, edgecolor="white")
ax.axvline(n_obj.median(), color=GRIS, ls="--", label=f"mediana = {n_obj.median():.0f}")
ax.set(title="¿Cuántos objetos anotados hay por imagen?", xlabel="Objetos por imagen", ylabel="Imágenes")
ax.legend(); plt.show()

### 1.3 ¿Qué tan chicos son los objetos?
Motiva: la elección del detector (YOLO-World rinde peor con objetos chicos) y la resolución de entrada.
Umbrales de referencia COCO: chico < 32² px, grande ≥ 96² px.

In [ ]:
chico = (objs.area < 32**2).mean()
grande = (objs.area >= 96**2).mean()
print(f"Chicos (<32² px): {chico:.0%} | Medianos: {1 - chico - grande:.0%} | Grandes (≥96² px): {grande:.0%}")
print("Percentiles de area_ratio:", objs.area_ratio.quantile([.1, .25, .5, .75, .9]).round(4).to_dict())

ax = np.log10(objs.area_ratio).plot.hist(bins=40, color=AZUL, edgecolor="white")
for v, t in [(-3, "0,1 %"), (-2, "1 %"), (-1, "10 %")]:
    ax.axvline(v, color=GRIS, ls=":")
    ax.text(v, ax.get_ylim()[1] * 0.93, t, ha="center", color=GRIS, backgroundcolor="white")
ax.set(title="¿Qué tan chicos son los objetos?", xlabel="log10(área del objeto / área de la imagen)", ylabel="Objetos")
plt.show()

### 1.4 ¿Qué clases aparecen mucho y cuáles poco?
Motiva: cómo armar el vocabulario y qué clases van a pesar más en el F1.

In [ ]:
freq = objs.category_name.value_counts()
imgs_por_clase = objs.groupby("category_name").image_id.nunique()
print(f"Clases con al menos una anotación: {freq.size} de {len(cats)}")
print(f"Clases con ≤ 3 instancias: {(freq <= 3).sum()}")
print(f"Clases presentes en una sola imagen: {(imgs_por_clase == 1).sum()}")
print(f"Las 10 clases más frecuentes concentran el {freq.head(10).sum() / freq.sum():.0%} de los objetos")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
freq.head(15)[::-1].plot.barh(ax=axes[0], color=AZUL)
axes[0].set(title="Las 15 clases más frecuentes", xlabel="Instancias")
freq.tail(15)[::-1].plot.barh(ax=axes[1], color=GRIS)
axes[1].set(title="Las 15 menos frecuentes", xlabel="Instancias")
plt.tight_layout(); plt.show()

### 1.5 ¿Qué parte de las predicciones se puede evaluar? (LVIS)
Un par imagen-clase sólo se evalúa si es positivo (tiene anotaciones) o negativo verificado (`neg_category_ids`).
Motiva: la métrica local tiene que ignorar los pares no evaluables, y hay que reportar la cobertura.

In [ ]:
pos = objs.groupby("image_id").category_id.apply(set).to_dict()
n_pos = sum(len(s) for s in pos.values())
n_neg = imgs.neg_category_ids.str.len().sum()
total = len(imgs) * len(cats)

print(f"Pares imagen-clase posibles: {total}")
print(f"  Positivos:             {n_pos:5d} ({n_pos / total:.1%})")
print(f"  Negativos verificados: {n_neg:5d} ({n_neg / total:.1%})")
print(f"  No evaluables:         {total - n_pos - n_neg:5d} ({1 - (n_pos + n_neg) / total:.1%})")

no_exh = sum(len(set(r.not_exhaustive_category_ids) & pos.get(r.id, set())) for r in imgs.itertuples())
print(f"Positivos marcados como no exhaustivos: {no_exh} ({no_exh / n_pos:.0%} de los positivos)")
solap = sum(len(set(r.neg_category_ids) & pos.get(r.id, set())) for r in imgs.itertuples())
print(f"Chequeo: clases a la vez positivas y negativas en una imagen: {solap} (debería ser 0)")

### 1.6 ¿Cuánto se solapan objetos de la misma clase?
Motiva: el umbral de NMS. Si objetos reales de la misma clase se solapan mucho, un NMS agresivo los borraría.

In [ ]:
def iou_xywh(a, b):
    """IoU entre dos cajas en formato [x, y, ancho, alto]."""
    iw = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)

ious = []
for _, g in objs.groupby(["image_id", "category_id"]):
    b = g.bbox.tolist()
    ious += [iou_xywh(b[i], b[j]) for i in range(len(b)) for j in range(i + 1, len(b))]
ious = np.array(ious)

print(f"Pares de la misma clase dentro de una imagen: {len(ious)}")
for t in (0.3, 0.5, 0.7):
    print(f"  con IoU > {t}: {(ious > t).sum()} ({(ious > t).mean():.1%})")

ax = pd.Series(ious).plot.hist(bins=30, color=AZUL, edgecolor="white")
ax.set(title="¿Cuánto se solapan objetos reales de la misma clase?", xlabel="IoU entre cajas", ylabel="Pares")
plt.show()

### 1.7 ¿Dónde caen los objetos en la grilla 3×3?
Motiva: valida la regla de posición del SceneJSON y muestra si hay sesgo espacial.

In [ ]:
col = np.minimum((objs.cx / (objs.width / 3)).astype(int), 2)
fila = np.minimum((objs.cy / (objs.height / 3)).astype(int), 2)
grid = pd.crosstab(fila, col).reindex(index=range(3), columns=range(3), fill_value=0)
print("Proporción por zona:")
print((grid / grid.values.sum()).round(2))

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(grid, cmap="Blues"); ax.grid(False)
for r in range(3):
    for c in range(3):
        v = grid.iloc[r, c]
        ax.text(c, r, v, ha="center", va="center", color="white" if v > grid.values.max() / 2 else "black")
ax.set(xticks=range(3), yticks=range(3), xticklabels=["left", "center", "right"],
       yticklabels=["top", "middle", "bottom"], title="¿Dónde caen los centros?")
plt.show()

### 1.8 ¿Qué piden las preguntas y cuántas son ambiguas?
Para cada pregunta contamos cuántas instancias reales hay de la clase preguntada.
Motiva: qué operaciones implementar primero y cómo tratar "la instancia" cuando hay varias.

In [ ]:
f2id = dict(zip(imgs.fname, imgs.id))
cnt = objs.groupby(["image_id", "category_name"]).size()

faltan = {c for l in qs.class_en for c in l} - set(cats.category_name)
print("Clases preguntadas que no están en el vocabulario:", faltan or "ninguna")
print("Imágenes de las preguntas que no aparecen en annotations:", set(qs.image_file) - set(f2id) or "ninguna")
print("Imágenes distintas en el benchmark:", qs.image_file.nunique())

qs["n_gt"] = qs.apply(lambda r: cnt.get((f2id[r.image_file], r.class_en[0]), 0) if r.class_en else np.nan, axis=1)
print("\nInstancias reales de la clase preguntada, por tipo:")
print(qs.groupby("type").n_gt.agg(["count", "mean", "min", "max"]).round(1))

amb = qs[qs.type.isin(["position", "size", "relation"]) & (qs.n_gt > 1)]
print(f"\nPreguntas sobre 'la instancia' con más de una instancia real: {len(amb)}")
print(amb[["question_id", "type", "class_en", "n_gt"]].to_string(index=False))
print("\nRespuestas de existencia que serían 'no' (0 instancias):", (qs[qs.type == "existence"].n_gt == 0).sum())

### 1.9 Verificación visual de las anotaciones
Dibujamos cajas (xywh) y polígonos para confirmar que el formato se interpreta bien. Mostramos la imagen con menos objetos, la mediana y la más cargada.

In [ ]:
def mostrar(img_id, ax):
    info = imgs.set_index("id").loc[img_id]
    ax.imshow(Image.open(DEV / "images" / info.fname)); ax.axis("off")
    for a in (x for x in ann["annotations"] if x["image_id"] == img_id):
        x, y, w, h = a["bbox"]
        ax.add_patch(Rectangle((x, y), w, h, fill=False, ec="yellow", lw=1.2))
        for poly in a["segmentation"]:
            ax.add_patch(Polygon(np.array(poly).reshape(-1, 2), fc="cyan", alpha=0.35))
        ax.text(x, y, id2name[a["category_id"]], fontsize=7, backgroundcolor="yellow")
    ax.set_title(f"{info.fname} · {n_obj[img_id]} objetos", fontsize=9)

orden = n_obj.sort_values()
elegidas = [orden.index[0], orden.index[len(orden) // 2], orden.index[-1]]
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for i, ax in zip(elegidas, axes):
    mostrar(i, ax)
plt.tight_layout(); plt.show()

### 1.10 Conclusiones del EDA: observación → decisión

| # | Observación | Decisión que motiva |
|---|---|---|
| 1.2 | 14,9 objetos por imagen en promedio (mediana 10,5; máx. 66) y entre 3 y 8 clases distintas por imagen. | El límite de 300 objetos del schema no es restricción. Dejamos un tope de detecciones holgado (~150) y no filtramos por cantidad. |
| 1.3 | El 54 % de los objetos es chico (< 32² px); la mediana ocupa el 0,29 % de la imagen. | Priorizamos detectores fuertes en objetos chicos (OWLv2 / Grounding DINO) sobre YOLO-World. La resolución de entrada es candidata a experimento controlado. |
| 1.4 | Las 120 clases tienen anotaciones; ninguna aparece en una sola imagen y las 10 más frecuentes suman sólo el 20 %. | Usamos el vocabulario completo de 120 clases: ninguna clase domina el F1, así que no conviene descartar clases raras. |
| 1.5 | El 93,9 % de los pares imagen-clase no es evaluable; sólo hay 118 negativos verificados (~1 por imagen) y el 6 % de los positivos es no exhaustivo. | Un falso positivo de una clase ausente casi nunca penaliza, pero los duplicados dentro de clases presentes sí. La métrica local implementa las reglas LVIS y reporta la cobertura. Esperamos un umbral óptimo bajo (a verificar con el barrido de F1). |
| 1.6 | Sólo el 0,5 % de los pares de objetos reales de la misma clase tiene IoU > 0,5. | NMS por clase con umbral 0,5 es seguro: casi no borra objetos reales. |
| 1.7 | Los centros se concentran en la fila del medio y la columna central (19 % en `center`), pero todas las zonas tienen al menos 7 %. | No hay sesgo fuerte: la regla de grilla 3×3 por centroide es adecuada. |
| 1.8 | Todas las clases preguntadas están en el vocabulario. En `position`, `size` y `relation` la clase tiene exactamente 1 instancia real; en `count` hay entre 2 y 4; 5 de 10 preguntas de `existence` tienen respuesta "no". | Si el detector predice varias instancias, se responde con la de mayor confianza. Para `existence` importa la precisión: un falso positivo cambia la respuesta. |
| 1.9 | Cajas y polígonos coinciden con los objetos de la foto. | Confirmado: `bbox` en formato `[x, y, ancho, alto]`. Para medir máscaras hay que rasterizar los polígonos (pycocotools). |

**Pendientes:** la definición oficial de `small / medium / large` y el orden del RLE (filas o columnas) se verifican en la Fase 3.

## 2. Detección de objetos

Objetivo de la fase: elegir y ajustar un detector open-vocabulary, medido con una métrica local que respeta las reglas de LVIS.

**Regla anti-fuga:** la función `detectar()` recibe sólo la imagen y el vocabulario (`categories.csv`). Las anotaciones se usan únicamente en `evaluar_cajas()`.

### 2.1 Métrica local (F1 de cajas con reglas LVIS)
Para cada imagen y clase:
- **Clase positiva** (tiene anotaciones): emparejamos predicciones con cajas reales de mayor a menor confianza, con IoU ≥ 0,5. Emparejada = TP; sin pareja = FP, salvo que la clase sea *no exhaustiva*, donde se ignora. Cajas reales sin pareja = FN.
- **Clase negativa verificada**: toda predicción es FP.
- **Resto**: la predicción se ignora (no es evaluable).

Supuesto: sumamos TP/FP/FN de todas las imágenes antes de calcular el F1 (micro). La métrica oficial de Kaggle puede agregar distinto.

In [ ]:
from collections import defaultdict

marcar_fase("1 EDA")
img_meta = {r.id: r for r in imgs.itertuples()}
gt_by = defaultdict(list)
for a in ann["annotations"]:
    gt_by[(a["image_id"], a["category_id"])].append(a["bbox"])

def evaluar_cajas(preds, iou_thr=0.5, detalle=False, imagenes=None):
    """preds: DataFrame con image_id, category_id, score y bbox [x, y, w, h]. Aplica las reglas LVIS.
    imagenes: si se indica, evalúa sólo esas imágenes (los objetos reales de las demás no cuentan como FN)."""
    grupos = {k: g.sort_values("score", ascending=False).bbox.tolist()
              for k, g in preds.groupby(["image_id", "category_id"])}
    cats_pred = defaultdict(set)
    for i, c in grupos:
        cats_pred[i].add(c)
    cnt = defaultdict(lambda: np.zeros(4, int))  # TP, FP, FN, ignoradas (por clase)
    for img_id, r in img_meta.items():
        if imagenes is not None and img_id not in imagenes:
            continue
        neg, noexh = set(r.neg_category_ids), set(r.not_exhaustive_category_ids)
        for c in pos.get(img_id, set()) | cats_pred[img_id]:
            pb, gts = grupos.get((img_id, c), []), gt_by.get((img_id, c), [])
            if gts:
                usado = [False] * len(gts)
                for b in pb:
                    ious = [-1 if usado[k] else iou_xywh(b, g) for k, g in enumerate(gts)]
                    k = int(np.argmax(ious))
                    if ious[k] >= iou_thr:
                        usado[k] = True; cnt[c][0] += 1
                    elif c in noexh:
                        cnt[c][3] += 1
                    else:
                        cnt[c][1] += 1
                cnt[c][2] += usado.count(False)
            elif c in neg:
                cnt[c][1] += len(pb)
            else:
                cnt[c][3] += len(pb)
    tp, fp, fn, ign = np.sum(list(cnt.values()), axis=0)
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    res = dict(P=P, R=R, F1=2 * P * R / (P + R) if P + R else 0.0, TP=tp, FP=fp, FN=fn, ignoradas=ign,
               cobertura=(tp + fp) / (tp + fp + ign) if tp + fp + ign else 0.0)
    if detalle:
        tabla = pd.DataFrame(cnt, index=["TP", "FP", "FN", "ign"]).T
        tabla.index = tabla.index.map(id2name)
        return res, tabla
    return res

# Pruebas de cordura de la métrica
gt_df = objs[["image_id", "category_id", "bbox"]].assign(score=1.0)
print("GT como predicción (esperado F1 = 1):", {k: round(float(v), 3) for k, v in evaluar_cajas(gt_df).items()})
corridas = gt_df.assign(bbox=gt_df.bbox.map(lambda b: [b[0] + b[2], b[1], b[2], b[3]]))
print("GT corrida un ancho (esperado F1 ≈ 0):", round(evaluar_cajas(corridas)["F1"], 3))
mitad = gt_df.sample(frac=0.5, random_state=0)
print("Sólo la mitad del GT (esperado P = 1, R ≈ 0,5):", {k: round(float(evaluar_cajas(mitad)[k]), 3) for k in ("P", "R")})

### 2.2 Línea base: OWLv2 con el vocabulario completo
Elegimos **OWLv2** (`google/owlv2-base-patch16-ensemble`) como línea base: es open-vocabulary, procesa las 120 clases en una sola pasada y rinde bien con objetos chicos (54 % del dataset, ver 1.3).
Guardamos todas las detecciones con confianza ≥ 0,05 y después elegimos el umbral midiendo, sin volver a usar la GPU.

In [ ]:
import time
import torch
from transformers import Owlv2Processor, Owlv2ForObjectDetection

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

MODELO = "google/owlv2-base-patch16-ensemble"
processor = Owlv2Processor.from_pretrained(MODELO)
model = Owlv2ForObjectDetection.from_pretrained(MODELO).to(device).eval()

# El nombre del post-procesado cambió entre versiones de transformers
post_proceso = getattr(processor, "post_process_grounded_object_detection", None) or processor.image_processor.post_process_object_detection

VOCAB = cats.category_name.tolist()
CAT_IDS = cats.category_id.tolist()
PROMPTS = [f"a photo of a {n}" for n in VOCAB]

@torch.no_grad()
def detectar(img_path, umbral=0.05):
    """Recibe SÓLO una imagen y el vocabulario. No lee anotaciones."""
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    inp = processor(text=[PROMPTS], images=im, return_tensors="pt").to(device)
    out = model(**inp)
    S = max(W, H)  # OWLv2 rellena la imagen hasta hacerla cuadrada
    res = post_proceso(out, threshold=umbral, target_sizes=torch.tensor([[S, S]], device=device))[0]
    filas = []
    for (x1, y1, x2, y2), s, l in zip(res["boxes"].tolist(), res["scores"].tolist(), res["labels"].tolist()):
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
        if x2 > x1 and y2 > y1:
            filas.append(dict(category_id=CAT_IDS[l], score=s, bbox=[x1, y1, x2 - x1, y2 - y1]))
    return filas

# Prueba con una imagen
t0 = time.time()
prueba = pd.DataFrame(detectar(DEV / "images" / imgs.fname.iloc[0]))
print(f"{len(prueba)} detecciones en {time.time() - t0:.1f} s")
print(prueba.assign(clase=prueba.category_id.map(id2name)).sort_values("score", ascending=False).head(10))

### 2.3 Detección sobre las 120 imágenes de desarrollo
Corremos OWLv2 una sola vez y guardamos las detecciones crudas en `/kaggle/working/`. Todas las decisiones que siguen (NMS, umbral) se ajustan sobre este archivo, sin gastar más GPU.

In [ ]:
SALIDA = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("salidas_local")
SALIDA.mkdir(exist_ok=True)
filas, tiempos = [], []
for img_id, fname in imgs[["id", "fname"]].itertuples(index=False):
    t0 = time.time()
    for d in detectar(DEV / "images" / fname, umbral=0.05):
        d["image_id"] = img_id
        filas.append(d)
    tiempos.append(time.time() - t0)

dets = pd.DataFrame(filas)
dets.to_pickle(SALIDA / "dets_owlv2_base_dev.pkl")
print(f"{len(dets)} detecciones crudas | {np.mean(tiempos):.2f} s por imagen | total {np.sum(tiempos) / 60:.1f} min")
print("Detecciones por imagen (umbral 0,05):", dets.groupby("image_id").size().describe().round(1).to_dict())

### 2.4 NMS por clase y elección del umbral de confianza
Aplicamos NMS por clase con IoU 0,5 (justificado en 1.6) y barremos el umbral de confianza. Elegimos el umbral que maximiza el F1 en desarrollo; también mostramos cuántas predicciones fueron evaluables (cobertura, ver 1.5).

In [ ]:
from torchvision.ops import batched_nms

def aplicar_nms(df, iou=0.5):
    """NMS por clase dentro de cada imagen. Las cajas vienen en [x, y, w, h]."""
    partes = []
    for _, g in df.groupby("image_id"):
        b = torch.tensor(g.bbox.tolist(), dtype=torch.float32)
        b[:, 2:] += b[:, :2]  # a [x1, y1, x2, y2]
        keep = batched_nms(b, torch.tensor(g.score.values, dtype=torch.float32), torch.tensor(g.category_id.values), iou)
        partes.append(g.iloc[keep.numpy()])
    return pd.concat(partes, ignore_index=True)

dets_nms = aplicar_nms(dets)
print(f"NMS: {len(dets)} -> {len(dets_nms)} detecciones")

umbrales = [0.05, 0.075, 0.1, 0.125, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
curva = pd.DataFrame([dict(umbral=u, **evaluar_cajas(dets_nms[dets_nms.score >= u])) for u in umbrales])
print(curva.round(3).to_string(index=False))
mejor = curva.loc[curva.F1.idxmax()]
UMBRAL = float(mejor.umbral)
print(f"\nMejor umbral: {UMBRAL} -> F1 = {mejor.F1:.3f} (P = {mejor.P:.3f}, R = {mejor.R:.3f})")
print("Sin NMS, mismo umbral:", round(evaluar_cajas(dets[dets.score >= UMBRAL])["F1"], 3))

ax = curva.plot(x="umbral", y=["P", "R", "F1"], marker="o", color=[GRIS, "#c98a2b", AZUL])
ax.axhline(0.40, color="#b03a2e", ls=":", label="mínimo aprobado (0,40)")
ax.axvline(UMBRAL, color=GRIS, ls="--")
ax.set(title="¿Qué umbral de confianza maximiza el F1?", xlabel="Umbral de confianza", ylabel="Valor")
ax.legend(); plt.show()

### 2.5 Análisis de errores: ¿dónde pierde puntos el detector?
Con el umbral elegido miramos qué objetos reales no se detectan (FN) según su tamaño y su clase, y qué clases generan más falsos positivos. Esto define el experimento controlado.

In [ ]:
def gt_detectados(preds, iou_thr=0.5):
    """Marca cada objeto real como detectado o no (mismo emparejamiento que evaluar_cajas)."""
    grupos = {k: g.sort_values("score", ascending=False).bbox.tolist()
              for k, g in preds.groupby(["image_id", "category_id"])}
    filas = []
    for (img_id, c), gts in gt_by.items():
        usado = [False] * len(gts)
        for b in grupos.get((img_id, c), []):
            ious = [-1 if usado[k] else iou_xywh(b, g) for k, g in enumerate(gts)]
            k = int(np.argmax(ious))
            if ious[k] >= iou_thr:
                usado[k] = True
        filas += [(img_id, c, g[2] * g[3], u) for g, u in zip(gts, usado)]
    return pd.DataFrame(filas, columns=["image_id", "category_id", "area", "detectado"])

sel = dets_nms[dets_nms.score >= UMBRAL]
gd = gt_detectados(sel)
gd["tamaño"] = pd.cut(gd.area, [0, 32**2, 96**2, np.inf], labels=["chico (<32²)", "mediano", "grande (≥96²)"])
print("Recall por tamaño del objeto:")
print(gd.groupby("tamaño", observed=True).detectado.agg(["count", "mean"]).rename(columns={"count": "objetos", "mean": "recall"}).round(3))

res, tabla = evaluar_cajas(sel, detalle=True)
tabla["recall"] = tabla.TP / (tabla.TP + tabla.FN).replace(0, np.nan)
print("\nLas 10 clases con peor recall (con al menos 5 objetos reales):")
print(tabla[(tabla.TP + tabla.FN) >= 5].sort_values("recall").head(10)[["TP", "FN", "recall"]].round(2))
print("\nLas 10 clases con más falsos positivos:")
print(tabla.sort_values("FP", ascending=False).head(10)[["TP", "FP", "FN"]])

ax = gd.groupby("tamaño", observed=True).detectado.mean().plot.bar(color=AZUL, rot=0)
ax.set(title="¿Qué tan bien detecta según el tamaño del objeto?", xlabel="", ylabel="Recall", ylim=(0, 1))
plt.show()

### 2.6 Diagnóstico: ¿por qué casi no detecta personas?
`person` tiene recall 0,03, algo raro para OWLv2. Hipótesis: el post-procesado estándar asigna **una sola clase por caja** (la de mayor puntaje). Una persona puede competir con clases que la cubren, como prendas de ropa, y perder. Para cada persona real buscamos la caja predicha que mejor la cubre y miramos qué clase recibió.

In [ ]:
def clase_asignada(nombre):
    """Para cada objeto real de la clase, qué clase tiene la caja cruda (umbral 0,05) que mejor lo cubre."""
    cid = cats.set_index("category_name").category_id[nombre]
    res = []
    for (img_id, c), gts in gt_by.items():
        if c != cid:
            continue
        d_img = dets[dets.image_id == img_id]
        for g in gts:
            ious = d_img.bbox.map(lambda b: iou_xywh(b, g))
            m = d_img[ious >= 0.5]
            res.append(id2name[m.loc[m.score.idxmax(), "category_id"]] if len(m) else "(ninguna caja)")
    return pd.Series(res).value_counts()

for nombre in ["person", "book", "shoe"]:
    print(f"--- {nombre}: clase que recibe la mejor caja ---")
    print(clase_asignada(nombre).head(6).to_string(), "\n")

**Resultado del diagnóstico:** de 29 personas, 17 tienen una caja etiquetada `person` pero con confianza baja (no superan el umbral 0,2), y 8 perdieron la etiqueta contra `shirt` o `jacket`. En `book`, 26 de 43 no tienen ninguna caja: es el problema de los objetos chicos (1.3).

### 2.7 Experimento controlado: una etiqueta por caja vs. varias
**Variable que cambiamos:** cuántas clases puede recibir cada caja predicha (k = 1, 2 o 3, las de mayor puntaje). **Fijo:** modelo, prompts, resolución, NMS 0,5 y la métrica.
Hipótesis: con k > 1 recuperamos objetos que pierden contra clases que se solapan (persona vs. ropa), a costa de algunos falsos positivos.
Control: con k = 1 tenemos que reproducir la línea base (F1 = 0,646).

In [ ]:
@torch.no_grad()
def detectar_multi(img_path, umbral=0.05, k_max=3):
    """Como detectar(), pero cada caja puede recibir hasta k_max clases. Recibe SÓLO la imagen y el vocabulario."""
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    inp = processor(text=[PROMPTS], images=im, return_tensors="pt").to(device)
    out = model(**inp)
    probs = out.logits[0].sigmoid()           # (cajas, clases)
    top_p, top_c = probs.topk(k_max, dim=1)   # las k clases con mayor puntaje por caja
    cajas = out.pred_boxes[0].tolist()        # (cx, cy, w, h) normalizadas al cuadrado rellenado
    S = max(W, H)
    filas = []
    for i, rango in (top_p >= umbral).nonzero().tolist():
        cx, cy, w, h = [v * S for v in cajas[i]]
        x1, y1, x2, y2 = max(0, cx - w / 2), max(0, cy - h / 2), min(W, cx + w / 2), min(H, cy + h / 2)
        if x2 > x1 and y2 > y1:
            filas.append(dict(category_id=CAT_IDS[top_c[i, rango].item()], score=top_p[i, rango].item(),
                              bbox=[x1, y1, x2 - x1, y2 - y1], rango=rango))
    return filas

filas = []
t0 = time.time()
for img_id, fname in imgs[["id", "fname"]].itertuples(index=False):
    for d in detectar_multi(DEV / "images" / fname):
        d["image_id"] = img_id
        filas.append(d)
dets_multi = pd.DataFrame(filas)
dets_multi.to_pickle(SALIDA / "dets_owlv2_base_dev_top3.pkl")
print(f"{len(dets_multi)} detecciones crudas en {(time.time() - t0) / 60:.1f} min")

resultados = []
for k in (1, 2, 3):
    dk = aplicar_nms(dets_multi[dets_multi.rango < k])
    for u in umbrales:
        resultados.append(dict(k=k, umbral=u, **evaluar_cajas(dk[dk.score >= u])))
exp = pd.DataFrame(resultados)
resumen = exp.loc[exp.groupby("k").F1.idxmax(), ["k", "umbral", "P", "R", "F1", "TP", "FP", "FN", "cobertura"]]
print("\nMejor F1 de cada variante:")
print(resumen.round(3).to_string(index=False))

ax = exp.pivot(index="umbral", columns="k", values="F1").plot(marker="o", color=[AZUL, "#c98a2b", GRIS])
ax.set(title="¿Mejora el F1 si cada caja puede tener varias clases?", xlabel="Umbral de confianza", ylabel="F1")
ax.legend(title="Clases por caja (k)"); plt.show()

### 2.8 Conclusiones de la Fase 2

| Configuración | Umbral | P | R | F1 | TP | FP | FN |
|---|---|---|---|---|---|---|---|
| OWLv2 base, sin NMS | 0,20 | | | 0,633 | | | |
| Línea base: OWLv2 base + NMS 0,5, 1 clase por caja | 0,20 | 0,732 | 0,578 | 0,646 | 1035 | 378 | 755 |
| **Elegida: 2 clases por caja** | 0,20 | 0,724 | 0,606 | **0,660** | 1084 | 413 | 706 |
| 3 clases por caja | 0,20 | 0,724 | 0,608 | 0,661 | 1089 | 415 | 701 |

**Lectura del experimento.** Permitir varias clases por caja recupera 49 objetos reales a cambio de 35 falsos positivos más, y sube el F1 en 1,4 puntos. Elegimos **k = 2**: k = 3 casi no agrega (+0,001) y genera más detecciones. El control con k = 1 reprodujo exactamente la línea base, así que la mejora se debe sólo a la variable cambiada. El umbral óptimo se mantuvo en 0,20 en las tres variantes.

**Qué sigue limitando el resultado:**
- **Recall bajo en objetos chicos:** 0,43 en chicos contra 0,72 en grandes. En `book`, 26 de 43 objetos no tienen ninguna caja. Candidato a experimento: OWLv2 *large* o mayor resolución de entrada.
- **Confianzas no calibradas entre clases:** `person` y `shoe` se detectan pero con puntaje bajo, y un umbral único los deja afuera. Un umbral por clase ajustado con sólo 120 imágenes tendría riesgo alto de sobreajuste, así que lo dejamos como limitación documentada.
- **Falsos positivos concentrados** en clases parecidas entre sí (`ski`, `streetlight`, `knife`, `traffic light`).

**Configuración que pasa a la Fase 3:** OWLv2 base, prompt `"a photo of a {clase}"`, hasta 2 clases por caja, NMS por clase 0,5 y umbral 0,20.

*Nota:* estas métricas son locales (F1 micro con reglas LVIS). El valor oficial se confirma con el primer envío a Kaggle.

### 2.9 Experimentos de mejora del detector (agregado en la v4)

El análisis de 2.5 deja dos cuellos de botella: **objetos chicos** (recall 0,43) y **confianzas bajas** en clases como `person`. Probamos cada cambio **por separado** y sobre las mismas 120 imágenes, con la misma métrica (2.1), el mismo NMS por clase 0,5 y el mismo barrido de umbral y de clases por caja (k):

| # | Hipótesis | Variable que cambia |
|---|---|---|
| E3 | Un backbone más grande separa mejor clases parecidas y ve mejor los objetos chicos | `owlv2-base-patch16` → `owlv2-large-patch14` (entrada 960 → 1008 px) |
| E7 | Para nombres raros de LVIS (`fireplug`, `jean`, `spectacles`…) y para `person`, sumar sinónimos sube la confianza correcta | Cada clase puede tener varias consultas de texto; el puntaje de la clase es el **máximo** entre sus consultas |
| E4 | Mirar la imagen por partes (estilo SAHI) agranda los objetos chicos: cada ventana se reescala a la entrada de OWLv2, así que los objetos se ven 1,7 veces más grandes por lado (2,8 veces en área) | Además de la imagen completa, 4 ventanas de 60 % × 60 % que se solapan. Se descartan las cajas que tocan un borde interno de la ventana (objeto cortado) y el NMS une los duplicados |
| E8 (v5) | Large solo rinde menos que base, pero puede encontrar objetos distintos: el ensamble suma recall | Se agregan las detecciones de large (imagen completa) a las de base y el NMS une los duplicados |

**Cómo se elige la configuración final:** la de mayor F1 de cajas en desarrollo. Cada variante se mide con su mejor umbral y su mejor k, así ninguna sale perjudicada por un umbral elegido para otra. El umbral se elige con un paso de 0,01 alrededor del óptimo grueso.

**Detalle de implementación (E7):** en OWLv2 el puntaje de una caja para una consulta de texto no depende de las otras consultas. Por eso una sola pasada con todas las consultas permite medir "con" y "sin sinónimos" sin volver a usar la GPU.

La configuración ganadora reemplaza a la de la Fase 2: las Fases 3 y 4 usan `detectar_multi`, `UMBRAL` y `K_MAX` tal como quedan al final de esta celda.

In [ ]:
import gc

# --- Consultas de texto: la plantilla de la línea base y, para algunas clases, sinónimos (E7) ---
# Sólo sinónimos que no son el nombre de otra clase del vocabulario (por ejemplo, no "cap" ni "door handle").
SINONIMOS = {
    "person": ["man", "woman", "child"], "fireplug": ["fire hydrant"], "jean": ["jeans"], "trousers": ["pants"],
    "short pants": ["shorts"], "spectacles": ["eyeglasses"], "television set": ["television"],
    "wall socket": ["electrical outlet"], "flower arrangement": ["bouquet of flowers"], "streetlight": ["street lamp"],
    "trash can": ["garbage can"], "cell phone": ["mobile phone"], "computer monitor": ["computer screen"],
    "handbag": ["purse"], "necktie": ["tie"], "flowerpot": ["plant pot"], "taillight": ["tail light"],
    "license plate": ["number plate"], "teddy bear": ["stuffed animal"], "jersey": ["sports jersey"],
    "vent": ["air vent"], "frisbee": ["flying disc"],
}
assert set(SINONIMOS) <= set(VOCAB)
Q_TEXTOS, Q_CLASE, Q_BASE = [], [], []
for i, nombre in enumerate(VOCAB):
    for j, t in enumerate([nombre] + SINONIMOS.get(nombre, [])):
        Q_TEXTOS.append(f"a photo of a {t}"); Q_CLASE.append(i); Q_BASE.append(j == 0)
Q_CLASE_T, Q_BASE_T = torch.tensor(Q_CLASE, device=device), torch.tensor(Q_BASE, device=device)
VARIANTES_Q = {"sin_sinonimos": Q_BASE_T, "con_sinonimos": torch.ones_like(Q_BASE_T)}
print(f"Consultas de texto: {len(Q_TEXTOS)} ({sum(Q_BASE)} de la línea base + {len(Q_TEXTOS) - sum(Q_BASE)} sinónimos)")

MODELO_BASE, MODELO_LARGE = MODELO, "google/owlv2-large-patch14-ensemble"
OWL = {MODELO_BASE: (processor, model)}

def cargar_owl(nombre):
    if nombre not in OWL:
        OWL[nombre] = (Owlv2Processor.from_pretrained(nombre), Owlv2ForObjectDetection.from_pretrained(nombre).to(device).eval())
    return OWL[nombre]

@torch.no_grad()
def owl_pasada(im, nombre, umbral=0.05, k_max=3):
    """Una pasada de OWLv2 sobre una imagen PIL (completa o ventana). Recibe SÓLO la imagen y el vocabulario.
    Devuelve {variante de consultas: filas}, con cajas [x, y, w, h] en píxeles de `im`."""
    proc, mod = cargar_owl(nombre)
    W, H = im.size
    S = max(W, H)                                               # OWLv2 rellena la imagen hasta hacerla cuadrada
    out = mod(**proc(text=[Q_TEXTOS], images=im, return_tensors="pt").to(device))
    pq = out.logits[0].sigmoid()                                # (cajas, consultas)
    cajas = (out.pred_boxes[0] * S).cpu().numpy()               # (cx, cy, w, h) en píxeles
    res = {}
    for var, usar in VARIANTES_Q.items():
        idx = Q_CLASE_T[usar].expand(pq.shape[0], -1)
        p = torch.zeros(pq.shape[0], len(VOCAB), device=pq.device).scatter_reduce(1, idx, pq[:, usar], reduce="amax", include_self=False)
        top_p, top_c = (t.cpu().numpy() for t in p.topk(k_max, dim=1))   # puntaje de la clase = máximo entre sus consultas
        filas = []
        for i, rango in zip(*np.nonzero(top_p >= umbral)):
            cx, cy, w, h = cajas[i]
            x1, y1, x2, y2 = max(0.0, cx - w / 2), max(0.0, cy - h / 2), min(W, cx + w / 2), min(H, cy + h / 2)
            if x2 > x1 and y2 > y1:
                filas.append(dict(category_id=CAT_IDS[top_c[i, rango]], score=float(top_p[i, rango]),
                                  bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)], rango=int(rango)))
        res[var] = filas
    return res

FRAC_VENTANA, BORDE_VENTANA = 0.6, 3

def ventanas(W, H, frac=FRAC_VENTANA):
    """4 ventanas de frac × frac en las esquinas: se solapan en la franja central."""
    tw, th = int(round(W * frac)), int(round(H * frac))
    return [(x, y, x + tw, y + th) for y in (0, H - th) for x in (0, W - tw)]

def owl_ventanas(im, nombre, umbral=0.05, k_max=3):
    """E4: OWLv2 sobre cada ventana. Descarta las cajas que tocan un borde INTERNO de la ventana (objeto cortado)
    y devuelve el resto en coordenadas de la imagen completa."""
    W, H = im.size
    res = defaultdict(list)
    for x1, y1, x2, y2 in ventanas(W, H):
        for var, filas in owl_pasada(im.crop((x1, y1, x2, y2)), nombre, umbral, k_max).items():
            for f in filas:
                bx, by, bw, bh = f["bbox"]
                cortada = ((x1 > 0 and bx <= BORDE_VENTANA) or (y1 > 0 and by <= BORDE_VENTANA) or
                           (x2 < W and bx + bw >= (x2 - x1) - BORDE_VENTANA) or (y2 < H and by + bh >= (y2 - y1) - BORDE_VENTANA))
                if not cortada:
                    res[var].append(dict(f, bbox=[bx + x1, by + y1, bw, bh]))
    return res

def pasada_dev(nombre, con_ventanas=False):
    """Corre una variante del detector sobre las 120 imágenes de desarrollo (umbral 0,05, k ≤ 3) y guarda las crudas."""
    filas, t0 = [], time.time()
    for img_id, fname in imgs[["id", "fname"]].itertuples(index=False):
        im = Image.open(DEV / "images" / fname).convert("RGB")
        for var, lista in (owl_ventanas if con_ventanas else owl_pasada)(im, nombre).items():
            filas += [dict(f, image_id=img_id, variante=var, fuente="ventana" if con_ventanas else "imagen") for f in lista]
    df = pd.DataFrame(filas)
    df.to_pickle(SALIDA / f"dets_{nombre.split('/')[-1]}_{'ventanas' if con_ventanas else 'completa'}_dev.pkl")
    print(f"{nombre.split('/')[-1]} {'(4 ventanas)' if con_ventanas else '(imagen completa)'}: "
          f"{len(df)} detecciones crudas en {(time.time() - t0) / 60:.1f} min")
    return df

UMBRALES_GRUESOS = [0.05, 0.075, 0.1, 0.125, 0.15, 0.175, 0.2, 0.225, 0.25, 0.3, 0.35, 0.4]

def mejor_config(crudas, ks=(1, 2, 3)):
    """Mejor (k, umbral) por F1 de cajas: barrido grueso y después fino (paso 0,01) alrededor del óptimo."""
    mejores = []
    for k in ks:
        dk = aplicar_nms(crudas[crudas.rango < k])
        curva = {u: evaluar_cajas(dk[dk.score >= u]) for u in UMBRALES_GRUESOS}
        u0 = max(curva, key=lambda u: curva[u]["F1"])
        for u in np.round(np.arange(max(0.03, u0 - 0.03), u0 + 0.031, 0.01), 3):
            if float(u) not in curva:
                curva[float(u)] = evaluar_cajas(dk[dk.score >= u])
        u = max(curva, key=lambda u: curva[u]["F1"])
        mejores.append(dict(k=k, umbral=float(u), **curva[u]))
    return max(mejores, key=lambda r: r["F1"])

resultados_det, candidatos = [], {}

def registrar(nombre, crudas, modelo, sinonimos, con_ventanas, ensamble=False):
    m = mejor_config(crudas)
    candidatos[nombre] = crudas
    resultados_det.append(dict(variante=nombre, modelo=modelo, sinonimos=sinonimos, ventanas=con_ventanas, ensamble_large=ensamble,
                               **{c: m[c] for c in ["k", "umbral", "P", "R", "F1", "TP", "FP", "FN", "cobertura"]}))
    print(f"  {nombre:42s} F1 {m['F1']:.4f}  (P {m['P']:.3f}, R {m['R']:.3f}, k = {m['k']}, umbral {m['umbral']:.2f})")

# Línea base: las mismas detecciones crudas de 2.7, con el barrido fino de umbral
print("Línea base y E3 (modelo):")
registrar("base", dets_multi.assign(variante="sin_sinonimos", fuente="imagen"), MODELO_BASE, False, False)
base_q = pasada_dev(MODELO_BASE)
ctrl = aplicar_nms(base_q[(base_q.variante == "sin_sinonimos") & (base_q.rango < 2)])
print("  Control: sin sinónimos, k = 2 y umbral 0,20 reproduce la Fase 2 (F1 0,660):", round(evaluar_cajas(ctrl[ctrl.score >= 0.2])["F1"], 3))
large_q = pasada_dev(MODELO_LARGE)
registrar("large", large_q[large_q.variante == "sin_sinonimos"], MODELO_LARGE, False, False)
F1_E3 = {r["modelo"]: r["F1"] for r in resultados_det}
M = max(F1_E3, key=F1_E3.get)
crudas_M = large_q if M == MODELO_LARGE else base_q
corto = "large" if M == MODELO_LARGE else "base"

print(f"E7 (sinónimos), sobre el mejor modelo de E3 ({corto}):")
registrar(f"{corto} + sinónimos", crudas_M[crudas_M.variante == "con_sinonimos"], M, True, False)

print(f"E4 (ventanas), sobre {corto}:")
vent = pasada_dev(M, con_ventanas=True)
for var, con_sin in (("sin_sinonimos", False), ("con_sinonimos", True)):
    combinadas = pd.concat([crudas_M[crudas_M.variante == var], vent[vent.variante == var]], ignore_index=True)
    registrar(f"{corto} + ventanas" + (" + sinónimos" if con_sin else ""), combinadas, M, con_sin, True)

# E8 (v5): ensamble con large. Large solo rinde menos, pero puede encontrar objetos que base no ve; el NMS une los duplicados
if M == MODELO_BASE:
    print("E8 (ensamble base + large):")
    ens = pd.concat([crudas_M[crudas_M.variante == "con_sinonimos"], vent[vent.variante == "con_sinonimos"],
                     large_q[large_q.variante == "con_sinonimos"]], ignore_index=True)
    registrar("base + ventanas + sinónimos + large", ens, M, True, True, ensamble=True)

tabla_det = pd.DataFrame(resultados_det)
RESULTADOS = SALIDA / "results"
RESULTADOS.mkdir(exist_ok=True)
tabla_det.to_csv(RESULTADOS / "det_experimentos.csv", index=False)
print("\n" + tabla_det.drop(columns=["modelo"]).round(4).to_string(index=False))

# --- Configuración ganadora: reemplaza a la de la Fase 2 ---
g = tabla_det.loc[tabla_det.F1.idxmax()]
MODELO, K_MAX, UMBRAL = g.modelo, int(g.k), float(g.umbral)
VAR_ELEGIDA = "con_sinonimos" if g.sinonimos else "sin_sinonimos"
DET_CONFIG = dict(modelo=MODELO, prompt="a photo of a {clase}", sinonimos=bool(g.sinonimos), ventanas=bool(g.ventanas),
                  frac_ventana=FRAC_VENTANA if g.ventanas else None, ensamble_large=bool(g.ensamble_large),
                  k_max=K_MAX, nms_iou=0.5, umbral=UMBRAL)
dets_multi = candidatos[g.variante]
processor, model = cargar_owl(MODELO)
for nombre in [n for n in OWL if n != MODELO and not (DET_CONFIG["ensamble_large"] and n == MODELO_LARGE)]:   # liberar la GPU
    del OWL[nombre]
gc.collect(); torch.cuda.empty_cache() if device == "cuda" else None

def detectar_multi(img_path, umbral=0.05, k_max=None):
    """Detector final (configuración elegida en 2.9). Recibe SÓLO la imagen y el vocabulario."""
    im = Image.open(img_path).convert("RGB")
    k_max = k_max or K_MAX
    filas = [dict(f, fuente="imagen") for f in owl_pasada(im, MODELO, umbral, k_max)[VAR_ELEGIDA]]   # fuente: linaje (v6)
    if DET_CONFIG["ventanas"]:
        filas += [dict(f, fuente="ventana") for f in owl_ventanas(im, MODELO, umbral, k_max)[VAR_ELEGIDA]]
    if DET_CONFIG["ensamble_large"]:
        filas += [dict(f, fuente="large") for f in owl_pasada(im, MODELO_LARGE, umbral, k_max)[VAR_ELEGIDA]]
    return filas

print(f"\nConfiguración elegida: {g.variante} -> F1 {g.F1:.4f} (línea base 0,660: {g.F1 - 0.660:+.3f})")
print(json.dumps(DET_CONFIG, ensure_ascii=False))

# ¿Mejoró donde decía la hipótesis? Recall por tamaño, línea base vs. elegida
def recall_por_tamano(crudas, k, u):
    dk = aplicar_nms(crudas[crudas.rango < k])
    gd = gt_detectados(dk[dk.score >= u])
    gd["tamaño"] = pd.cut(gd.area, [0, 32**2, 96**2, np.inf], labels=["chico (<32²)", "mediano", "grande (≥96²)"])
    return gd.groupby("tamaño", observed=True).detectado.mean()
rb = tabla_det.iloc[0]
comp_tam = pd.DataFrame({"línea base": recall_por_tamano(candidatos["base"], int(rb.k), float(rb.umbral)),
                         "elegida": recall_por_tamano(dets_multi, K_MAX, UMBRAL)})
print("\nRecall por tamaño del objeto:\n", comp_tam.round(3).to_string())

ax = tabla_det.set_index("variante").F1.plot.barh(color=[AZUL if v == g.variante else GRIS for v in tabla_det.variante])
ax.axvline(0.50, color="#b03a2e", ls=":", label="mínimo promoción (0,50)")
ax.set(title="F1 de cajas en desarrollo por variante del detector", xlabel="F1 (mejor k y umbral de cada una)", ylabel="")
ax.set_xlim(0.4, max(0.8, tabla_det.F1.max() + 0.02)); ax.legend(); plt.show()
marcar_fase("2 detección")

## 3. Segmentación (SAM2) y propiedades derivadas

El detector dice **qué es y dónde está, aproximadamente** (caja). SAM2 dice **exactamente qué píxeles** ocupa, pero no sabe qué es: necesita que le pasemos la caja como prompt. Por eso va segundo en la pipeline.

De la máscara salen las propiedades **derivadas** del SceneJSON: `centroid`, `area_ratio`, `position` (grilla 3×3) y `size`; y entre pares de objetos, las `relations`.

### 3.1 Carga de SAM2 y prueba con una imagen

In [ ]:
try:
    from transformers import Sam2Processor, Sam2Model
    SAM_ID = "facebook/sam2.1-hiera-small"
    sam_proc = Sam2Processor.from_pretrained(SAM_ID)
    sam = Sam2Model.from_pretrained(SAM_ID).to(device).eval()
    ES_SAM2 = True
except Exception as e:  # respaldo si esta versión de transformers no trae SAM2
    print("SAM2 no disponible, uso SAM v1:", repr(e)[:200])
    from transformers import SamProcessor, SamModel
    SAM_ID = "facebook/sam-vit-base"
    sam_proc = SamProcessor.from_pretrained(SAM_ID)
    sam = SamModel.from_pretrained(SAM_ID).to(device).eval()
    ES_SAM2 = False
print("Segmentador:", SAM_ID)

@torch.no_grad()
def segmentar(im, cajas_xywh, lote=64):
    """Recibe la imagen y las cajas del detector. Devuelve una máscara booleana (H, W) por caja."""
    mascaras = []
    for i in range(0, len(cajas_xywh), lote):
        xyxy = [[x, y, x + w, y + h] for x, y, w, h in cajas_xywh[i:i + lote]]
        inp = sam_proc(images=im, input_boxes=[xyxy], return_tensors="pt").to(device)
        out = sam(**inp, multimask_output=False)
        if ES_SAM2:
            m = sam_proc.post_process_masks(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
        else:
            m = sam_proc.post_process_masks(out.pred_masks.cpu(), inp["original_sizes"].cpu(), inp["reshaped_input_sizes"].cpu())[0]
        mascaras += [mm[0].numpy().astype(bool) for mm in m]
    return mascaras

# Detecciones finales: configuración elegida en 2.9 (hasta K_MAX clases por caja, NMS 0,5, umbral UMBRAL)
dk = aplicar_nms(dets_multi[dets_multi.rango < K_MAX])
dets_final = dk[dk.score >= UMBRAL].reset_index(drop=True)
print(f"Detecciones finales en desarrollo: {len(dets_final)} (umbral {UMBRAL})")

# Prueba con la imagen de la verificación visual 1.9 (11 objetos)
img_id = elegidas[1]
info = img_meta[img_id]
im = Image.open(DEV / "images" / info.fname).convert("RGB")
d_img = dets_final[dets_final.image_id == img_id]
t0 = time.time()
masks = segmentar(im, d_img.bbox.tolist())
print(f"{len(masks)} máscaras en {time.time() - t0:.2f} s | forma {masks[0].shape} | imagen {info.height}x{info.width}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(im); ax.axis("off")
colores = plt.cm.tab10(np.arange(len(masks)) % 10)
for m, (_, d), col in zip(masks, d_img.iterrows(), colores):
    capa = np.zeros((*m.shape, 4)); capa[m] = [*col[:3], 0.45]
    ax.imshow(capa)
    x, y, w, h = d.bbox
    ax.add_patch(Rectangle((x, y), w, h, fill=False, ec=col, lw=1.5))
    ax.text(x, y, f"{id2name[d.category_id]} {d.score:.2f}", fontsize=7, color="white", backgroundcolor=col)
ax.set_title(f"{info.fname}: cajas del detector + máscaras de SAM", fontsize=10)
plt.show()

### 3.2 Codificación RLE y test de ida y vuelta
El schema pide la máscara como RLE **no comprimido**: `{"size": [alto, ancho], "counts": [...]}`, donde `counts` son largos de corridas alternadas que **arrancan contando ceros**. El `size` en orden `[alto, ancho]` es la convención de COCO (lo confirma `sample_submission.csv`), así que usamos también su orden de aplanado: **por columnas** (Fortran).

Un error acá deja el F1 de máscaras casi en 0 aunque las máscaras sean perfectas, por eso lo testeamos tres veces: contra el ejemplo oficial, con máscaras al azar y contra `pycocotools`.

In [ ]:
from pycocotools import mask as mask_utils

def rle_encode(mask):
    """Máscara booleana (H, W) -> RLE no comprimido estilo COCO (por columnas, arranca contando ceros)."""
    v = mask.ravel(order="F").astype(np.uint8)
    cambios = np.flatnonzero(np.diff(v)) + 1
    counts = np.diff(np.concatenate([[0], cambios, [v.size]])).tolist()
    if v[0] == 1:
        counts = [0] + counts
    return {"size": [int(mask.shape[0]), int(mask.shape[1])], "counts": counts}

def rle_decode(rle):
    h, w = rle["size"]
    v = np.zeros(h * w, dtype=bool)
    pos, valor = 0, False
    for c in rle["counts"]:
        if valor:
            v[pos:pos + c] = True
        pos += c
        valor = not valor
    return v.reshape((h, w), order="F")

# Test 1: el ejemplo oficial de sample_submission (un solo píxel en la esquina)
ej = json.loads(sub.iloc[0].SceneJSON)["objects"][0]["mask"]
m_ej = rle_decode(ej)
print("Test 1 | píxeles en 1:", m_ej.sum(), "| en (0,0):", m_ej[0, 0], "| re-codifica igual:", rle_encode(m_ej) == ej)

# Test 2: ida y vuelta con 200 máscaras al azar, incluso las que arrancan en 1 o están vacías
rng = np.random.default_rng(0)
ok = all(np.array_equal(rle_decode(rle_encode(m)), m)
         for m in [rng.random((rng.integers(1, 60), rng.integers(1, 60))) < rng.random() for _ in range(200)])
print("Test 2 | ida y vuelta en 200 máscaras al azar:", ok)

# Test 3: pycocotools tiene que leer nuestro RLE y obtener la misma máscara (mismo orden de píxeles)
ok3 = True
for m in masks[:10]:
    r = rle_encode(m)
    dec = mask_utils.decode(mask_utils.frPyObjects(r, *r["size"]))
    ok3 &= np.array_equal(dec.astype(bool), m)
print("Test 3 | pycocotools decodifica igual (10 máscaras de SAM):", ok3)
largo = np.mean([len(rle_encode(m)["counts"]) for m in masks])
print(f"Largo medio de counts: {largo:.0f} números por máscara (contra {masks[0].size} píxeles)")

### 3.3 Segmentación de las 120 imágenes y F1 de máscaras
Segmentamos todas las detecciones finales de la Fase 2 y medimos el F1 de máscaras con las mismas reglas LVIS, pero con IoU entre máscaras.

**Experimento controlado:** comparamos SAM2 contra una línea base trivial que usa **la caja rellena como máscara**. Mismo detector, mismas detecciones, misma métrica: sólo cambia de dónde sale la máscara. Si SAM2 no le gana a la caja, no está aportando nada.

In [ ]:
# Máscaras reales: polígonos de annotations.json -> RLE de pycocotools (sólo para evaluar)
gt_rle = defaultdict(list)
for a in ann["annotations"]:
    r = img_meta[a["image_id"]]
    gt_rle[(a["image_id"], a["category_id"])].append(
        mask_utils.merge(mask_utils.frPyObjects(a["segmentation"], r.height, r.width)))

def a_rle_coco(rle):
    """Acepta nuestro RLE no comprimido o uno ya comprimido por pycocotools."""
    return mask_utils.frPyObjects(rle, *rle["size"]) if isinstance(rle["counts"], list) else rle

def evaluar_mascaras(preds, col="rle", iou_thr=0.5):
    """Mismas reglas LVIS que evaluar_cajas, pero con IoU entre máscaras."""
    grupos = {k: g.sort_values("score", ascending=False)[col].tolist()
              for k, g in preds.groupby(["image_id", "category_id"])}
    cats_pred = defaultdict(set)
    for i, c in grupos:
        cats_pred[i].add(c)
    tp = fp = fn = ign = 0
    for img_id, r in img_meta.items():
        neg, noexh = set(r.neg_category_ids), set(r.not_exhaustive_category_ids)
        for c in pos.get(img_id, set()) | cats_pred[img_id]:
            pm, gts = grupos.get((img_id, c), []), gt_rle.get((img_id, c), [])
            if gts:
                usado = np.zeros(len(gts), bool)
                if pm:
                    M = np.atleast_2d(mask_utils.iou([a_rle_coco(m) for m in pm], gts, [0] * len(gts)))
                    for i in range(len(pm)):
                        fila = np.where(usado, -1, M[i])
                        k = int(np.argmax(fila))
                        if fila[k] >= iou_thr:
                            usado[k] = True; tp += 1
                        elif c in noexh:
                            ign += 1
                        else:
                            fp += 1
                fn += int((~usado).sum())
            elif c in neg:
                fp += len(pm)
            else:
                ign += len(pm)
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    return dict(P=P, R=R, F1=2 * P * R / (P + R) if P + R else 0.0, TP=tp, FP=fp, FN=fn, ignoradas=ign)

gt_pred = pd.DataFrame([dict(image_id=i, category_id=c, score=1.0, rle=m) for (i, c), ms in gt_rle.items() for m in ms])
print("Cordura | máscaras reales como predicción (esperado F1 = 1):", round(evaluar_mascaras(gt_pred)["F1"], 3))

def caja_a_mascara(b, H, W):
    m = np.zeros((H, W), bool)
    x, y, w, h = b
    m[int(round(y)):int(round(y + h)), int(round(x)):int(round(x + w))] = True
    return m

rle_sam, rle_caja, vacias = {}, {}, 0
t0 = time.time()
for img_id, d_img in dets_final.groupby("image_id", sort=False):
    r = img_meta[img_id]
    im = Image.open(DEV / "images" / r.fname).convert("RGB")
    for idx, b, m in zip(d_img.index, d_img.bbox, segmentar(im, d_img.bbox.tolist())):
        mc = caja_a_mascara(b, r.height, r.width)
        if m.sum() == 0:  # SAM no devolvió píxeles: usamos la caja como respaldo
            m, vacias = mc, vacias + 1
        rle_sam[idx], rle_caja[idx] = rle_encode(m), rle_encode(mc)
dets_final["rle_sam"] = pd.Series(rle_sam)
dets_final["rle_caja"] = pd.Series(rle_caja)
print(f"{len(dets_final)} máscaras en {(time.time() - t0) / 60:.1f} min | máscaras vacías reemplazadas por la caja: {vacias}")
dets_final.to_pickle(SALIDA / "dets_final_dev_con_mascaras.pkl")

comp = pd.DataFrame([dict(fuente="Caja como máscara (línea base)", **evaluar_mascaras(dets_final, "rle_caja")),
                     dict(fuente="SAM2 con la caja como prompt", **evaluar_mascaras(dets_final, "rle_sam"))])
print("\nF1 de máscaras (IoU de máscara ≥ 0,5):")
print(comp.round(3).to_string(index=False))
print("\nReferencia, F1 de cajas de las mismas detecciones:", round(evaluar_cajas(dets_final)["F1"], 3))

### 3.3b Experimentos de segmentación (agregado en la v4)

Tres preguntas, con las mismas detecciones finales y la misma métrica de 3.3:

| # | Pregunta | Cómo se mide |
|---|---|---|
| E6 | ¿Un SAM2 más grande da mejores máscaras con las mismas cajas? | F1 de máscaras con `sam2.1-hiera-small` vs `sam2.1-hiera-large` |
| E5 | ¿Cuánto pierde SAM2 por sí solo, con cajas perfectas? (oráculo) | SAM2 recibe las **cajas reales** y se mide el IoU de cada máscara contra su máscara real. Separa los errores del detector de los del segmentador |
| — | ¿Conviene reemplazar la caja del detector por la caja que encierra la máscara? | F1 de cajas con una y otra caja |

El segmentador con mayor F1 de máscaras pasa a ser el de la pipeline. Las anotaciones se usan sólo para medir: en el oráculo, las cajas reales nunca llegan al envío.

In [ ]:
@torch.no_grad()
def segmentar_con(proc, mod, im, cajas_xywh, lote=64):
    """Igual que segmentar(), pero con el segmentador que se le pasa (para comparar modelos)."""
    mascaras = []
    for i in range(0, len(cajas_xywh), lote):
        xyxy = [[x, y, x + w, y + h] for x, y, w, h in cajas_xywh[i:i + lote]]
        inp = proc(images=im, input_boxes=[xyxy], return_tensors="pt").to(device)
        out = mod(**inp, multimask_output=False)
        m = proc.post_process_masks(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
        mascaras += [mm[0].numpy().astype(bool) for mm in m]
    return mascaras

def caja_de_mascara(m):
    """Caja [x, y, w, h] que encierra los píxeles de la máscara."""
    ys, xs = np.nonzero(m)
    return [float(xs.min()), float(ys.min()), float(xs.max() - xs.min() + 1), float(ys.max() - ys.min() + 1)]

res_seg = [dict(segmentador=SAM_ID, **evaluar_mascaras(dets_final, "rle_sam"))]
SAM_LARGE = "facebook/sam2.1-hiera-large"
if ES_SAM2:
    sam_proc_L = Sam2Processor.from_pretrained(SAM_LARGE)
    sam_L = Sam2Model.from_pretrained(SAM_LARGE).to(device).eval()
    rle_L, t0 = {}, time.time()
    for img_id, d_img in dets_final.groupby("image_id", sort=False):
        r = img_meta[img_id]
        im = Image.open(DEV / "images" / r.fname).convert("RGB")
        for idx, b, m in zip(d_img.index, d_img.bbox, segmentar_con(sam_proc_L, sam_L, im, d_img.bbox.tolist())):
            rle_L[idx] = rle_encode(m if m.sum() else caja_a_mascara(b, r.height, r.width))
    dets_final["rle_sam_large"] = pd.Series(rle_L)
    res_seg.append(dict(segmentador=SAM_LARGE, **evaluar_mascaras(dets_final, "rle_sam_large")))
    print(f"SAM2 large: {len(rle_L)} máscaras en {(time.time() - t0) / 60:.1f} min")
tabla_seg = pd.DataFrame(res_seg)
print("\nE6 · F1 de máscaras con las mismas cajas:")
print(tabla_seg.round(4).to_string(index=False))

# El mejor segmentador pasa a la pipeline (Fase 4 usa sam_proc, sam y SAM_ID)
if len(res_seg) > 1 and res_seg[1]["F1"] > res_seg[0]["F1"]:
    sam_proc, sam, SAM_ID = sam_proc_L, sam_L, SAM_LARGE
    dets_final["rle_sam"] = dets_final["rle_sam_large"]
elif ES_SAM2:
    del sam_proc_L, sam_L
gc.collect(); torch.cuda.empty_cache() if device == "cuda" else None
print("Segmentador elegido:", SAM_ID)
dets_final.to_pickle(SALIDA / "dets_final_dev_con_mascaras.pkl")

# Caja del detector vs. caja que encierra la máscara
cajas_m = dets_final.rle_sam.map(lambda r: caja_de_mascara(rle_decode(r)))
F1_CAJA_DET = evaluar_cajas(dets_final)["F1"]
F1_CAJA_MASC = evaluar_cajas(dets_final.assign(bbox=cajas_m))["F1"]
CAJA_DESDE_MASCARA = F1_CAJA_MASC > F1_CAJA_DET
print(f"\nF1 de cajas | caja del detector: {F1_CAJA_DET:.4f} | caja de la máscara: {F1_CAJA_MASC:.4f}"
      f" -> se usa la {'de la máscara' if CAJA_DESDE_MASCARA else 'del detector'}")

# E5 · Oráculo: SAM2 (el elegido) con las cajas reales
seg_por_ann = {a["id"]: a["segmentation"] for a in ann["annotations"]}
filas_or, t0 = [], time.time()
for img_id, g_img in objs.groupby("image_id"):
    r = img_meta[img_id]
    im = Image.open(DEV / "images" / r.fname).convert("RGB")
    for a, m in zip(g_img.itertuples(), segmentar(im, g_img.bbox.tolist())):
        gt = mask_utils.decode(mask_utils.merge(mask_utils.frPyObjects(seg_por_ann[a.id], r.height, r.width))).astype(bool)
        union = (m | gt).sum()
        filas_or.append(dict(area=a.area, iou=float((m & gt).sum() / union) if union else 0.0))
oraculo = pd.DataFrame(filas_or)
oraculo["tamaño"] = pd.cut(oraculo.area, [0, 32**2, 96**2, np.inf], labels=["chico (<32²)", "mediano", "grande (≥96²)"])
print(f"\nE5 · Oráculo ({len(oraculo)} objetos reales, {(time.time() - t0) / 60:.1f} min): IoU medio {oraculo.iou.mean():.3f} | "
      f"máscaras con IoU ≥ 0,5: {(oraculo.iou >= 0.5).mean():.1%}")
print(oraculo.groupby("tamaño", observed=True).iou.agg(objetos="size", iou_medio="mean",
                                                        iou_ge_05=lambda s: (s >= 0.5).mean()).round(3).to_string())
tabla_seg.to_csv(RESULTADOS / "seg_experimentos.csv", index=False)
oraculo.to_csv(RESULTADOS / "seg_oraculo.csv", index=False)
marcar_fase("3 segmentación")

### 3.4 Propiedades derivadas y relaciones espaciales
Estas propiedades **no las predice ningún modelo**: se calculan con código a partir de la máscara. Son *derivadas* y por eso son auditables.

| Campo | Regla |
|---|---|
| `centroid` | promedio de las coordenadas (x, y) de los píxeles de la máscara |
| `area_ratio` | píxeles de la máscara / píxeles de la imagen |
| `position` | celda de la grilla 3×3 donde cae el centroide |
| `size` | umbrales de COCO sobre el área en píxeles: `small` < 32², `medium` < 96², `large` ≥ 96² |
| `left_of` / `right_of` | diferencia horizontal de centroides mayor al 5 % del ancho |
| `above` / `below` | diferencia vertical de centroides mayor al 5 % del alto (y crece hacia abajo) |
| `overlaps` | IoU entre máscaras > 0,01 (simétrica) |

El margen del 5 % evita relaciones ruidosas entre objetos casi alineados. Los umbrales de `size` no están en el enunciado: usamos los de COCO porque el dataset deriva de COCO/LVIS y los documentamos como decisión propia.

In [ ]:
POS = [["top_left", "top_center", "top_right"],
       ["middle_left", "center", "middle_right"],
       ["bottom_left", "bottom_center", "bottom_right"]]

def propiedades(mask, W, H):
    """Propiedades DERIVADAS de la máscara (las calcula código, no un modelo)."""
    ys, xs = np.nonzero(mask)
    cx, cy, area = float(xs.mean()), float(ys.mean()), len(xs)
    fila, col = min(int(cy / (H / 3)), 2), min(int(cx / (W / 3)), 2)
    tam = "small" if area < 32**2 else ("medium" if area < 96**2 else "large")
    return dict(centroid=[round(cx, 1), round(cy, 1)], area_ratio=area / (W * H), position=POS[fila][col], size=tam)

def relaciones(objs_img, W, H, margen=0.05, iou_min=0.01):
    """Relaciones entre todos los pares de objetos. objs_img: lista de dicts con id, centroid y rle."""
    rels = []
    for a in objs_img:
        for b in objs_img:
            if a["id"] == b["id"]:
                continue
            (ax, ay), (bx, by) = a["centroid"], b["centroid"]
            if ax < bx - margen * W: rels.append(dict(subject=a["id"], predicate="left_of", object=b["id"]))
            if ax > bx + margen * W: rels.append(dict(subject=a["id"], predicate="right_of", object=b["id"]))
            if ay < by - margen * H: rels.append(dict(subject=a["id"], predicate="above", object=b["id"]))
            if ay > by + margen * H: rels.append(dict(subject=a["id"], predicate="below", object=b["id"]))
    if len(objs_img) > 1:
        R = [a_rle_coco(o["rle"]) for o in objs_img]
        M = np.atleast_2d(mask_utils.iou(R, R, [0] * len(R)))
        for i, j in zip(*np.nonzero(M > iou_min)):
            if i != j:
                rels.append(dict(subject=objs_img[i]["id"], predicate="overlaps", object=objs_img[j]["id"]))
    return rels

# Calculamos todo sobre las 120 imágenes de desarrollo
filas, n_objs, n_rels, ejemplo = [], [], [], None
for img_id, d_img in dets_final.groupby("image_id"):
    r = img_meta[img_id]
    objs_img = []
    for k, (_, d) in enumerate(d_img.sort_values("score", ascending=False).iterrows(), 1):
        p = propiedades(rle_decode(d.rle_sam), r.width, r.height)
        objs_img.append(dict(id=f"obj_{k}", category=id2name[d.category_id], rle=d.rle_sam, **p))
        filas.append(p)
    rels = relaciones(objs_img, r.width, r.height)
    n_objs.append(len(objs_img)); n_rels.append(len(rels))
    if img_id == elegidas[1]:
        ejemplo = (objs_img, rels)
props = pd.DataFrame(filas)

gt_size = pd.cut(objs.area, [0, 32**2, 96**2, np.inf], right=False, labels=["small", "medium", "large"])
print("Distribución de size: predicho vs. real")
print(pd.DataFrame({"predicho": props["size"].value_counts(normalize=True), "real (EDA 1.3)": gt_size.value_counts(normalize=True)}).round(2))
print(f"\nObjetos por imagen: máx {max(n_objs)} (límite del schema 300) | relaciones por imagen: media {np.mean(n_rels):.0f}, máx {max(n_rels)} (límite 20000)")

objs_ej, rels_ej = ejemplo
print(f"\nEjemplo ({img_meta[elegidas[1]].fname}): 5 objetos y 6 relaciones")
print(pd.DataFrame(objs_ej)[["id", "category", "centroid", "area_ratio", "position", "size"]].head(5).round(4).to_string(index=False))
print(pd.DataFrame(rels_ej).head(6).to_string(index=False))

### 3.5 Conclusiones de la Fase 3

**RLE verificado.** El formato `{"size": [alto, ancho], "counts": [...]}` aplanado **por columnas** pasó los tres tests: el ejemplo oficial, 200 máscaras al azar y la decodificación de `pycocotools`. Queda resuelta la duda del EDA sobre el orden del RLE.

**Experimento controlado: ¿de dónde sale la máscara?** Mismas 3477 detecciones, misma métrica.

| Fuente de la máscara | P | R | F1 máscaras | TP | FP | FN |
|---|---|---|---|---|---|---|
| Caja rellena (línea base) | 0,470 | 0,383 | 0,422 | 686 | 774 | 1104 |
| **SAM2 (hiera-small) con la caja como prompt** | 0,682 | 0,568 | **0,620** | 1016 | 474 | 774 |

SAM2 sube el F1 de máscaras **20 puntos** sobre la caja: recupera 330 objetos que la caja no alcanza a cubrir con IoU ≥ 0,5. Frente al F1 de cajas (0,660) se pierden sólo 4 puntos. Eso indica que el cuello de botella sigue siendo el **detector**, no la segmentación: si la caja es buena, SAM2 casi siempre acierta la máscara. Ninguna máscara salió vacía (el respaldo con la caja quedó sin usar).

**Propiedades.** La distribución de `size` predicha (49 % / 38 % / 13 %) es parecida a la real del EDA (54 % / 33 % / 13 %). En el ejemplo revisado, posiciones y relaciones coinciden con la imagen: el barrilete en `center` está `above` y `right_of` el arreglo floral.

**Riesgo detectado para la Fase 4:** la imagen más cargada tiene 88 objetos y genera **13 318 relaciones**, cerca del límite de 20 000 del schema. Como crecen con el cuadrado de los objetos, una imagen de test con ~110 objetos lo pasaría. Al armar el SceneJSON vamos a poner un tope: si se supera, se calculan relaciones sólo entre los objetos de mayor confianza.

**Pendiente de la Fase 4:** armar el SceneJSON completo, validarlo con `jsonschema`, guardar la traza por módulo y hacer el **primer envío** para confirmar con la métrica oficial el formato y el orden del RLE.

### 3.6 Atributos derivados y proximidad (agregado en la v3)

Además de las propiedades del schema, cada objeto lleva atributos que se calculan con código a partir de su máscara y de los píxeles de la imagen. Son **derivados**: no los predice un modelo y LVIS no trae ground truth para ellos.

| Campo | Regla |
|---|---|
| `attributes.color` | el color con más píxeles dentro de la máscara (erosionada 1 px para no tomar el fondo), clasificado por reglas HSV en 12 nombres |
| `attributes.color_ratio` | fracción de los píxeles de la máscara que tienen ese color: indica qué tan dominante es |
| `attributes.occlusion` | fracción de la caja del objeto tapada por máscaras de **otros** objetos (estimación de oclusión) |
| `attributes.touches_border` | la caja toca el borde de la imagen, así que el objeto puede estar cortado |
| `extra_relations` (`near`) | separación entre cajas ≤ 2 % de la diagonal de la imagen. Va en un campo aparte porque el schema sólo admite 5 predicados en `relations` |

Las reglas y los umbrales son decisiones propias. Quedan en `PARAMS_ATRIBUTOS`, que también se guarda en la traza de cada imagen.

**Cómo se eligieron los umbrales de color:** no hay ground truth de color, así que usamos como aproximación las clases con un color esperable (brócoli verde, cebra blanca, negra o gris, elefante gris o marrón…). Con las máscaras reales de desarrollo, subir el umbral de saturación de "gris" de 0,15 a 0,25 llevó el promedio de aciertos plausibles de 0,48 a 0,55. Más que eso no se logra con reglas HSV simples: las bananas con luz cálida caen en naranja y el brócoli amarillento en amarillo. Por eso el color es un **atributo aproximado**, y la confianza de una respuesta de color viene acompañada de `color_ratio`.

In [ ]:
PARAMS_ATRIBUTOS = dict(color="HSV, 12 nombres, gris si S < 0.25, máscara erosionada 1 px, hasta 4000 píxeles",
                        near_max_frac_diag=0.02, oclusion="caja cubierta por máscaras de otros objetos", borde_px=2)
COLORES = ["black", "white", "gray", "red", "orange", "brown", "yellow", "green", "cyan", "blue", "purple", "pink"]
C = {c: i for i, c in enumerate(COLORES)}

def nombres_de_color(hsv):
    """hsv: (N, 3) con H, S, V en 0-255 (formato de PIL). Devuelve el índice en COLORES de cada píxel."""
    h, s, v = hsv[:, 0] * (360 / 255), hsv[:, 1] / 255, hsv[:, 2] / 255
    tono = np.select([(h < 12) | (h >= 345), h < 40, h < 70, h < 165, h < 195, h < 255, h < 290],
                     [C["red"], C["orange"], C["yellow"], C["green"], C["cyan"], C["blue"], C["purple"]], default=C["pink"])
    tono = np.where((h >= 10) & (h < 45) & (v < 0.6), C["brown"], tono)   # naranja oscuro = marrón
    gris = s < 0.25                                                       # poca saturación = acromático
    return np.where(v < 0.2, C["black"], np.where(gris & (v > 0.8), C["white"], np.where(gris, C["gray"], tono)))

def erosionar(m):
    """Erosión de 1 píxel (vecindad en cruz). Si la máscara desaparece, devuelve la original."""
    e = m.copy()
    e[1:, :] &= m[:-1, :]; e[:-1, :] &= m[1:, :]; e[:, 1:] &= m[:, :-1]; e[:, :-1] &= m[:, 1:]
    return e if e.any() else m

def color_dominante(hsv_img, m, max_px=4000):
    ys, xs = np.nonzero(erosionar(m))
    if len(ys) > max_px:                                   # submuestreo determinista
        idx = np.linspace(0, len(ys) - 1, max_px).astype(int)
        ys, xs = ys[idx], xs[idx]
    cuenta = np.bincount(nombres_de_color(hsv_img[ys, xs].astype(float)), minlength=len(COLORES))
    k = int(cuenta.argmax())
    return COLORES[k], round(float(cuenta[k] / cuenta.sum()), 3)

def relaciones_near(objetos, W, H):
    """Pares cuya separación entre cajas es ≤ 2 % de la diagonal. Simétrica: se guarda una vez por par."""
    diag = (W ** 2 + H ** 2) ** 0.5
    rels = []
    for i, a in enumerate(objetos):
        for b in objetos[i + 1:]:
            dx = max(0.0, b["bbox"][0] - a["bbox"][2], a["bbox"][0] - b["bbox"][2])
            dy = max(0.0, b["bbox"][1] - a["bbox"][3], a["bbox"][1] - b["bbox"][3])
            d = (dx ** 2 + dy ** 2) ** 0.5 / diag
            if d <= PARAMS_ATRIBUTOS["near_max_frac_diag"]:
                rels.append(dict(subject=a["id"], predicate="near", object=b["id"], distance=round(d, 4)))
    return rels

def agregar_atributos(escena, im):
    """Agrega `attributes` a cada objeto y `extra_relations` a la escena. Usa sólo la imagen y el SceneJSON."""
    W, H = escena["image"]["width"], escena["image"]["height"]
    hsv = np.asarray(im.convert("HSV"))
    objetos = escena["objects"]
    masks = [rle_decode(o["mask"]) for o in objetos]
    cobertura = np.zeros((H, W), np.uint16)                # cuántas máscaras cubren cada píxel
    for m in masks:
        cobertura += m
    b = PARAMS_ATRIBUTOS["borde_px"]
    for o, m in zip(objetos, masks):
        x1, y1 = int(o["bbox"][0]), int(o["bbox"][1])
        x2, y2 = max(int(np.ceil(o["bbox"][2])), x1 + 1), max(int(np.ceil(o["bbox"][3])), y1 + 1)
        mc = m[y1:y2, x1:x2]
        tapado = ((cobertura[y1:y2, x1:x2] - mc) > 0) & ~mc   # píxeles de la caja cubiertos por otro objeto
        color, ratio = color_dominante(hsv, m)
        o["attributes"] = dict(color=color, color_ratio=ratio, occlusion=round(float(tapado.mean()), 3),
                               touches_border=bool(o["bbox"][0] <= b or o["bbox"][1] <= b
                                                   or o["bbox"][2] >= W - b or o["bbox"][3] >= H - b))
    escena["extra_relations"] = relaciones_near(objetos, W, H)
    return escena["extra_relations"]

# --- Demostración sobre las detecciones de desarrollo de la Fase 3 ---
# Armamos una escena mínima por imagen con las detecciones finales y sus máscaras de SAM2.
t0 = time.time()
filas_attr, n_near = [], []
for img_id, d_img in dets_final.groupby("image_id"):
    r = img_meta[img_id]
    esc_tmp = dict(image=dict(id=r.fname, width=r.width, height=r.height),
                   objects=[dict(id=f"obj_{k}", category_id=int(d.category_id), confidence=float(d.score),
                                 bbox=[d.bbox[0], d.bbox[1], d.bbox[0] + d.bbox[2], d.bbox[1] + d.bbox[3]], mask=d.rle_sam)
                            for k, d in enumerate(d_img.sort_values("score", ascending=False).itertuples(), 1)])
    n_near.append(len(agregar_atributos(esc_tmp, Image.open(DEV / "images" / r.fname).convert("RGB"))))
    filas_attr += [dict(image_id=img_id, clase=id2name[o["category_id"]], **o["attributes"]) for o in esc_tmp["objects"]]
attr = pd.DataFrame(filas_attr)
print(f"Atributos de {len(attr)} objetos en {len(n_near)} imágenes ({time.time() - t0:.0f} s) | relaciones near por imagen: media {np.mean(n_near):.0f}, máx {max(n_near)}")
print("\nDistribución de colores:", attr.color.value_counts(normalize=True).round(2).to_dict())
print("Oclusión media: {:.2f} | objetos con oclusión > 0,3: {:.0%} | tocan el borde: {:.0%}".format(
    attr.occlusion.mean(), (attr.occlusion > 0.3).mean(), attr.touches_border.mean()))

# Chequeo de plausibilidad del color: clases con un color esperable (aproximación, no es ground truth)
ESPERADO = {"broccoli": {"green"}, "banana": {"yellow", "green"}, "zebra": {"black", "white", "gray"},
            "elephant": {"gray", "brown"}, "giraffe": {"brown", "orange", "yellow"}}
plaus = attr[attr.clase.isin(ESPERADO)].assign(plausible=lambda d: [c in ESPERADO[k] for k, c in zip(d.clase, d.color)])
print("\nPlausibilidad del color en clases con color esperable:")
print(plaus.groupby("clase").agg(objetos=("color", "size"), plausible=("plausible", "mean"),
                                  colores=("color", lambda s: s.value_counts().head(3).to_dict())).round(2).to_string())

## 4. SceneJSON, traza y primer envío

Juntamos las Fases 2 y 3 en **una sola función `procesar_imagen()`** que recibe sólo la imagen y devuelve el SceneJSON. Es la pipeline que se evalúa: no lee anotaciones, así que se puede probar con fotos nuevas.

```
imagen -> detector (OWLv2, 2 clases por caja) -> NMS + umbral 0,20 -> SAM2 -> propiedades -> relaciones -> validación -> SceneJSON
```

**Topes para respetar el schema:** como máximo 300 objetos (los de mayor confianza). Si las relaciones pasan de 20 000, se recalculan con el 90 % de los objetos más confiables, hasta quedar debajo del límite.

**Traza:** cada módulo registra qué corrió, con qué parámetros, cuánto tardó, si falló y cuántas salidas produjo. Si una imagen falla, la traza dice en qué módulo, y esa fila sale con una escena vacía pero válida en lugar de romper el envío.

### 4.1 La pipeline completa

In [ ]:
import jsonschema

schema = json.load(open(ROOT / "scene_schema.json"))
validador = jsonschema.Draft202012Validator(schema)
MAX_OBJ = schema["properties"]["objects"]["maxItems"]      # 300
MAX_REL = schema["properties"]["relations"]["maxItems"]    # 20000
print("Topes del schema:", MAX_OBJ, "objetos y", MAX_REL, "relaciones")

# v6: configuración de percepción y su hash. Va en el campo `trace` de cada SceneJSON, junto con el RUN_ID
CONFIG_PERCEPCION = dict(detector=DET_CONFIG, segmentador=SAM_ID, caja_desde_mascara=bool(CAJA_DESDE_MASCARA),
                         propiedades=dict(bbox="xyxy", size="COCO 32^2 / 96^2", grilla="3x3 por centroide"),
                         relaciones=dict(margen=0.05, iou_overlaps=0.01, max_rel=MAX_REL), atributos=PARAMS_ATRIBUTOS)
TRAZA_ESCENA = dict(run_id=RUN_ID, config_percepcion=huella(CONFIG_PERCEPCION))
print("Traza de cada escena:", TRAZA_ESCENA)

def paso(traza, image_id, modulo, params, fn):
    """Ejecuta un módulo y deja registro en la traza (tiempo, parámetros, error, cantidad de salidas)."""
    t0, inicio = time.time(), time.strftime("%H:%M:%S")
    fila = dict(run_id=RUN_ID, image_id=image_id, modulo=modulo, params=json.dumps(params, ensure_ascii=False), inicio=inicio)
    try:
        out = fn()
        n = len(out[0]) if isinstance(out, tuple) else (len(out) if hasattr(out, "__len__") else 1)
        traza.append(dict(fila, duracion_ms=round((time.time() - t0) * 1000, 1), ok=True, n_salidas=n, error=""))
        return out
    except Exception as e:
        traza.append(dict(fila, duracion_ms=round((time.time() - t0) * 1000, 1), ok=False, n_salidas=0, error=repr(e)[:300]))
        raise

def caja_xyxy(b, W, H):
    """[x, y, ancho, alto] (formato interno, COCO) -> [xmin, ymin, xmax, ymax] (formato que exige la métrica)."""
    x, y, w, h = [float(v) for v in b]
    return [round(max(0.0, x), 2), round(max(0.0, y), 2), round(min(float(W), x + w), 2), round(min(float(H), y + h), 2)]

def caja_valida(b):
    return 0 <= b[0] < b[2] and 0 <= b[1] < b[3]

def relaciones_con_tope(objetos, W, H):
    """Si hay demasiadas relaciones, recalcula con el 90 % de los objetos más confiables hasta respetar el límite."""
    base = [dict(o, rle=o["mask"]) for o in objetos]  # ya vienen ordenados por confianza
    n = len(base)
    while True:
        rels = relaciones(base[:n], W, H)
        if len(rels) <= MAX_REL:
            return rels, n
        n = int(n * 0.9)

def escena_vacia(image_id, W, H):
    return dict(image=dict(id=image_id, width=W, height=H), objects=[], relations=[], questions=[], questions_llm=[],
                trace=TRAZA_ESCENA)

def procesar_imagen(path, image_id, traza, linaje=None):
    """PIPELINE COMPLETA. Recibe SÓLO la imagen y el vocabulario: nunca lee anotaciones."""
    im = Image.open(path).convert("RGB")
    W, H = im.size
    try:
        crudas = paso(traza, image_id, "detector", dict(DET_CONFIG, umbral_crudo=0.05),
                      lambda: pd.DataFrame(detectar_multi(path, umbral=0.05, k_max=K_MAX)))
        if len(crudas):
            crudas["image_id"] = image_id
            finales = paso(traza, image_id, "nms_umbral", dict(nms_iou=0.5, umbral=UMBRAL, max_obj=MAX_OBJ),
                           lambda: (lambda d: d[d.score >= UMBRAL])(aplicar_nms(crudas))
                           .sort_values("score", ascending=False).head(MAX_OBJ).reset_index(drop=True))
        else:
            finales = crudas
        masks = paso(traza, image_id, "segmentador", dict(modelo=SAM_ID),
                     lambda: segmentar(im, finales.bbox.tolist()) if len(finales) else [])
        objetos = []
        def armar_objetos():
            for d, m in zip(finales.itertuples(), masks):
                respaldo = m.sum() == 0
                if respaldo:
                    m = caja_a_mascara(d.bbox, H, W)   # respaldo: la caja como máscara
                if m.sum() == 0:
                    continue
                caja = caja_xyxy(caja_de_mascara(m) if CAJA_DESDE_MASCARA else d.bbox, W, H)   # v4: decisión de 3.3b
                if not caja_valida(caja):
                    continue                           # caja degenerada (ancho o alto 0 tras recortar al borde)
                objetos.append(dict(id=f"obj_{len(objetos) + 1}", category_id=int(d.category_id),
                                    category_name=id2name[d.category_id], confidence=round(float(d.score), 4),
                                    bbox=caja, mask=rle_encode(m), **propiedades(m, W, H)))
                if linaje is not None:   # v6: de qué detección sale cada objeto y qué pasó en cada paso
                    linaje.append(dict(run_id=RUN_ID, image_id=image_id, obj_id=objetos[-1]["id"], clase=id2name[d.category_id],
                                       score=round(float(d.score), 4), rango_clase=int(d.rango), fuente=getattr(d, "fuente", "imagen"),
                                       bbox_detector=[round(float(v), 1) for v in d.bbox], segmentador=SAM_ID,
                                       mascara_px=int(m.sum()), mascara_respaldo=bool(respaldo),
                                       relleno_caja=round(float(m.sum()) / max(1.0, d.bbox[2] * d.bbox[3]), 3)))
            return objetos
        paso(traza, image_id, "propiedades", dict(bbox="xyxy", size="COCO 32^2 / 96^2", grilla="3x3 por centroide"), armar_objetos)
        rels, n_usados = paso(traza, image_id, "relaciones", dict(margen=0.05, iou_overlaps=0.01, max_rel=MAX_REL),
                              lambda: relaciones_con_tope(objetos, W, H))
        escena = dict(image=dict(id=image_id, width=W, height=H), objects=objetos, relations=rels, questions=[], questions_llm=[],
                      trace=TRAZA_ESCENA)
        paso(traza, image_id, "atributos", PARAMS_ATRIBUTOS, lambda: agregar_atributos(escena, im))   # v3: 3.6
        errores = paso(traza, image_id, "validacion", dict(schema="scene_schema.json", regla_bbox="0<=xmin<xmax, 0<=ymin<ymax"),
                       lambda: [e.message for e in validador.iter_errors(escena)]
                               + [f"bbox inválida en {o['id']}" for o in objetos if not caja_valida(o["bbox"])])
        if errores:
            raise ValueError(f"SceneJSON inválido: {errores[:2]}")
        return escena
    except Exception as e:
        print(f"[{image_id}] falló, se envía una escena vacía válida: {repr(e)[:150]}")
        if linaje is not None:   # la escena sale vacía: su linaje también
            linaje[:] = [fila for fila in linaje if fila["image_id"] != image_id]
        return escena_vacia(image_id, W, H)

# Prueba con una imagen de desarrollo
traza_prueba = []
r = img_meta[elegidas[1]]
esc = procesar_imagen(DEV / "images" / r.fname, Path(r.fname).stem, traza_prueba)
texto = json.dumps(esc, separators=(",", ":"))
print(f"{len(esc['objects'])} objetos | {len(esc['relations'])} relaciones | JSON de {len(texto) / 1024:.0f} KB | válido: {validador.is_valid(esc)}")
print("Cajas que cumplen 0 <= xmin < xmax y 0 <= ymin < ymax:", sum(caja_valida(o["bbox"]) for o in esc["objects"]), "de", len(esc["objects"]))
print(pd.DataFrame(traza_prueba)[["modulo", "duracion_ms", "ok", "n_salidas"]].to_string(index=False))
print(json.dumps({**esc["objects"][0], "mask": "..."}, ensure_ascii=False))

### 4.2 Armado del archivo de envío (80 filas)
Corremos `procesar_imagen()` sobre las 80 imágenes de `sample_submission.csv`, en el mismo orden: 60 de test y 20 de desarrollo. Antes de guardar verificamos que haya 80 filas en el mismo orden, que todos los SceneJSON pasen el schema y cuánto pesa el archivo.

**Chequeo de serialización:** en las 20 filas de desarrollo volvemos a leer el CSV, reconstruimos las predicciones desde el JSON y las medimos. Tienen que dar exactamente lo mismo que las detecciones de la Fase 3 sobre esas imágenes; si no, algo se rompió al escribir el JSON.

In [ ]:
rutas = {p.stem: p for p in ROOT.rglob("*.jpg")}
faltan = [i for i in sub.image_id if i not in rutas]
print("Imágenes del envío sin archivo encontrado:", faltan or "ninguna")
print("test_images.csv:", list(pd.read_csv(TEST / "test_images.csv").columns))

traza, filas_env, linaje = [], [], []
t0 = time.time()
for iid in sub.image_id:
    esc = procesar_imagen(rutas[iid], iid, traza, linaje)
    filas_env.append(dict(image_id=iid, SceneJSON=json.dumps(esc, separators=(",", ":"), ensure_ascii=False)))
envio = pd.DataFrame(filas_env)
traza = pd.DataFrame(traza)
print(f"\n80 imágenes procesadas en {(time.time() - t0) / 60:.1f} min")

# Verificaciones antes de guardar
assert list(envio.columns) == ["image_id", "SceneJSON"] and list(envio.image_id) == list(sub.image_id), "filas u orden distintos al sample"
escenas = [json.loads(s) for s in envio.SceneJSON]
print("Filas:", len(envio), "| mismo orden que el sample: True")
print("SceneJSON válidos según el schema:", sum(validador.is_valid(e) for e in escenas), "de", len(escenas))
cajas_malas = sum(not caja_valida(o["bbox"]) for e in escenas for o in e["objects"])
print("Cajas que NO cumplen 0 <= xmin < xmax y 0 <= ymin < ymax:", cajas_malas)
assert cajas_malas == 0, "hay cajas inválidas: Kaggle rechazaría el envío"
print("Escenas vacías (fallas):", sum(len(e["objects"]) == 0 for e in escenas))
n_o = [len(e["objects"]) for e in escenas]; n_r = [len(e["relations"]) for e in escenas]
print(f"Objetos por imagen: media {np.mean(n_o):.0f}, máx {max(n_o)} | relaciones: media {np.mean(n_r):.0f}, máx {max(n_r)}")
recortes = traza[(traza.modulo == "relaciones")].n_salidas.ge(MAX_REL).sum()
print("Imágenes donde se aplicó el tope de relaciones:", recortes)

envio.to_csv(SALIDA / "submission.csv", index=False)
envio.to_csv(SALIDA / "solution.csv", index=False)   # el Overview lo llama solution.csv; dejamos ambos nombres
traza.to_csv(SALIDA / "traza_envio.csv", index=False)
linaje = pd.DataFrame(linaje)                        # v6: una fila por objeto de cada SceneJSON
linaje.to_csv(SALIDA / "traza_objetos.csv", index=False)
print(f"Linaje: {len(linaje)} objetos en traza_objetos.csv | por fuente: {linaje.fuente.value_counts().to_dict()}")
marcar_fase("4 SceneJSON y envío")
print(f"submission.csv: {(SALIDA / 'submission.csv').stat().st_size / 1e6:.1f} MB")

print("\nTraza: tiempo medio por módulo y fallas")
print(traza.groupby("modulo", sort=False).agg(ms_medio=("duracion_ms", "mean"), fallas=("ok", lambda s: int((~s).sum()))).round(1))

# Chequeo de serialización en las 20 filas de desarrollo (el JSON trae xyxy; la métrica local usa xywh)
leido = pd.read_csv(SALIDA / "submission.csv")
stem2id = {Path(r.fname).stem: r.id for r in img_meta.values()}
desde_json = pd.DataFrame([dict(image_id=stem2id[e["image"]["id"]], category_id=o["category_id"], score=o["confidence"],
                                bbox=[o["bbox"][0], o["bbox"][1], o["bbox"][2] - o["bbox"][0], o["bbox"][3] - o["bbox"][1]],
                                rle=o["mask"])
                           for e in map(json.loads, leido.SceneJSON) if e["image"]["id"] in stem2id for o in e["objects"]])
ids_dev = set(desde_json.image_id)
fase3 = dets_final[dets_final.image_id.isin(ids_dev)]
print(f"\nSerialización en {len(ids_dev)} imágenes de desarrollo (F1 sobre el total, por eso es bajo; lo que importa es que coincidan):")
print("  cajas    | Fase 3:", round(evaluar_cajas(fase3)["F1"], 4), "| leído del CSV:", round(evaluar_cajas(desde_json)["F1"], 4))
print("  máscaras | Fase 3:", round(evaluar_mascaras(fase3, "rle_sam")["F1"], 4), "| leído del CSV:", round(evaluar_mascaras(desde_json, "rle")["F1"], 4))

### 4.3 Conclusiones de la Fase 4

| Verificación | v2 (OWLv2 base, SAM2 small) | v5 y v6 (configuración final: ventanas + sinónimos, SAM2 large) |
|---|---|---|
| Filas y orden iguales a `sample_submission.csv` | 80 de 80, mismo orden | 80 de 80, mismo orden |
| SceneJSON válidos según `scene_schema.json` | 80 de 80 | 80 de 80 |
| Imágenes que fallaron (escena vacía) | 0 | 0 |
| Objetos por imagen: media / máximo (límite 300) | — / 81 | 30,3 / 112 |
| Relaciones por imagen: máximo (límite 20 000) | 11 258 | 17 206 |
| Imágenes donde se aplicó el tope de relaciones | 0 | 0 (el tope queda como protección) |
| Serialización: métrica antes y después de escribir el CSV | idéntica en cajas y máscaras | idéntica |
| Tamaño de `submission.csv` | 11,5 MB | 14,8 MB |
| Tiempo total de la pipeline | 1,2 min para 80 imágenes (≈ 0,9 s por imagen) | 4,9 min (≈ 3,7 s por imagen) |

**Traza.** En la configuración final el detector es el módulo más caro: 2,4 s por imagen, porque hace 5 pasadas de OWLv2 (la imagen completa y 4 ventanas). Le sigue SAM2 large, con 1,0 s. Ningún módulo falló. La imagen con más relaciones llegó a 17 206, cerca del límite del schema: el tope de 4.1 no se activó, pero es la protección que evita perder la fila en una imagen más cargada. La traza completa queda en `traza_envio.csv`, una fila por imagen y módulo; desde la v6 cada fila lleva el `RUN_ID` y se complementa con el linaje por objeto (`traza_objetos.csv`, Fase 8).

**Por qué el chequeo de serialización da un F1 bajo:** la métrica se calcula sobre las 120 imágenes, pero el envío trae sólo 20 de desarrollo. Las otras 100 cuentan como no detectadas. Lo que importa es que el valor sea **idéntico** antes y después de escribir el CSV: confirma que el JSON no pierde ni deforma información.

**Primer envío.** `questions` y `questions_llm` van vacías (el motor de preguntas es la Fase 5), así que el 10 % de QA del puntaje queda en 0. El objetivo de este envío es validar el formato con la métrica oficial y obtener la primera referencia del leaderboard.

### 4.4 Lo que encontró el primer envío
El primer envío (v1) fue **rechazado** por Kaggle con el mensaje: *"bbox debe cumplir 0 <= xmin < xmax y 0 <= ymin < ymax"*.

**Causa:** la métrica oficial espera la caja como `[xmin, ymin, xmax, ymax]` (dos esquinas). Nosotros enviábamos `[x, y, ancho, alto]`, el formato COCO de `annotations.json`. El ejemplo `[0, 0, 1, 1]` de `sample_submission.csv` era ambiguo: se lee igual en los dos formatos.

**Corrección:** internamente seguimos usando `[x, y, w, h]` (la métrica local y las anotaciones están en ese formato), pero `procesar_imagen()` convierte a `[xmin, ymin, xmax, ymax]` al escribir el SceneJSON, recortando al borde de la imagen. Agregamos esa regla a la validación de cada escena y un `assert` antes de guardar el CSV, así un error así no vuelve a llegar a Kaggle.

**Lección:** la validación con `jsonschema` no alcanza. El schema sólo exige "4 números", no su significado. Por eso conviene enviar temprano: la métrica oficial detecta errores de contrato que ninguna prueba local ve.

## 5. Preguntas en modo determinista

Objetivo de la fase: responder las consignas del benchmark **leyendo sólo el SceneJSON predicho**. No se usa LLM, y el código decide cada conteo, filtro y relación.

```
pregunta + campos estructurados ─► consulta intermedia (JSON validado) ─► motor determinista ─► respuesta normalizada + evidencia
```

La consulta intermedia es el contrato entre "entender la pregunta" y "responderla". En esta fase la arma un parser con reglas. En la Fase 6 la va a armar un LLM, y la va a ejecutar **el mismo motor**.

Operaciones soportadas:

| Intención | Salida normalizada | Cómo se resuelve |
|---|---|---|
| `count` | entero | cantidad de objetos de la clase |
| `existence` | `yes` / `no` | hay al menos un objeto de la clase |
| `position` | una de las 9 zonas | `position` de la instancia elegida |
| `size` | `small` / `medium` / `large` | `size` de la instancia elegida |
| `relation` | `yes` / `no` | la tripleta está en `relations` (o en `extra_relations` si es `near`) |
| `attribute` | nombre de color | `attributes.color` de la instancia elegida |
| `compare` | `yes` / `no` | compara las cantidades de dos clases |
| `description` | texto | plantilla con conteos, zonas y el objeto más grande |

Casos especiales, siempre explícitos:
- **Ambigua:** hay varias instancias de la clase. Se responde por la de mayor confianza, `status = "ambiguous"`, las demás van en `alternatives` y la confianza se reparte entre ellas.
- **Información insuficiente:** la clase no se detectó. En posición, tamaño, atributo y relación se responde `unknown` en lugar de inventar.
- **No soportada:** clase fuera del vocabulario, operación inexistente o atributo no calculado. Se responde `unsupported`.

Regla anti-fuga: `ejecutar()` recibe la consulta y el SceneJSON predicho. Las anotaciones aparecen recién en 5.5, sólo para medir.

### 5.1 La consulta intermedia y su validación

La misma validación vale para los dos modos. Primero se chequea la forma con JSON Schema. Después, la semántica: cantidad de clases según la intención, clases dentro del vocabulario, predicado o atributo presente. Una consulta con problemas no se ejecuta: se responde `unsupported` y se informa el motivo.

In [ ]:
import re
import unicodedata

name2id = dict(zip(cats.category_name, cats.category_id))
INTENCIONES = ["count", "existence", "position", "size", "relation", "attribute", "compare", "description", "unsupported"]
PREDICADOS = ["left_of", "right_of", "above", "below", "overlaps", "near"]
SALIDA_ESPERADA = dict(count="integer", existence="boolean", position="position_3x3", size="size_class", relation="boolean",
                       attribute="color", compare="boolean", description="free_text", unsupported="none")
N_CLASES = dict(count=1, existence=1, position=1, size=1, attribute=1, relation=2, compare=2, description=0, unsupported=0)

QUERY_SCHEMA = {
    "type": "object",
    "required": ["intent", "classes"],
    "additionalProperties": False,
    "properties": {
        "intent": {"enum": INTENCIONES},
        "classes": {"type": "array", "items": {"type": "string", "minLength": 1}, "maxItems": 2},
        "predicate": {"enum": PREDICADOS + [None]},
        "attribute": {"enum": ["color", None]},
        "comparator": {"enum": ["more", "fewer", "same", None]},
        "output": {"enum": sorted(set(SALIDA_ESPERADA.values()))},
        "reason": {"type": "string"},
    },
}
validador_query = jsonschema.Draft202012Validator(QUERY_SCHEMA)

def validar_query(query):
    """Valida forma y semántica. Devuelve (consulta normalizada, problemas). No mira la imagen ni la escena."""
    problemas = [e.message for e in validador_query.iter_errors(query)]
    if problemas:
        return None, problemas
    q = {**dict(predicate=None, attribute=None, comparator=None), **query}
    q["output"] = SALIDA_ESPERADA[q["intent"]]
    if q["intent"] == "unsupported":
        return q, [q.get("reason") or "consulta fuera de alcance"]
    if len(q["classes"]) != N_CLASES[q["intent"]]:
        problemas.append(f"'{q['intent']}' necesita {N_CLASES[q['intent']]} clase(s) y recibió {len(q['classes'])}")
    fuera = [c for c in q["classes"] if c not in name2id]
    if fuera:
        problemas.append(f"clase fuera del vocabulario: {', '.join(fuera)}")
    if q["intent"] == "relation" and q["predicate"] is None:
        problemas.append("no se reconoce la relación espacial")
    if q["intent"] == "attribute" and q["attribute"] is None:
        problemas.append("falta el atributo")
    if q["intent"] == "compare" and q["comparator"] is None:
        problemas.append("falta el tipo de comparación")
    return q, problemas

# Pruebas de la validación
for prueba in [dict(intent="count", classes=["doorknob"]),
               dict(intent="count", classes=["unicorn"]),
               dict(intent="relation", classes=["clock", "ring"]),
               dict(intent="size"),
               dict(intent="count", classes=["car"], color="red")]:
    q, prob = validar_query(prueba)
    print(f"{json.dumps(prueba, ensure_ascii=False):60s} -> {'OK' if not prob else prob}")

### 5.2 Parser determinista

En modo determinista la cátedra entrega `type` y `class_en` junto con la pregunta, así que la consulta sale casi directa de esos campos. Hay un dato que **no viene estructurado**: en las preguntas de relación, cuál es la relación ("por encima de", "a la derecha de"…). Ese predicado se lee del texto con expresiones regulares, sin LLM. Lo mismo vale para el comparador de las preguntas `compare` ("más", "menos").

In [ ]:
def sin_acentos(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower()) if unicodedata.category(c) != "Mn")

PATRONES_PREDICADO = [(r"por encima d|arriba d|encima d", "above"), (r"por debajo d|debajo d|abajo d", "below"),
                      (r"a la izquierda d", "left_of"), (r"a la derecha d", "right_of"),
                      (r"superpone|solapa", "overlaps"), (r"cerca d|junto a|al lado d", "near")]
PATRONES_COMPARADOR = [(r"\bmas\b", "more"), (r"\bmenos\b", "fewer"), (r"misma cantidad|igual cantidad|tant[oa]s", "same")]

def buscar_patron(texto, patrones):
    t = sin_acentos(texto)
    return next((valor for patron, valor in patrones if re.search(patron, t)), None)

def query_determinista(p):
    """Consulta a partir de los campos estructurados (type, class_en). Del texto sólo se lee el predicado o el comparador."""
    intent = p["type"] if p["type"] in INTENCIONES else "unsupported"
    q = dict(intent=intent, classes=list(p.get("class_en") or []))
    if intent == "relation":
        q["predicate"] = p.get("predicate") or buscar_patron(p["question"], PATRONES_PREDICADO)
    elif intent == "attribute":
        q["attribute"] = p.get("attribute") or "color"
    elif intent == "compare":
        q["comparator"] = p.get("comparator") or buscar_patron(p["question"], PATRONES_COMPARADOR)
    elif intent == "unsupported":
        q["classes"] = []
        q["reason"] = p.get("reason") or f"tipo de pregunta no soportado: {p['type']}"
    return q

# Las 10 preguntas de relación del benchmark: el predicado tiene que salir del texto en todas
rel = qs[qs.type == "relation"]
predicados = rel.question.map(lambda t: buscar_patron(t, PATRONES_PREDICADO))
print("Predicados reconocidos:", predicados.notna().sum(), "de", len(rel))
print(pd.DataFrame({"pregunta": rel.question.str.slice(0, 75), "predicado": predicados}).to_string(index=False))

### 5.3 Motor de ejecución

`ejecutar(consulta, escena)` resuelve la consulta con código sobre el SceneJSON y devuelve los campos que pide el schema:

- `answer`: frase legible.
- `normalized_answer`: el valor que se evalúa.
- `evidence`: los `id` de los objetos de **esa misma escena** que sustentan la respuesta.
- `confidence`: sale de las confianzas del detector, no de un modelo de lenguaje:
  - conteo: el promedio;
  - existencia: el máximo;
  - relación: el mínimo de los dos objetos;
  - si la pregunta es ambigua, se multiplica por la parte de la confianza total que se lleva la instancia elegida;
  - "no se detectó": 0,5, porque no detectar un objeto no prueba que no esté (EDA 1.5).

Además se agregan `status` (`ok`, `ambiguous`, `insufficient_information` o `unsupported`) y `alternatives` (las otras instancias candidatas). El schema permite campos adicionales.

In [ ]:
CONF_AUSENCIA = 0.5   # no detectar no prueba ausencia (EDA 1.5): confianza media, decisión documentada

def instancias(escena, clase):
    cid = name2id[clase]
    return sorted((o for o in escena["objects"] if o["category_id"] == cid), key=lambda o: -o["confidence"])

def elegir(objs):
    """La instancia de mayor confianza. Si hay varias, se marca ambigua y se reparte la confianza."""
    top = objs[0]
    return top, len(objs) > 1, top["confidence"] / sum(o["confidence"] for o in objs)

def respuesta(texto, normalizada, evidencia, conf, estado, **extra):
    return dict(answer=texto, normalized_answer=normalizada, evidence=list(dict.fromkeys(evidencia)),
                confidence=round(float(np.clip(conf, 0, 1)), 4), status=estado, **extra)

def sin_info(motivo):
    return respuesta(f"{motivo}: no hay información suficiente para responder.", "unknown", [], 0.0, "insufficient_information")

def describir(escena, max_clases=6):
    """Descripción factual por plantilla: sólo cuenta y ubica lo que está en el SceneJSON."""
    objs = escena["objects"]
    if not objs:
        return respuesta("No se detectaron objetos en la escena.", "empty_scene", [], CONF_AUSENCIA, "insufficient_information")
    por_clase = defaultdict(list)
    for o in objs:
        por_clase[id2name[o["category_id"]]].append(o)
    orden = sorted(por_clase.items(), key=lambda kv: (-len(kv[1]), -kv[1][0]["confidence"]))
    partes, evid = [], []
    for nombre, lista in orden[:max_clases]:
        zona = Counter(o["position"] for o in lista).most_common(1)[0][0]
        partes.append(f"{len(lista)} {nombre} (sobre todo en {zona})" if len(lista) > 1 else f"1 {nombre} ({zona})")
        evid += [o["id"] for o in lista]
    grande = max(objs, key=lambda o: o["area_ratio"])
    evid.append(grande["id"])
    texto = (f"Se detectaron {len(objs)} objetos de {len(por_clase)} clases. Lo más frecuente: {', '.join(partes)}. "
             f"El objeto que más área ocupa es {id2name[grande['category_id']]} ({grande['position']}, {grande['size']}).")
    usados = {o["id"]: o["confidence"] for o in objs if o["id"] in set(evid)}
    return respuesta(texto, texto, evid, np.mean(list(usados.values())), "ok")

def ejecutar(query, escena):
    """Motor determinista: resuelve la consulta usando SÓLO el SceneJSON predicho."""
    q, problemas = validar_query(query)
    if problemas:
        return respuesta(f"No puedo responder con la información de la escena: {'; '.join(problemas)}.",
                         "unsupported", [], 0.0, "unsupported")
    it = q["intent"]
    if it == "description":
        return describir(escena)
    grupos = [instancias(escena, c) for c in q["classes"]]
    clase = q["classes"][0]

    if it in ("count", "existence", "compare"):
        todos = [o for g in grupos for o in g]
        ids, confs = [o["id"] for o in todos], [o["confidence"] for o in todos]
        conf = np.mean(confs) if confs else CONF_AUSENCIA
        n = len(grupos[0])
        if it == "count":
            return respuesta(f"Hay {n} instancia(s) de '{clase}' en la escena.", n, ids, conf, "ok")
        if it == "existence":
            return respuesta(f"Sí, hay {n} instancia(s) de '{clase}'." if n else f"No se detectó ninguna instancia de '{clase}'.",
                             "yes" if n else "no", ids, max(confs) if n else CONF_AUSENCIA, "ok")
        m = len(grupos[1])
        cumple = {"more": n > m, "fewer": n < m, "same": n == m}[q["comparator"]]
        return respuesta(f"{'Sí' if cumple else 'No'}: hay {n} '{clase}' y {m} '{q['classes'][1]}'.",
                         "yes" if cumple else "no", ids, conf, "ok")

    if it in ("position", "size", "attribute"):
        if not grupos[0]:
            return sin_info(f"No se detectó ninguna instancia de '{clase}'")
        o, ambigua, peso = elegir(grupos[0])
        valor = o.get("attributes", {}).get(q["attribute"]) if it == "attribute" else o[it]
        if valor is None:
            return respuesta(f"El atributo '{q['attribute']}' no está disponible en la escena.", "unsupported", [], 0.0, "unsupported")
        texto = {"position": f"La instancia de '{clase}' está en la zona {valor}.",
                 "size": f"La instancia de '{clase}' es de tamaño {valor}.",
                 "attribute": f"La instancia de '{clase}' es de color {valor}."}[it]
        if ambigua:
            texto += f" Hay {len(grupos[0])} instancias: respondo por la de mayor confianza ({o['id']})."
        return respuesta(texto, valor, [o["id"]], o["confidence"] * (peso if ambigua else 1),
                         "ambiguous" if ambigua else "ok", alternatives=[x["id"] for x in grupos[0][1:]])

    # relation
    a_objs, b_objs = grupos
    if q["classes"][0] == q["classes"][1]:
        b_objs = a_objs[1:]                     # misma clase: se comparan sus dos instancias más confiables
    if not a_objs or not b_objs:
        return sin_info(f"No se detectó ninguna instancia de '{q['classes'][0] if not a_objs else q['classes'][1]}'")
    a, amb_a, pa = elegir(a_objs)
    b, amb_b, pb = elegir(b_objs)
    fuente = escena.get("extra_relations", []) if q["predicate"] == "near" else escena["relations"]
    tripletas = {(r["subject"], r["predicate"], r["object"]) for r in fuente}
    hay = (a["id"], q["predicate"], b["id"]) in tripletas
    if q["predicate"] in ("near", "overlaps"):  # simétricas
        hay = hay or (b["id"], q["predicate"], a["id"]) in tripletas
    conf = min(a["confidence"], b["confidence"]) * (pa if amb_a else 1) * (pb if amb_b else 1)
    texto = (f"{'Sí' if hay else 'No'}: en la escena, '{q['classes'][0]}' ({a['id']}) {'' if hay else 'no '}cumple "
             f"{q['predicate']} respecto de '{q['classes'][1]}' ({b['id']}).")
    return respuesta(texto, "yes" if hay else "no", [a["id"], b["id"]], conf, "ambiguous" if (amb_a or amb_b) else "ok",
                     alternatives=[x["id"] for x in a_objs[1:] + b_objs[1:]])

### 5.4 Respuestas del benchmark (20 imágenes de desarrollo)

Cada consigna de `questions_dev.jsonl` pasa por el parser y el motor, sobre el SceneJSON que produjo la pipeline en la Fase 4. Cada respuesta queda en la traza, con su consulta, su tiempo y su estado. El ítem conserva `question_id`, `type` y `question` tal como los publica la cátedra, y agrega `query` para que se pueda auditar.

In [ ]:
escena_por_id = {e["image"]["id"]: e for e in escenas}
qs["image_key"] = qs.image_file.map(lambda f: Path(f).stem)
faltan = set(qs.image_key) - set(escena_por_id)
assert not faltan, f"preguntas sobre imágenes sin SceneJSON: {faltan}"

traza_qa, preguntas_det, filas_qa = [], defaultdict(list), []
for p in qs.to_dict("records"):
    escena = escena_por_id[p["image_key"]]
    q = query_determinista(p)
    r = paso(traza_qa, p["image_key"], "qa_determinista", dict(question_id=p["question_id"], query=q),
             lambda: ejecutar(q, escena))
    preguntas_det[p["image_key"]].append(dict(question_id=p["question_id"], type=p["type"], question=p["question"], **r, query=q))
    filas_qa.append(dict(question_id=p["question_id"], image_id=p["image_key"], type=p["type"], clases=", ".join(q["classes"]),
                         predicado=q.get("predicate"), respuesta=r["normalized_answer"], status=r["status"],
                         confianza=r["confidence"], n_evidencia=len(r["evidence"])))
qa_det = pd.DataFrame(filas_qa)

print("Consignas respondidas:", len(qa_det), "| por estado:", qa_det.status.value_counts().to_dict())
print(qa_det[qa_det.type != "description"].drop(columns="image_id").to_string(index=False))
ej = preguntas_det[qs.image_key.iloc[0]][0]
print("\nEjemplo de descripción por plantilla:\n", ej["answer"])

### 5.5 Evaluación: exactitud y fidelidad

Las respuestas de referencia de la cátedra son privadas. Para medir localmente armamos la **escena de ground truth** de cada imagen a partir de las anotaciones: máscaras reales con las mismas reglas de propiedades, relaciones y atributos. Después corremos **el mismo motor** sobre esa escena. Las anotaciones se usan sólo acá, para evaluar.

Se miden tres cosas distintas:
- **Exactitud:** ¿la respuesta coincide con la que da la escena real? Mide la pipeline completa (percepción + motor). Supuesto: la cátedra no publica cómo define tamaño y relaciones, así que la referencia usa nuestras reglas (3.4).
- **Fidelidad:** ¿la respuesta se puede leer tal cual del SceneJSON predicho? Un verificador independiente del motor recalcula cada respuesta desde el JSON y chequea que la evidencia exista. Mide que el motor no invente.
- **Origen del error:** cuando una respuesta falla, ¿el problema está en la percepción (objeto no detectado, conteo distinto, otra instancia) o en el motor?

In [ ]:
stem2id = {Path(r.fname).stem: r.id for r in img_meta.values()}
anns_por_img = defaultdict(list)
for a in ann["annotations"]:
    anns_por_img[a["image_id"]].append(a)

def escena_gt(image_key):
    """SceneJSON armado con las ANOTACIONES. Sólo para evaluar: mismas reglas de propiedades, relaciones y atributos."""
    r = img_meta[stem2id[image_key]]
    objetos = []
    for a in anns_por_img[r.id]:
        m = mask_utils.decode(mask_utils.merge(mask_utils.frPyObjects(a["segmentation"], r.height, r.width))).astype(bool)
        if m.sum() == 0:
            continue
        x, y, w, h = a["bbox"]
        objetos.append(dict(id=f"gt_{a['id']}", category_id=a["category_id"], confidence=1.0, bbox=[x, y, x + w, y + h],
                            mask=rle_encode(m), **propiedades(m, r.width, r.height)))
    esc = dict(image=dict(id=image_key, width=r.width, height=r.height), objects=objetos,
               relations=relaciones_con_tope(objetos, r.width, r.height)[0])
    agregar_atributos(esc, Image.open(DEV / "images" / r.fname).convert("RGB"))
    return esc

escenas_gt = {k: escena_gt(k) for k in qs.image_key.unique()}

def iou_obj(o1, o2):
    return float(mask_utils.iou([a_rle_coco(o1["mask"])], [a_rle_coco(o2["mask"])], [0])[0][0])

def origen_error(p, q, esc, esc_gt, r_pred, r_gt):
    """Clasifica por qué una respuesta no coincide con la de la escena real."""
    n_pred = [len(instancias(esc, c)) for c in q["classes"]]
    n_gt = [len(instancias(esc_gt, c)) for c in q["classes"]]
    if p["type"] in ("count", "existence"):
        if n_pred[0] == 0:
            return "percepción: clase no detectada"
        return "percepción: sobran instancias (FP o duplicados)" if n_pred[0] > n_gt[0] else "percepción: faltan instancias"
    if r_pred["status"] == "insufficient_information":
        return "percepción: clase no detectada (unknown)"
    ids_pred = {o["id"]: o for o in esc["objects"]}
    ids_gt = {o["id"]: o for o in esc_gt["objects"]}
    for e_pred, e_gt in zip(r_pred["evidence"], r_gt["evidence"]):
        if iou_obj(ids_pred[e_pred], ids_gt[e_gt]) < 0.5:
            return "percepción: se eligió otra instancia (FP o mala localización)"
    return "borde de regla: misma instancia, valor distinto (zona, tamaño o margen)"

filas_ev = []
for p in qs[qs.type != "description"].to_dict("records"):
    q = query_determinista(p)
    item = next(x for x in preguntas_det[p["image_key"]] if x["question_id"] == p["question_id"])
    r_gt = ejecutar(q, escenas_gt[p["image_key"]])
    ok = item["normalized_answer"] == r_gt["normalized_answer"]
    filas_ev.append(dict(question_id=p["question_id"], type=p["type"], clases=", ".join(q["classes"]),
                         pred=item["normalized_answer"], ref=r_gt["normalized_answer"], correcta=ok, status=item["status"],
                         n_pred=len(instancias(escena_por_id[p["image_key"]], q["classes"][0])),
                         n_gt=len(instancias(escenas_gt[p["image_key"]], q["classes"][0])),
                         origen="" if ok else origen_error(p, q, escena_por_id[p["image_key"]], escenas_gt[p["image_key"]], item, r_gt)))
ev_det = pd.DataFrame(filas_ev)

def clases_nombradas(texto):
    """Clases del vocabulario que aparecen en un texto. Primero las más largas, para no leer 'glove' dentro de 'baseball glove'."""
    encontradas = set()
    for c in sorted(name2id, key=len, reverse=True):
        patron = rf"\b{re.escape(c)}\b"
        if re.search(patron, texto):
            encontradas.add(c)
            texto = re.sub(patron, " ", texto)
    return encontradas

def verificar_fidelidad(item, escena):
    """Verificador independiente del motor: ¿la respuesta se lee tal cual del SceneJSON predicho?"""
    q, objs = item["query"], {o["id"]: o for o in escena["objects"]}
    if not set(item["evidence"]) <= set(objs):
        return False                                       # evidencia que no existe en la escena
    if item["status"] in ("unsupported", "insufficient_information"):
        return item["evidence"] == [] and item["normalized_answer"] in ("unsupported", "unknown", "empty_scene")
    it = q["intent"]
    if it in ("count", "existence"):
        n = sum(o["category_id"] == name2id[q["classes"][0]] for o in escena["objects"])
        return item["normalized_answer"] == (n if it == "count" else ("yes" if n else "no"))
    if it in ("position", "size", "attribute"):
        o = objs[item["evidence"][0]]
        valor = o["attributes"][q["attribute"]] if it == "attribute" else o[it]
        return o["category_id"] == name2id[q["classes"][0]] and valor == item["normalized_answer"]
    if it == "relation":
        a, b = item["evidence"]
        trip = {(r["subject"], r["predicate"], r["object"]) for r in escena["relations"] + escena.get("extra_relations", [])}
        hay = (a, q["predicate"], b) in trip or (q["predicate"] in ("near", "overlaps") and (b, q["predicate"], a) in trip)
        return (item["normalized_answer"] == "yes") == hay
    if it == "compare":
        n, m = [sum(o["category_id"] == name2id[c] for o in escena["objects"]) for c in q["classes"]]
        return item["normalized_answer"] == ("yes" if {"more": n > m, "fewer": n < m, "same": n == m}[q["comparator"]] else "no")
    if it == "description":   # toda clase nombrada en el texto tiene que estar en la escena
        return clases_nombradas(item["answer"]) <= {id2name[o["category_id"]] for o in escena["objects"]}
    return False

fid = pd.DataFrame([dict(type=x["type"], fiel=verificar_fidelidad(x, escena_por_id[k]))
                    for k, lista in preguntas_det.items() for x in lista])

print("Exactitud contra la escena real (preguntas cerradas):")
resumen = ev_det.groupby("type").agg(preguntas=("correcta", "size"), exactitud=("correcta", "mean"))
resumen.loc["TOTAL"] = [len(ev_det), ev_det.correcta.mean()]
print(resumen.round(3).to_string())
print("\nFidelidad al SceneJSON (verificador independiente):")
print(fid.groupby("type").fiel.agg(["size", "mean"]).rename(columns={"size": "respuestas", "mean": "fieles"}).round(3).to_string())
print(f"Total: {fid.fiel.mean():.1%} de {len(fid)} respuestas")
print("\nRespuestas incorrectas y origen del error:")
print(ev_det[~ev_det.correcta][["question_id", "type", "clases", "pred", "ref", "n_pred", "n_gt", "origen"]].to_string(index=False))
print("\nOrigen de los errores:", ev_det[~ev_det.correcta].origen.value_counts().to_dict())

### 5.6 Preguntas extra: ambigüedad, atributos, comparación y fuera de alcance

El benchmark no tiene preguntas ambiguas, de color, de comparación ni fuera de alcance. Para probar esos casos armamos un set extra sobre las mismas 20 imágenes. Se genera **por código**, para que sea reproducible y sin etiquetado manual:

- **Ambigua (posición):** la clase con más instancias reales en la imagen.
- **Atributo (color):** una clase con una sola instancia real.
- **Comparación:** las dos clases más frecuentes ("¿hay más A que B?").
- **Proximidad (`near`):** el par formado por esas dos clases.
- **Fuera de alcance:** preguntas fijas sobre acciones, material o estado, más una clase que no está en el vocabulario.

Las anotaciones se usan sólo para **elegir** qué preguntar y para tener una referencia; el motor sigue leyendo únicamente el SceneJSON. Se evalúa si cada caso se maneja como corresponde: marcar la ambigüedad, responder `unsupported` cuando no hay forma de responder, y acertar las comparaciones y la proximidad. El set también se reutiliza en el modo LLM (Fase 6), así que cada pregunta trae su texto en español.

In [ ]:
# Nombres en español para redactar las preguntas extra (los del benchmark se toman tal cual de questions_dev)
NOMBRES_ES = {
    "sofa": "sofá", "laptop": "computadora portátil", "pillow": "almohada", "bench": "banco", "jacket": "chaqueta",
    "signboard": "cartel", "backpack": "mochila", "traffic light": "semáforo", "umbrella": "paraguas", "horse": "caballo",
    "handle": "manija", "chair": "silla", "clock": "reloj", "napkin": "servilleta", "pizza": "pizza", "flag": "bandera",
    "person": "persona", "trousers": "pantalones", "trash can": "tacho de basura", "sock": "calcetín", "vase": "jarrón",
    "bicycle": "bicicleta", "tennis racket": "raqueta de tenis", "necklace": "collar", "cell phone": "teléfono celular",
    "book": "libro", "sunglasses": "anteojos de sol", "lamp": "lámpara", "bottle": "botella", "glove": "guante",
    "shoe": "zapato", "blanket": "manta", "computer keyboard": "teclado de computadora", "dog": "perro", "jean": "jean",
    "short pants": "pantalones cortos", "flower arrangement": "arreglo floral", "ski": "esquí", "cat": "gato",
    "wheel": "rueda", "faucet": "grifo", "helmet": "casco", "table": "mesa", "drinking glass": "vaso",
    "headlight": "faro delantero", "streetlight": "farola", "television set": "televisor", "belt": "cinturón",
    "bird": "pájaro", "knife": "cuchillo", "watch": "reloj de pulsera", "handbag": "bolso", "street sign": "señal de tránsito",
    "bowl": "cuenco", "ski pole": "bastón de esquí", "hat": "sombrero", "plate": "plato", "car": "auto", "curtain": "cortina",
    "elephant": "elefante", "bed": "cama", "drawer": "cajón", "earring": "aro", "boat": "barco", "necktie": "corbata",
    "skateboard": "patineta", "wall socket": "tomacorriente", "fork": "tenedor", "bus": "colectivo", "giraffe": "jirafa",
    "toilet": "inodoro", "surfboard": "tabla de surf", "motorcycle": "moto", "cabinet": "gabinete", "taillight": "luz trasera",
    "train": "tren", "baseball cap": "gorra de béisbol", "airplane": "avión", "jersey": "camiseta deportiva", "pole": "poste",
    "doorknob": "pomo de puerta", "baseball bat": "bate de béisbol", "shirt": "camisa", "license plate": "patente",
    "banana": "banana", "mirror": "espejo", "kite": "barrilete", "frisbee": "frisbee", "sink": "pileta", "spectacles": "gafas",
    "button": "botón", "goggles": "antiparras", "zebra": "cebra", "suitcase": "valija", "baseball glove": "guante de béisbol",
    "basket": "canasta", "refrigerator": "refrigerador", "ski boot": "bota de esquí", "tablecloth": "mantel",
    "bracelet": "pulsera", "teddy bear": "oso de peluche", "windshield wiper": "limpiaparabrisas",
    "computer mouse": "mouse de computadora", "box": "caja", "cup": "taza", "dress": "vestido", "computer monitor": "monitor",
    "broccoli": "brócoli", "strap": "correa", "knob": "perilla", "plastic bag": "bolsa de plástico", "ring": "anillo",
    "awning": "toldo", "vent": "rejilla de ventilación", "banner": "pancarta", "cow": "vaca", "flowerpot": "maceta",
    "tag": "etiqueta", "fireplug": "hidrante", "boot": "bota",
}
NOMBRES_ES_PROPIOS = dict(NOMBRES_ES)   # copia previa a los nombres de la cátedra: el modo LLM (Fase 6) sólo puede usar ésta
for p in qs[qs.class_en.str.len() > 0].to_dict("records"):   # mismos nombres que usa la cátedra
    NOMBRES_ES.update(zip(p["class_en"], p["class_es"]))
assert set(NOMBRES_ES) == set(name2id), set(name2id) ^ set(NOMBRES_ES)

FUERA_DE_ALCANCE = [("action", "¿Qué está haciendo la persona en la imagen?"),
                    ("material", "¿De qué material está hecha la mesa?"),
                    ("state", "¿El televisor está encendido?"),
                    ("text", "¿Qué dice el cartel?"),
                    ("emotion", "¿La persona parece contenta?")]

preguntas_extra = []
for i, k in enumerate(sorted(qs.image_key.unique())):
    frec = Counter(id2name[o["category_id"]] for o in escenas_gt[k]["objects"]).most_common()
    a, b = frec[0][0], frec[1][0]
    unicas = [c for c, n in frec if n == 1]
    base = dict(image_key=k)
    if frec[0][1] > 1:
        preguntas_extra.append(dict(base, caso="ambigua", type="position", class_en=[a],
                                    question=f"¿En qué zona de la imagen está la instancia de \"{NOMBRES_ES[a]}\"?"))
    if unicas:
        preguntas_extra.append(dict(base, caso="atributo", type="attribute", class_en=[unicas[0]], attribute="color",
                                    question=f"¿De qué color es la instancia de \"{NOMBRES_ES[unicas[0]]}\"?"))
    preguntas_extra.append(dict(base, caso="comparación", type="compare", class_en=[a, b], comparator="more",
                                question=f"¿Hay más instancias de \"{NOMBRES_ES[a]}\" que de \"{NOMBRES_ES[b]}\" en la imagen?"))
    preguntas_extra.append(dict(base, caso="proximidad", type="relation", class_en=[a, b], predicate="near",
                                question=f"¿La instancia de \"{NOMBRES_ES[a]}\" está cerca de la instancia de \"{NOMBRES_ES[b]}\"?"))
    tipo, texto = FUERA_DE_ALCANCE[i % len(FUERA_DE_ALCANCE)]
    preguntas_extra.append(dict(base, caso="fuera de alcance", type=tipo, class_en=[], question=texto))
    if i % 4 == 0:
        preguntas_extra.append(dict(base, caso="fuera del vocabulario", type="existence", class_en=["unicorn"],
                                    question="¿Hay al menos una instancia de la categoría \"unicornio\" en la imagen?"))
for j, p in enumerate(preguntas_extra, 1):
    p["question_id"] = f"EXTRA-{j:03d}"

def manejo_correcto(p, r, r_ref):
    """¿Se manejó el caso como corresponde?"""
    if p["caso"] in ("fuera de alcance", "fuera del vocabulario"):
        return r["normalized_answer"] == "unsupported"
    if p["caso"] == "ambigua":   # si hay varias instancias predichas, tiene que avisarlo
        n = len(instancias(escena_por_id[p["image_key"]], p["class_en"][0]))
        return r["status"] == ("ambiguous" if n > 1 else "ok") or (n == 0 and r["status"] == "insufficient_information")
    if p["caso"] == "atributo":   # sin ground truth de color: tiene que dar un color o declarar que no hay información
        return r["status"] in ("ok", "ambiguous", "insufficient_information")
    return r["normalized_answer"] == r_ref["normalized_answer"]   # comparación y proximidad: contra la escena real

filas_extra, respuestas_extra = [], {}
for p in preguntas_extra:
    q = query_determinista(p)
    r = ejecutar(q, escena_por_id[p["image_key"]])
    r_ref = ejecutar(q, escenas_gt[p["image_key"]])
    respuestas_extra[p["question_id"]] = r
    filas_extra.append(dict(question_id=p["question_id"], caso=p["caso"], pregunta=p["question"][:70],
                            respuesta=r["normalized_answer"], status=r["status"], ref=r_ref["normalized_answer"],
                            correcto=manejo_correcto(p, r, r_ref)))
ev_extra = pd.DataFrame(filas_extra)
print(f"{len(ev_extra)} preguntas extra sobre {len({p['image_key'] for p in preguntas_extra})} imágenes")
print(ev_extra.groupby("caso").agg(preguntas=("correcto", "size"), manejo_correcto=("correcto", "mean")).round(3).to_string())
print("\nEjemplos:")
for caso in ev_extra.caso.unique():
    p = next(x for x in preguntas_extra if x["caso"] == caso)
    print(f"[{caso}] {p['question']}\n   -> {respuestas_extra[p['question_id']]['answer']}")

### 5.7 Envío con `questions` (v3)

Las respuestas se agregan a las 20 filas de desarrollo. `questions_llm` sigue vacía hasta la Fase 6. Antes de escribir el CSV se valida todo lo que, según la página de evaluación, anula una fila:

- el schema;
- RLE consistente (los `counts` suman alto × ancho);
- referencias: todo `id` usado en `relations`, `extra_relations` o `evidence` existe en la misma escena.

Las tablas de resultados se guardan en `results/` para que el reporte cite los números desde archivos.

In [ ]:
def problemas_de_fila(e):
    """Lo que según la métrica oficial invalida una fila: JSON fuera del schema, RLE inconsistente o referencias rotas."""
    errores = [x.message[:120] for x in validador.iter_errors(e)]
    ids = {o["id"] for o in e["objects"]}
    if len(ids) != len(e["objects"]):
        errores.append("ids de objetos repetidos")
    for o in e["objects"]:
        h, w = o["mask"]["size"]
        if sum(o["mask"]["counts"]) != h * w or [h, w] != [e["image"]["height"], e["image"]["width"]]:
            errores.append(f"RLE inconsistente en {o['id']}")
    for r in e["relations"] + e.get("extra_relations", []):
        if r["subject"] not in ids or r["object"] not in ids:
            errores.append(f"relación con id inexistente: {r}")
    for x in e.get("questions", []) + e.get("questions_llm", []):
        errores += [f"evidencia inexistente {i} en {x['question_id']}" for i in x["evidence"] if i not in ids]
    return errores

for e in escenas:
    e["questions"] = preguntas_det.get(e["image"]["id"], [])
    e.setdefault("questions_llm", [])

problemas = {e["image"]["id"]: problemas_de_fila(e) for e in escenas}
malas = {k: v for k, v in problemas.items() if v}
print("Filas con problemas:", len(malas), "de", len(escenas))
assert not malas, list(malas.items())[:3]
con_preguntas = sum(len(e["questions"]) > 0 for e in escenas)
print("Filas con questions:", con_preguntas, "| preguntas:", sum(len(e["questions"]) for e in escenas))
assert con_preguntas == qs.image_key.nunique()

envio = pd.DataFrame(dict(image_id=[e["image"]["id"] for e in escenas],
                          SceneJSON=[json.dumps(e, separators=(",", ":"), ensure_ascii=False) for e in escenas]))
assert list(envio.image_id) == list(sub.image_id), "filas u orden distintos al sample"
envio.to_csv(SALIDA / "submission.csv", index=False)
envio.to_csv(SALIDA / "solution.csv", index=False)

RESULTADOS = SALIDA / "results"
RESULTADOS.mkdir(exist_ok=True)
qa_det.to_csv(RESULTADOS / "qa_det_respuestas.csv", index=False)
ev_det.to_csv(RESULTADOS / "qa_det_exactitud.csv", index=False)
fid.to_csv(RESULTADOS / "qa_det_fidelidad.csv", index=False)
ev_extra.to_csv(RESULTADOS / "qa_extra_manejo.csv", index=False)
pd.DataFrame(traza_qa).to_csv(SALIDA / "traza_qa.csv", index=False)
print(f"submission.csv: {(SALIDA / 'submission.csv').stat().st_size / 1e6:.1f} MB | resultados en {RESULTADOS}")
marcar_fase("5 QA determinista")

### 5.8 Conclusiones de la Fase 5

*Números de la configuración final (v5 y v6: OWLv2 base con ventanas y sinónimos, SAM2 large). Entre paréntesis, los de la v3 (OWLv2 base sola, SAM2 small).*

| Medida | Resultado |
|---|---|
| Consignas respondidas | 60 de 60: 51 `ok`, 6 `ambiguous`, 3 `insufficient_information` |
| Exactitud en preguntas cerradas (contra la escena real) | **0,775** (31 de 40) (v3: 0,75) |
| Exactitud por tipo | conteo 0,50 · existencia 0,90 · posición 1,00 (v3: 0,80) · relación 0,80 · tamaño 0,80 |
| Fidelidad al SceneJSON (verificador independiente) | **100 %** (60 de 60) |
| Casos extra bien manejados (105 preguntas) | ambiguas, atributo, fuera de alcance y fuera del vocabulario: 100 % · comparación 0,50 · proximidad 0,55 |
| Filas inválidas en el envío | 0 de 80 |

**Lectura.**
- **El motor no agrega errores.** La fidelidad es del 100 %: cada respuesta se puede leer tal cual del SceneJSON. Los 9 errores vienen de la percepción:
  - en 4 conteos faltan instancias;
  - en 3 preguntas (dos relaciones y un tamaño) la clase no se detectó y se respondió `unknown`;
  - en 1 conteo la clase no se detectó (`doorknob`: 0 contra 2);
  - en 1 existencia sobran instancias (`napkin`: el detector ve servilletas que no están anotadas).
- **El conteo es la operación más débil (0,50).** Para contar bien, el detector tiene que encontrar todas las instancias y ninguna de más. Es coherente con el recall de 0,65 de 2.9. Las comparaciones (0,50) heredan el mismo problema, porque comparan dos conteos, y la proximidad (0,55) falla sobre todo cuando una de las dos clases no se detectó.
- **No se inventa.** Las 3 respuestas `unknown` corresponden a clases que no se detectaron. Cuentan como error en la exactitud, pero el sistema declara que no tiene información en lugar de adivinar.
- **La mejora del detector se ve en QA:** con la configuración final la posición pasa de 0,80 a 1,00 (el banco del borde de zona ahora cae en la zona correcta) y la exactitud total sube de 0,75 a 0,775.
- **Límite del método de evaluación.** La referencia se calcula con nuestras reglas de tamaño y relaciones, porque la cátedra no publica las suyas. El puntaje oficial de QA puede diferir en las preguntas de tamaño y de relación.

La Fase 8 lleva cada uno de estos errores hasta la etapa exacta de la percepción donde se originó (umbral, detector, NMS o SAM2). Lo que sigue (Fase 6): la misma consulta intermedia, pero armada por un LLM a partir del texto solo, y ejecutada por este mismo motor.

## 6. Modo con LLM (agregado en la v4)

Objetivo de la fase: que un **LLM liviano de texto** interprete la pregunta, sin los campos estructurados, y que el **mismo motor determinista** de la Fase 5 la resuelva sobre el SceneJSON predicho.

```
texto de la pregunta ─► Qwen2.5-3B ─► consulta JSON ─► normalización y validación (código) ─► motor ejecutar() ─► respuesta normalizada + evidencia
                                         └─ 1 reintento si el JSON no es válido                    └─► Qwen redacta la respuesta ─► control de fidelidad
resumen del SceneJSON (código) ─► Qwen redacta la descripción ─► control de alucinaciones ─► descripción + evidencia
```

**Qué recibe el LLM y qué no:**
- Recibe el **texto** de la pregunta, la lista de operaciones soportadas y el **vocabulario de 120 clases**, en inglés y con un nombre en español de un diccionario propio (`NOMBRES_ES_PROPIOS`, 5.6), que no se arma con los campos `class_es` de las preguntas. Es la configuración del sistema, no un dato de la pregunta.
- **No** recibe la imagen, las anotaciones ni los campos `type`, `class_en` y `class_es`. `type` se copia en la salida sólo porque el contrato del envío lo exige, y se usa después para **medir** la interpretación.
- Para describir la escena recibe un **resumen textual del SceneJSON predicho** que arma el código: conteos, zonas y tamaños.

**Qué decide el código, no el LLM:**
- Conteos, filtros, relaciones, comparaciones, `evidence` y `confidence` salen del motor de la Fase 5.
- La clase que propone el LLM se valida contra el vocabulario. Si no coincide exacto, el código prueba el singular, un diccionario español → inglés propio y una coincidencia aproximada, y registra cuál usó. Si nada coincide, la consulta queda `unsupported`: el código nunca reemplaza un objeto por otro parecido.
- La redacción del LLM se acepta sólo si conserva el valor del motor, y la descripción sólo si nombra clases que están en la escena. Si no, se usa la plantilla y la traza lo registra.

**Modelo:** `Qwen/Qwen2.5-3B-Instruct` en fp16 con decodificación greedy, así la corrida es reproducible. Es liviano (3 000 millones de parámetros), entra en una T4 junto con la percepción y maneja bien español y JSON.

### 6.1 Carga del LLM y generación reproducible
Si hay dos GPU (T4 x2), el LLM va a la segunda y la percepción queda en la primera. La generación es greedy y por lotes.

In [ ]:
import os
import difflib
from transformers import AutoTokenizer, AutoModelForCausalLM

# TPI_LLM_ID sólo se usa para probar la plomería en CPU con un modelo más chico; en Kaggle queda el de 3B
LLM_ID = os.environ.get("TPI_LLM_ID", "Qwen/Qwen2.5-3B-Instruct")
LLM_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else device
LLM_DTYPE = torch.float16 if LLM_DEVICE != "cpu" else torch.float32
torch.manual_seed(0)
tok = AutoTokenizer.from_pretrained(LLM_ID)
tok.padding_side = "left"                     # generación por lotes: el relleno va a la izquierda
try:
    llm = AutoModelForCausalLM.from_pretrained(LLM_ID, dtype=LLM_DTYPE)
except TypeError:                             # versiones de transformers anteriores a `dtype`
    llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=LLM_DTYPE)
llm = llm.to(LLM_DEVICE).eval()
LLM_CONFIG = dict(modelo=LLM_ID, dtype=str(LLM_DTYPE).replace("torch.", ""), decodificacion="greedy", dispositivo=LLM_DEVICE)

@torch.no_grad()
def generar(conversaciones, max_new_tokens=160, lote=8):
    """Generación greedy por lotes. Cada conversación es una lista de mensajes {role, content}."""
    textos = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in conversaciones]
    salidas = []
    for i in range(0, len(textos), lote):
        enc = tok(textos[i:i + lote], return_tensors="pt", padding=True).to(LLM_DEVICE)
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.0,
                           pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
        salidas += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [s.strip() for s in salidas]

t0 = time.time()
prueba_llm = generar([[{"role": "user", "content": "Respondé sólo con la palabra: listo"}]], max_new_tokens=5)
print(json.dumps(LLM_CONFIG), "|", prueba_llm, f"| {time.time() - t0:.1f} s")

### 6.2 Intérprete: del texto a la consulta intermedia

El prompt describe las operaciones del motor (las mismas de la Fase 5), incluye el vocabulario y trae ejemplos resueltos de cada intención. Los ejemplos usan **otras clases** que las del benchmark, para que el LLM aprenda el formato y no las respuestas.

Después, el **código** limpia la salida: toma el primer objeto JSON, se queda con las claves conocidas (cada una sólo en su intención: `predicate` sólo en `relation`, etc.), normaliza las clases y valida con `validar_query()` (5.1). La normalización prueba, en orden: nombre exacto, singular, **alias en inglés** (sinónimos que no son otra clase, por ejemplo `glasses` → `spectacles`), el **diccionario español → inglés propio** (con sinónimos y variantes regionales: `mango` y `manija` → `handle`, `heladera` → `refrigerator`…) y una coincidencia aproximada. Cada paso queda registrado en la traza.

**Reintentos (uno como máximo):**
- **Formato:** si el JSON no se puede leer o no cumple el formato, se le muestra el error al LLM.
- **Vocabulario (v6):** si el LLM propone una clase fuera del vocabulario pero la pregunta **nombra** un objeto que sí está (según el diccionario propio), es un posible error de traducción. Se le avisa cuáles objetos del vocabulario nombra la pregunta y el LLM decide. Si la segunda respuesta no es válida, se conserva la primera.

Una clase fuera del vocabulario **sin** ninguna mención en el texto (por ejemplo "unicornio") no se reintenta: `unsupported` es la respuesta correcta, y pedirle otra clase invitaría a que la reemplace por una parecida.

**Por qué se agregaron en la v6:** en la v5 hubo dos errores de interpretación, los dos por el nombre de la clase. En "gafas" el LLM escribió `glasses` (en el vocabulario, `spectacles`). En una pregunta sobre "mango" copió la palabra tal cual: es el nombre que usa la cátedra para `handle`, pero nuestro diccionario sólo tenía "manija". Las dos correcciones se motivaron con estas mismas preguntas, así que la interpretación final de la v6 es optimista. Por eso 6.4 mide tres cosas: si el LLM escribió las clases **exactamente** como en el vocabulario (sin ayuda del código), la interpretación al **primer intento** (con la normalización) y la **final** (con el reintento).

In [ ]:
PROMPT_INTERPRETE = f"""Sos el módulo de interpretación de un sistema que responde preguntas sobre una escena.
NO ves la imagen y NO respondés la pregunta: sólo la traducís a una consulta JSON que después ejecuta un motor determinista.

Devolvé un único objeto JSON, sin texto adicional, con estas claves:
- "intent": count, existence, position, size, relation, attribute, compare, description o unsupported.
- "classes": los objetos mencionados, en inglés y escritos EXACTAMENTE como en el vocabulario. count, existence, position, size y attribute llevan 1 clase; relation y compare llevan 2, en el orden de la pregunta; description y unsupported llevan [].
- "predicate": sólo en relation: left_of, right_of, above, below, overlaps o near.
- "attribute": sólo en attribute: "color".
- "comparator": sólo en compare: more, fewer o same.
- "reason": sólo en unsupported: por qué no se puede responder.

Qué significa cada intención:
- count: cuántas instancias hay de un objeto. existence: si hay al menos una.
- position: en qué zona de la imagen (grilla 3x3) está un objeto. size: qué tamaño relativo tiene (small, medium o large).
- relation: si un objeto está arriba (above), abajo (below), a la izquierda (left_of), a la derecha (right_of), superpuesto (overlaps) o cerca (near) de otro.
- attribute: el color de un objeto. compare: si hay más (more), menos (fewer) o la misma cantidad (same) de un objeto que de otro.
- description: describir la escena.
- unsupported: cualquier otra cosa (acciones, materiales, estados, texto escrito, emociones, intenciones, etc.).

Reglas:
- Traducí cada objeto al nombre del vocabulario que significa lo mismo y escribí en "classes" sólo el nombre en inglés. Si ninguno significa lo mismo, escribí el nombre en inglés tal cual: nunca lo reemplaces por otro objeto parecido.
- No inventes clases ni operaciones.

Vocabulario ({len(VOCAB)} clases, en inglés y con un nombre en español entre paréntesis):
{", ".join(f"{c} ({NOMBRES_ES_PROPIOS.get(c, c)})" for c in VOCAB)}"""

EJEMPLOS_INTERPRETE = [
    ('¿Cuántas instancias de la categoría "silla" aparecen en la imagen?', {"intent": "count", "classes": ["chair"]}),
    ('¿Hay al menos una instancia de la categoría "paraguas" en la imagen?', {"intent": "existence", "classes": ["umbrella"]}),
    ('¿En qué zona de la imagen está la instancia de "computadora portátil"?', {"intent": "position", "classes": ["laptop"]}),
    ('¿Qué tamaño relativo tiene la instancia de "perro"?', {"intent": "size", "classes": ["dog"]}),
    ('¿La instancia de "lámpara" está a la izquierda de la instancia de "sofá"?',
     {"intent": "relation", "classes": ["lamp", "sofa"], "predicate": "left_of"}),
    ("¿La taza está cerca del plato?", {"intent": "relation", "classes": ["cup", "plate"], "predicate": "near"}),
    ("¿De qué color es el auto?", {"intent": "attribute", "classes": ["car"], "attribute": "color"}),
    ("¿Hay más botellas que cuencos en la imagen?", {"intent": "compare", "classes": ["bottle", "bowl"], "comparator": "more"}),
    ("Contame qué hay en la foto.", {"intent": "description", "classes": []}),
    ("¿Qué está comiendo el gato?", {"intent": "unsupported", "classes": [], "reason": "pregunta por una acción"}),
    ('¿Hay al menos una instancia de la categoría "dragón" en la imagen?', {"intent": "existence", "classes": ["dragon"]}),
]
MENSAJES_INTERPRETE = [{"role": "system", "content": PROMPT_INTERPRETE}]
for pregunta, consulta in EJEMPLOS_INTERPRETE:
    MENSAJES_INTERPRETE += [{"role": "user", "content": pregunta},
                            {"role": "assistant", "content": json.dumps(consulta, ensure_ascii=False)}]

INV_ES = {sin_acentos(v): k for k, v in NOMBRES_ES_PROPIOS.items()}   # diccionario propio, no el de la cátedra

# v6: sinónimos y variantes regionales en español (diccionario propio). Motivo: en la v5 la pregunta decía "mango" y el
# diccionario sólo tenía "manija" para `handle`. Sólo palabras que no son el nombre de otra clase del vocabulario.
SINONIMOS_ES = {
    "handle": ["mango", "asa", "agarradera"], "spectacles": ["anteojos", "lentes"], "sunglasses": ["gafas de sol", "lentes de sol"],
    "doorknob": ["picaporte", "pomo"], "wall socket": ["enchufe"], "cell phone": ["celular", "telefono", "movil"],
    "car": ["coche", "automovil", "carro"], "bus": ["autobus", "omnibus"], "refrigerator": ["heladera", "nevera"],
    "television set": ["television", "tele"], "faucet": ["canilla"], "sink": ["lavabo", "lavamanos", "bacha"],
    "jacket": ["campera"], "kite": ["cometa"], "earring": ["arete"], "signboard": ["letrero"], "trash can": ["cesto de basura", "basurero", "papelera"],
    "baseball glove": ["manopla"], "flowerpot": ["matera"], "bowl": ["bol"], "skateboard": ["monopatin"], "motorcycle": ["motocicleta"],
    "bicycle": ["bici"], "streetlight": ["farol"], "license plate": ["matricula"], "headlight": ["faro"], "backpack": ["morral"],
    "suitcase": ["maleta"], "handbag": ["cartera"], "teddy bear": ["peluche"], "computer mouse": ["raton"], "laptop": ["notebook", "portatil"],
    "cabinet": ["armario", "alacena"], "jean": ["jeans", "vaqueros"], "short pants": ["shorts", "bermuda"], "baseball cap": ["gorra"],
    "blanket": ["frazada", "cobija"], "pillow": ["almohadon", "cojin"], "toilet": ["retrete"], "umbrella": ["sombrilla"],
    "boat": ["bote", "lancha"], "trousers": ["pantalon"], "fireplug": ["boca de incendio"], "vent": ["rejilla"],
}
assert set(SINONIMOS_ES) <= set(name2id)
assert not {s for l in SINONIMOS_ES.values() for s in l} & set(INV_ES), "un sinónimo coincide con el nombre propio de una clase"
INV_ES.update({s: c for c, l in SINONIMOS_ES.items() for s in l})

# v6: sinónimos en inglés que no son otra clase del vocabulario (los del detector de 2.9 más algunos comunes).
# Motivo: en la v5 el LLM tradujo "gafas" a "glasses", que no es el nombre LVIS ("spectacles").
ALIAS_EN = {s: c for c, sinonimos in SINONIMOS.items() for s in sinonimos}
ALIAS_EN.update({"glasses": "spectacles", "eyeglasses": "spectacles", "tv": "television set", "television": "television set",
                 "pants": "trousers", "jeans": "jean", "mobile phone": "cell phone", "smartphone": "cell phone", "phone": "cell phone",
                 "couch": "sofa", "door handle": "handle", "fire hydrant": "fireplug", "hydrant": "fireplug", "tie": "necktie",
                 "purse": "handbag", "mug": "cup", "outlet": "wall socket", "socket": "wall socket", "monitor": "computer monitor",
                 "keyboard": "computer keyboard", "mouse": "computer mouse", "bike": "bicycle", "motorbike": "motorcycle"})
assert set(ALIAS_EN.values()) <= set(name2id) and not set(ALIAS_EN) & set(name2id)

def normalizar_clase(c):
    """Lleva la clase que propone el LLM al vocabulario. Devuelve (clase, cómo se resolvió)."""
    c0 = " ".join(str(c).strip().lower().split())
    if c0 in name2id:
        return c0, "exacta"
    for sing in (c0[:-1] if c0.endswith("s") else None, c0[:-2] if c0.endswith("es") else None):
        if sing in name2id:
            return sing, "singular"
    for alias in (c0, c0[:-1] if c0.endswith("s") else None):
        if alias in ALIAS_EN:
            return ALIAS_EN[alias], "alias_en"
    if sin_acentos(c0) in INV_ES:
        return INV_ES[sin_acentos(c0)], "diccionario_es"
    cerca = difflib.get_close_matches(c0, VOCAB, n=1, cutoff=0.88)
    if cerca:
        return cerca[0], "aproximada"
    return c0, "sin_coincidencia"

def extraer_json(texto):
    """Primer objeto JSON de la salida del LLM (o None)."""
    if "{" not in texto:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(texto[texto.index("{"):])
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def preparar_query(salida):
    """Salida cruda del LLM -> (consulta limpia, problemas, normalizaciones). El código decide, no el LLM."""
    crudo = extraer_json(salida)
    if crudo is None:
        return None, ["la salida no contiene un objeto JSON válido"], []
    q = {k: crudo[k] for k in ("intent", "classes", "predicate", "attribute", "comparator", "reason") if crudo.get(k) is not None}
    propias = dict(predicate="relation", attribute="attribute", comparator="compare", reason="unsupported")
    q = {k: v for k, v in q.items() if k not in propias or q.get("intent") == propias[k]}   # cada clave sólo en su intención
    if isinstance(q.get("classes"), str):
        q["classes"] = [q["classes"]]
    q.setdefault("classes", [])
    norm = []
    if isinstance(q["classes"], list) and all(isinstance(c, str) for c in q["classes"]):
        resueltas = [normalizar_clase(c) for c in q["classes"]]
        norm = [dict(llm=c, clase=r, como=como) for c, (r, como) in zip(q["classes"], resueltas)]
        q["classes"] = [r for r, _ in resueltas]
    _, problemas = validar_query(q)
    if q.get("intent") == "unsupported":
        problemas = []
    return q, problemas, norm

def hay_que_reintentar(q, problemas):
    """Se reintenta por errores de formato, no por una clase fuera del vocabulario (eso es una respuesta válida)."""
    return q is None or any(not p.startswith("clase fuera del vocabulario") for p in problemas)

def menciones_vocab(texto):
    """Clases del vocabulario que el texto nombra, en español (diccionario propio) o en inglés. Primero los nombres más largos."""
    t = sin_acentos(texto.lower())
    nombres = {**INV_ES, **{c: c for c in name2id}}
    encontradas = []
    for n in sorted(nombres, key=len, reverse=True):
        patron = rf"\b{re.escape(n)}(s|es)?\b"
        if re.search(patron, t):
            encontradas.append(nombres[n])
            t = re.sub(patron, " ", t)
    return list(dict.fromkeys(encontradas))

def pista_vocabulario(texto, r):
    """v6: si el LLM propuso una clase fuera del vocabulario pero la pregunta nombra objetos que sí están, arma la pista del
    reintento. Es un posible error de traducción de la clase. Elige el LLM; el código sólo avisa.
    No se usa si la intención es unsupported o description: ahí las clases no se ejecutan, y un reintento podría
    convertir una pregunta fuera de alcance en una soportada."""
    if (r["query"] or {}).get("intent") in (None, "unsupported", "description"):
        return None
    fuera = [n["llm"] for n in r["normalizacion"] if n["como"] == "sin_coincidencia"]
    candidatas = [c for c in menciones_vocab(texto) if c not in (r["query"] or {}).get("classes", [])]
    if not fuera or not candidatas:
        return None
    return (f"Las clases {', '.join(fuera)} no están en el vocabulario. La pregunta nombra estos objetos que sí están: "
            f"{', '.join(f'{c} ({NOMBRES_ES_PROPIOS.get(c, c)})' for c in candidatas)}. Si alguno es el objeto por el que "
            "se pregunta, usá su nombre en inglés; si no, dejá la clase como estaba. Devolvé sólo el JSON.")

def interpretar(preguntas):
    """Lista de textos -> lista de dicts con la consulta, los problemas y el registro de la interpretación."""
    t0 = time.time()
    convs = [MENSAJES_INTERPRETE + [{"role": "user", "content": t}] for t in preguntas]
    salidas = generar(convs, max_new_tokens=100)
    res = []
    for s in salidas:
        q, prob, norm = preparar_query(s)
        res.append(dict(query=q, query_1=q, problemas=prob, normalizacion=norm, normalizacion_1=norm, salida_1=s, intentos=1,
                        motivo_reintento="", valida_1=not hay_que_reintentar(q, prob)))
    # Dos motivos de reintento: formato inválido, o clase fuera del vocabulario cuando la pregunta nombra una que sí está
    reintentar = {}
    for i, (t, r) in enumerate(zip(preguntas, res)):
        if not r["valida_1"]:
            reintentar[i] = ("formato", f"Esa consulta no es válida: {'; '.join(r['problemas'])}. Corregila y devolvé sólo el JSON.")
        elif pista := pista_vocabulario(t, r):
            reintentar[i] = ("vocabulario", pista)
    if reintentar:
        idx = list(reintentar)
        convs2 = [convs[i] + [{"role": "assistant", "content": res[i]["salida_1"]}, {"role": "user", "content": reintentar[i][1]}]
                  for i in idx]
        for i, s in zip(idx, generar(convs2, max_new_tokens=100)):
            q, prob, norm = preparar_query(s)
            res[i].update(salida_2=s, intentos=2, motivo_reintento=reintentar[i][0])
            if reintentar[i][0] == "formato" or not hay_que_reintentar(q, prob):   # una pista nunca empeora una consulta válida
                res[i].update(query=q, problemas=prob, normalizacion=norm)
    ms = (time.time() - t0) * 1000 / max(1, len(preguntas))
    for r in res:
        r["valida_final"] = not hay_que_reintentar(r["query"], r["problemas"])
        r["ms_por_pregunta"] = round(ms, 1)
    return res

# Prueba con preguntas que no están en el benchmark
print("Menciones del vocabulario en un texto de prueba:", menciones_vocab('¿El "reloj de pulsera" está cerca de las sillas?'))
for t, r in zip(pruebas := ["¿Cuántos inodoros hay?", "¿El reloj está a la derecha de la ventana?", "¿Qué hora marca el reloj?"],
                interpretar(pruebas)):
    print(f"{t:45s} -> {json.dumps(r['query'], ensure_ascii=False)} | problemas: {r['problemas'] or '-'} | intentos: {r['intentos']}")

### 6.3 Redacción con control: respuestas y descripción

**Respuestas.** El LLM recibe la pregunta, el resultado del motor y el valor normalizado, y lo redacta en una oración. Un control con código acepta el texto sólo si conserva el valor: el número en un conteo, "sí" o "no" al principio en existencia, relación y comparación, y la zona, el tamaño o el color en las demás. Si el control falla se usa la frase del motor.

**Descripción.** El código arma un resumen del SceneJSON predicho (cuántos objetos hay de cada clase, en qué zona y de qué tamaño) y el LLM lo redacta en 2 o 3 oraciones, nombrando cada clase en español y con su nombre del vocabulario entre paréntesis. El control de alucinaciones acepta el texto sólo si:
- nombra al menos una clase del vocabulario y **todas** están en la escena;
- cada número que aparece está en el resumen, y cada "N objetos (clase)" coincide con el conteo de esa clase en la escena.

Si el control falla hay un reintento. Si vuelve a fallar se usa la descripción por plantilla de la Fase 5. `evidence` son los objetos de las clases que la descripción nombra.

**Ejemplos resueltos (v5).** En la v4 el control rechazó el 70 % de las redacciones: el modelo contestaba "No" a secas o armaba listas con el nombre en español entre paréntesis. Desde la v5, los dos prompts traen ejemplos de la salida esperada, y el control exige una oración de al menos 12 caracteres.

In [ ]:
ZONAS_ES = dict(top_left="arriba a la izquierda", top_center="arriba al centro", top_right="arriba a la derecha",
                middle_left="en el medio a la izquierda", center="en el centro", middle_right="en el medio a la derecha",
                bottom_left="abajo a la izquierda", bottom_center="abajo al centro", bottom_right="abajo a la derecha")
TAMANOS_ES = dict(small="chico", medium="mediano", large="grande")
COLORES_ES = dict(black="negro", white="blanco", gray="gris", red="rojo", orange="naranja", brown="marrón", yellow="amarillo",
                  green="verde", cyan="celeste", blue="azul", purple="violeta", pink="rosa")

PROMPT_VERBALIZAR = ("Redactás en español, en UNA oración completa, la respuesta de un sistema que analiza una imagen. "
                     "El resultado ya lo calculó un motor determinista: copiá su valor sin cambiarlo, no agregues datos y no expliques "
                     "el método. Escribí los números con dígitos. Si el valor es sí o no, empezá la oración con 'Sí,' o 'No,' según ese valor.")
SI_NO_ES = dict(yes="sí", no="no")

def valor_es(v):
    v = str(v)
    return ZONAS_ES.get(v) or TAMANOS_ES.get(v) or COLORES_ES.get(v) or SI_NO_ES.get(v) or v

def turno_verbalizar(pregunta, texto_motor, valor):
    return f"Pregunta: {pregunta}\nResultado del motor: {texto_motor}\nValor: {valor} ({valor_es(valor)})"

EJEMPLOS_VERBALIZAR = [
    (turno_verbalizar('¿Cuántas instancias de la categoría "silla" aparecen en la imagen?', "Hay 3 instancia(s) de 'chair' en la escena.", 3),
     "En la imagen aparecen 3 sillas."),
    (turno_verbalizar('¿Hay al menos una instancia de la categoría "perro" en la imagen?', "Sí, hay 1 instancia(s) de 'dog'.", "yes"),
     "Sí, en la imagen hay 1 perro."),
    (turno_verbalizar('¿La instancia de "lámpara" está a la izquierda de la instancia de "sofá"?',
                      "No: en la escena, 'lamp' (obj_4) no cumple left_of respecto de 'sofa' (obj_1).", "no"),
     "No, la lámpara no está a la izquierda del sofá."),
    (turno_verbalizar('¿En qué zona de la imagen está la instancia de "computadora portátil"?',
                      "La instancia de 'laptop' está en la zona bottom_left.", "bottom_left"),
     "La computadora portátil está abajo a la izquierda de la imagen."),
    (turno_verbalizar('¿Qué tamaño relativo tiene la instancia de "paraguas"?',
                      "No se detectó ninguna instancia de 'umbrella': no hay información suficiente para responder.", "unknown"),
     "No se detectó ningún paraguas, así que no hay información suficiente para responder."),
]
MENSAJES_VERBALIZAR = [{"role": "system", "content": PROMPT_VERBALIZAR}]
for u, a in EJEMPLOS_VERBALIZAR:
    MENSAJES_VERBALIZAR += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]

def verbalizacion_fiel(texto, r, intent):
    """Control con código: el texto del LLM tiene que ser una oración y conservar el valor que calculó el motor."""
    t, v = sin_acentos(texto), r["normalized_answer"]
    if not texto or len(texto) < 12 or len(texto) > 400:
        return False
    if r["status"] in ("unsupported", "insufficient_information"):
        return bool(re.search(r"\bno\b", t))
    if intent == "count":
        return str(v) in re.findall(r"\d+", t) or (v == 0 and bool(re.search(r"\bningun|\bno hay\b", t)) and not re.search(r"\d", t))
    if v in ("yes", "no"):
        return t.startswith("si" if v == "yes" else "no")
    return sin_acentos(valor_es(v)) in t or str(v).lower() in t

def mensajes_verbalizar(pregunta, r):
    return MENSAJES_VERBALIZAR + [{"role": "user", "content": turno_verbalizar(pregunta, r["answer"], r["normalized_answer"])}]

def resumen_escena(escena, max_clases=8):
    """Resumen textual del SceneJSON predicho para el LLM: sólo conteos, zonas y tamaños (los calcula el código)."""
    por_clase = defaultdict(list)
    for o in escena["objects"]:
        por_clase[id2name[o["category_id"]]].append(o)
    orden = sorted(por_clase.items(), key=lambda kv: (-len(kv[1]), -kv[1][0]["confidence"]))[:max_clases]
    lineas = [f"Objetos detectados: {len(escena['objects'])}, de {len(por_clase)} clases."]
    for nombre, lista in orden:
        zona = Counter(o["position"] for o in lista).most_common(1)[0][0]
        mayor = max(lista, key=lambda o: o["area_ratio"])
        lineas.append(f"- {NOMBRES_ES_PROPIOS.get(nombre, nombre)} ({nombre}): {len(lista)}; zona más frecuente: {ZONAS_ES[zona]}; "
                      f"tamaño del más grande: {TAMANOS_ES[mayor['size']]}")
    if escena["objects"]:
        g = max(escena["objects"], key=lambda o: o["area_ratio"])
        nombre_g = id2name[g["category_id"]]
        lineas.append(f"El objeto que más área ocupa: {NOMBRES_ES_PROPIOS.get(nombre_g, nombre_g)} ({nombre_g}), "
                      f"{ZONAS_ES[g['position']]}.")
    numeros = {str(len(escena["objects"])), str(len(por_clase))} | {str(len(l)) for _, l in orden}
    return "\n".join(lineas), numeros

PROMPT_DESCRIBIR = ("Escribís descripciones breves de escenas a partir de datos de un detector de objetos. Usá SOLO los datos que te pasan: "
                    "no agregues objetos, colores, acciones, lugares ni relaciones que no estén. Escribí un párrafo de 2 o 3 oraciones "
                    "en español (no una lista), con los números en dígitos, y nombrá cada objeto en español seguido de su nombre EN INGLÉS "
                    "entre paréntesis, tal como aparece en los datos, por ejemplo: 2 sillas (chair).")
EJEMPLO_DESCRIBIR = (
    "Datos de la escena:\nObjetos detectados: 9, de 3 clases.\n"
    "- silla (chair): 4; zona más frecuente: abajo al centro; tamaño del más grande: mediano\n"
    "- persona (person): 4; zona más frecuente: en el centro; tamaño del más grande: grande\n"
    "- mesa (table): 1; zona más frecuente: en el centro; tamaño del más grande: grande\n"
    "El objeto que más área ocupa: mesa (table), en el centro.",
    "La escena tiene 9 objetos detectados de 3 clases. Se ven 4 sillas (chair), sobre todo abajo al centro, y 4 personas (person) "
    "en el centro. La mesa (table) es el objeto que más área ocupa y está en el centro de la imagen.")

def control_descripcion(texto, escena, numeros):
    """Control de alucinaciones: clases nombradas ⊆ clases de la escena, al menos una, y números sustentados."""
    presentes = {id2name[o["category_id"]] for o in escena["objects"]}
    nombradas = clases_nombradas(texto.lower())
    problemas = []
    if not nombradas:
        problemas.append("no nombra ninguna clase del vocabulario")
    if nombradas - presentes:
        problemas.append(f"nombra clases que no están en la escena: {sorted(nombradas - presentes)}")
    sobran = set(re.findall(r"\d+", texto)) - numeros
    if sobran:
        problemas.append(f"números sin sustento: {sorted(sobran)}")
    conteo = Counter(id2name[o["category_id"]] for o in escena["objects"])
    for n, clase in re.findall(r"(\d+)\s+[^\d()]{0,40}?\(([a-z ]+)\)", texto.lower()):   # "2 sillas (chair)"
        if clase.strip() in conteo and int(n) != conteo[clase.strip()]:
            problemas.append(f"dice {n} {clase.strip()} y la escena tiene {conteo[clase.strip()]}")
    return nombradas, problemas

def describir_llm(escenas_a_describir):
    """Descripción redactada por el LLM con control de alucinaciones; si falla dos veces, plantilla de la Fase 5."""
    resumenes = [resumen_escena(e) for e in escenas_a_describir]
    convs = [[{"role": "system", "content": PROMPT_DESCRIBIR}, {"role": "user", "content": EJEMPLO_DESCRIBIR[0]},
              {"role": "assistant", "content": EJEMPLO_DESCRIBIR[1]}, {"role": "user", "content": f"Datos de la escena:\n{res}"}]
             for res, _ in resumenes]
    salidas = generar(convs, max_new_tokens=160)
    controles = [control_descripcion(s, e, n) for s, e, (_, n) in zip(salidas, escenas_a_describir, resumenes)]
    reintentar = [i for i, (_, p) in enumerate(controles) if p]
    intentos = [1] * len(salidas)
    if reintentar:
        convs2 = [convs[i] + [{"role": "assistant", "content": salidas[i]},
                              {"role": "user", "content": f"La descripción tiene problemas: {'; '.join(controles[i][1])}. "
                                                          "Reescribila usando sólo los datos."}] for i in reintentar]
        for i, s in zip(reintentar, generar(convs2, max_new_tokens=160)):
            salidas[i], controles[i], intentos[i] = s, control_descripcion(s, escenas_a_describir[i], resumenes[i][1]), 2
    respuestas = []
    for e, s, (nombradas, prob), n_int in zip(escenas_a_describir, salidas, controles, intentos):
        if prob or not e["objects"]:
            r = describir(e)
            respuestas.append(dict(r, redaccion="plantilla", intentos=n_int, problemas_redaccion=prob))
            continue
        evid = [o["id"] for o in e["objects"] if id2name[o["category_id"]] in nombradas]
        conf = np.mean([o["confidence"] for o in e["objects"] if o["id"] in set(evid)])
        respuestas.append(dict(respuesta(s, s, evid, conf, "ok"), redaccion="llm", intentos=n_int, problemas_redaccion=[]))
    return respuestas

# Prueba sobre la primera escena del benchmark
esc_prueba = escena_por_id[qs.image_key.iloc[0]]
print(resumen_escena(esc_prueba)[0])
r_prueba = describir_llm([esc_prueba])[0]
print(f"\n[{r_prueba['redaccion']}] {r_prueba['answer']}\nevidencia: {len(r_prueba['evidence'])} objetos")

### 6.4 Benchmark en modo LLM y evaluación

Las 60 consignas del benchmark y las preguntas extra de 5.6 pasan por el intérprete, el motor y la redacción. Medimos:

- **Interpretación:** ¿la consulta del LLM coincide con la que sale de los campos estructurados (intención, clases y predicado)? Es la exactitud del intérprete, aislada del resto de la pipeline.
- **JSON válido** al primer intento y después del reintento.
- **Exactitud final** contra la escena real (mismo criterio que 5.5) y **acuerdo** con el modo determinista. Si la interpretación es correcta, las dos respuestas tienen que ser idénticas, porque las ejecuta el mismo motor.
- **Descripciones:** cuántas pasaron el control de alucinaciones y qué parte de las clases principales de la escena nombran (cobertura).
- **Casos extra:** el mismo criterio de manejo de 5.6 (ambiguas, fuera de alcance, fuera del vocabulario, comparación y proximidad).

Cada paso queda en `traza_llm.csv`, con la salida cruda del LLM, los intentos, la normalización de clases y la redacción usada.

In [ ]:
def responder_llm(preguntas, escenas_de):
    """Modo LLM completo. El LLM recibe SÓLO p['question']; image_key indica sobre qué SceneJSON ejecuta el motor."""
    interp = interpretar([p["question"] for p in preguntas])
    salida, a_verbalizar, a_describir = [None] * len(preguntas), [], []
    for i, (p, it) in enumerate(zip(preguntas, interp)):
        q = it["query"]
        if q and q.get("intent") == "description" and not it["problemas"]:
            a_describir.append(i)
        else:
            salida[i] = ejecutar(q or {}, escenas_de[p["image_key"]])
            a_verbalizar.append(i)
    for i, r in zip(a_describir, describir_llm([escenas_de[preguntas[i]["image_key"]] for i in a_describir])):
        salida[i] = r
    textos = generar([mensajes_verbalizar(preguntas[i]["question"], salida[i]) for i in a_verbalizar], max_new_tokens=80)
    for i, t in zip(a_verbalizar, textos):
        fiel = verbalizacion_fiel(t, salida[i], (interp[i]["query"] or {}).get("intent"))
        salida[i] = dict(salida[i], answer=t if fiel else salida[i]["answer"], redaccion="llm" if fiel else "plantilla", texto_llm=t)
    return interp, salida

def misma_consulta(q_llm, q_ref):
    """¿El LLM interpretó lo mismo que dicen los campos estructurados? (intención, clases en orden y predicado/comparador)"""
    q_llm = q_llm or {}
    if q_ref["intent"] != q_llm.get("intent"):
        return False
    if q_ref["intent"] in ("description", "unsupported"):
        return True
    return (list(q_llm.get("classes", [])) == q_ref["classes"] and q_llm.get("predicate") == q_ref.get("predicate")
            and q_llm.get("comparator") == q_ref.get("comparator"))

# --- Benchmark: 60 consignas sobre las 20 imágenes de desarrollo ---
t0 = time.time()
preg_bench = qs.to_dict("records")
interp_b, resp_b = responder_llm(preg_bench, escena_por_id)
print(f"Benchmark en modo LLM: {len(preg_bench)} consignas en {(time.time() - t0) / 60:.1f} min")

preguntas_llm, filas_llm, traza_llm = defaultdict(list), [], []
for p, it, r in zip(preg_bench, interp_b, resp_b):
    iq = it["query"] if it["query"] is not None else {"error": "; ".join(it["problemas"])}
    item = dict(question_id=p["question_id"], type=p["type"], question=p["question"], answer=r["answer"],
                normalized_answer=r["normalized_answer"], evidence=r["evidence"], confidence=r["confidence"], status=r["status"],
                interpreted_query=iq, llm=dict(modelo=LLM_ID, intentos=it["intentos"], normalizacion=it["normalizacion"],
                                               redaccion=r.get("redaccion")))
    if r.get("alternatives"):
        item["alternatives"] = r["alternatives"]
    preguntas_llm[p["image_key"]].append(item)

    q_det = query_determinista(p)
    item_det = next(x for x in preguntas_det[p["image_key"]] if x["question_id"] == p["question_id"])
    fila = dict(question_id=p["question_id"], type=p["type"], interpretacion_ok=misma_consulta(it["query"], q_det),
                interpretacion_ok_1=misma_consulta(it["query_1"], q_det), motivo_reintento=it["motivo_reintento"],
                clases_exactas_1=all(n["como"] == "exacta" for n in it["normalizacion_1"]),
                json_valido_1=it["valida_1"], json_valido_final=it["valida_final"], intentos=it["intentos"],
                consulta_llm=json.dumps(it["query"], ensure_ascii=False), respuesta_llm=r["normalized_answer"],
                respuesta_det=item_det["normalized_answer"], redaccion=r.get("redaccion"), status=r["status"])
    if p["type"] != "description":
        ref = ejecutar(q_det, escenas_gt[p["image_key"]])["normalized_answer"]
        fila.update(ref=ref, correcta_llm=r["normalized_answer"] == ref, correcta_det=item_det["normalized_answer"] == ref,
                    igual_det=r["normalized_answer"] == item_det["normalized_answer"])
    filas_llm.append(fila)
    traza_llm.append(dict(image_id=p["image_key"], question_id=p["question_id"], params=json.dumps(LLM_CONFIG),
                          salida_interprete_1=it["salida_1"], salida_interprete_2=it.get("salida_2", ""), intentos=it["intentos"],
                          motivo_reintento=it["motivo_reintento"],
                          normalizacion=json.dumps(it["normalizacion"], ensure_ascii=False), ms_interprete=it["ms_por_pregunta"],
                          consulta=json.dumps(it["query"], ensure_ascii=False), status_motor=r["status"],
                          redaccion=r.get("redaccion"), texto_llm=r.get("texto_llm", r["answer"]), ok=it["valida_final"]))
ev_llm = pd.DataFrame(filas_llm)

print("\nInterpretación (consulta del LLM = consulta de los campos estructurados), por tipo:")
print(ev_llm.groupby("type").agg(consignas=("interpretacion_ok", "size"), interpretacion_1er_intento=("interpretacion_ok_1", "mean"),
                                 interpretacion=("interpretacion_ok", "mean"), json_1er_intento=("json_valido_1", "mean"),
                                 json_final=("json_valido_final", "mean")).round(3).to_string())
print("Reintentos por motivo:", ev_llm.motivo_reintento.replace("", "sin reintento").value_counts().to_dict())
print(f"Clases escritas por el LLM exactamente como en el vocabulario (1.er intento, sin normalizar): {ev_llm.clases_exactas_1.mean():.1%}")
cerr = ev_llm[ev_llm.type != "description"]
print(f"\nPreguntas cerradas ({len(cerr)}): exactitud LLM {cerr.correcta_llm.astype(bool).mean():.3f} | determinista {cerr.correcta_det.astype(bool).mean():.3f} | "
      f"misma respuesta que el determinista: {cerr.igual_det.astype(bool).mean():.1%}")
print("Redacción aceptada por el control (cerradas):", f"{(cerr.redaccion == 'llm').mean():.1%}")
mal = ev_llm[~ev_llm.interpretacion_ok]
if len(mal):
    print("\nConsignas mal interpretadas:")
    print(mal[["question_id", "type", "consulta_llm"]].to_string(index=False))

# Descripciones: control de alucinaciones y cobertura de las clases principales
filas_desc = []
for p, r in zip(preg_bench, resp_b):
    if p["type"] != "description":
        continue
    esc = escena_por_id[p["image_key"]]
    principales = [c for c, _ in Counter(id2name[o["category_id"]] for o in esc["objects"]).most_common(3)]
    nombradas = clases_nombradas(r["answer"].lower())
    filas_desc.append(dict(question_id=p["question_id"], redaccion=r.get("redaccion"), intentos=r.get("intentos"),
                           clases_nombradas=len(nombradas), alucinadas=len(nombradas - {id2name[o["category_id"]] for o in esc["objects"]}),
                           cobertura_top3=np.mean([c in nombradas for c in principales]) if principales else np.nan,
                           n_evidencia=len(r["evidence"]), texto=r["answer"]))
ev_desc = pd.DataFrame(filas_desc)
print(f"\nDescripciones: {(ev_desc.redaccion == 'llm').sum()} de {len(ev_desc)} redactadas por el LLM "
      f"({(ev_desc.intentos == 1).sum()} al primer intento) | clases alucinadas en las aceptadas: "
      f"{ev_desc[ev_desc.redaccion == 'llm'].alucinadas.sum()} | cobertura de las 3 clases principales: {ev_desc.cobertura_top3.mean():.2f}")
print("Ejemplo:", ev_desc.texto.iloc[0])

# --- Casos extra de 5.6 en modo LLM ---
interp_x, resp_x = responder_llm(preguntas_extra, escena_por_id)
filas_x = []
for p, it, r in zip(preguntas_extra, interp_x, resp_x):
    q_ref = query_determinista(p)
    if p["caso"] == "fuera de alcance":
        q_ref = dict(intent="unsupported", classes=[])
    r_ref = ejecutar(query_determinista(p), escenas_gt[p["image_key"]])
    filas_x.append(dict(question_id=p["question_id"], caso=p["caso"], pregunta=p["question"][:70],
                        interpretacion_ok=misma_consulta(it["query"], q_ref), respuesta=r["normalized_answer"], status=r["status"],
                        correcto=manejo_correcto(p, r, r_ref), consulta_llm=json.dumps(it["query"], ensure_ascii=False)))
ev_extra_llm = pd.DataFrame(filas_x)
print(f"\nCasos extra en modo LLM ({len(ev_extra_llm)} preguntas):")
print(ev_extra_llm.groupby("caso").agg(preguntas=("correcto", "size"), interpretacion=("interpretacion_ok", "mean"),
                                       manejo_correcto=("correcto", "mean")).round(3).to_string())

### 6.5 Envío con `questions` y `questions_llm` (v4)

Se agregan las respuestas del modo LLM a las 20 filas del benchmark y se repite la validación de 5.7: schema (ahora incluye `interpreted_query`), RLE e ids referenciados en `relations`, `extra_relations` y en la `evidence` de los dos modos. Al final, una tabla resume los números de desarrollo de esta versión, leída de las variables de la corrida.

In [ ]:
for e in escenas:
    e["questions_llm"] = preguntas_llm.get(e["image"]["id"], [])

problemas = {e["image"]["id"]: problemas_de_fila(e) for e in escenas}
malas = {k: v for k, v in problemas.items() if v}
print("Filas con problemas:", len(malas), "de", len(escenas))
assert not malas, list(malas.items())[:3]
con_llm = sum(len(e["questions_llm"]) > 0 for e in escenas)
print("Filas con questions_llm:", con_llm, "| respuestas:", sum(len(e["questions_llm"]) for e in escenas))
assert con_llm == qs.image_key.nunique()

envio = pd.DataFrame(dict(image_id=[e["image"]["id"] for e in escenas],
                          SceneJSON=[json.dumps(e, separators=(",", ":"), ensure_ascii=False) for e in escenas]))
assert list(envio.image_id) == list(sub.image_id), "filas u orden distintos al sample"
envio.to_csv(SALIDA / "submission.csv", index=False)
envio.to_csv(SALIDA / "solution.csv", index=False)

ev_llm.to_csv(RESULTADOS / "qa_llm_respuestas.csv", index=False)
ev_desc.to_csv(RESULTADOS / "qa_llm_descripciones.csv", index=False)
ev_extra_llm.to_csv(RESULTADOS / "qa_llm_extra.csv", index=False)
pd.DataFrame(traza_llm).to_csv(SALIDA / "traza_llm.csv", index=False)

# Resumen de la versión: todos los números salen de las variables de esta corrida
seg_elegido = tabla_seg.loc[tabla_seg.segmentador == SAM_ID].iloc[0]
resumen_corrida = pd.DataFrame([
    dict(etapa="detección", metrica="F1 de cajas (dev, IoU 0,5, LVIS)", valor=round(float(g.F1), 4), linea_base=0.660),
    dict(etapa="segmentación", metrica="F1 de máscaras (dev)", valor=round(float(seg_elegido.F1), 4), linea_base=0.620),
    dict(etapa="QA determinista", metrica="exactitud en cerradas", valor=round(float(ev_det.correcta.mean()), 4), linea_base=0.75),
    dict(etapa="QA LLM", metrica="exactitud en cerradas", valor=round(float(cerr.correcta_llm.astype(bool).mean()), 4), linea_base=None),
    dict(etapa="QA LLM", metrica="interpretación correcta al primer intento", valor=round(float(ev_llm.interpretacion_ok_1.mean()), 4), linea_base=None),
    dict(etapa="QA LLM", metrica="interpretación correcta", valor=round(float(ev_llm.interpretacion_ok.mean()), 4), linea_base=None),
    dict(etapa="QA LLM", metrica="descripciones redactadas por el LLM", valor=round(float((ev_desc.redaccion == "llm").mean()), 4), linea_base=None),
])
resumen_corrida.to_csv(RESULTADOS / "resumen_corrida.csv", index=False)
print("\nConfiguración:", json.dumps(dict(detector=DET_CONFIG, segmentador=SAM_ID, caja_desde_mascara=bool(CAJA_DESDE_MASCARA),
                                           llm=LLM_CONFIG), ensure_ascii=False))
print(resumen_corrida.to_string(index=False))
print(f"\nsubmission.csv: {(SALIDA / 'submission.csv').stat().st_size / 1e6:.1f} MB | resultados en {RESULTADOS}")
marcar_fase("6 modo LLM")

### 6.6 Conclusiones de la Fase 6

Los valores de esta corrida están en la tabla final de 6.5 (`results/resumen_corrida.csv`) y en `results/qa_llm_*.csv`. Comparación con la v5, la primera versión con el intérprete bilingüe:

| Medida | v5 | v6 |
|---|---|---|
| Clases escritas por el LLM exactamente como en el vocabulario | — | 55 de 60 |
| Interpretación correcta al primer intento (con la normalización del código) | 58 de 60 | 60 de 60 |
| JSON válido | 100 % | 100 % |
| Exactitud en preguntas cerradas | 0,725 | 0,775 (igual que el modo determinista) |
| Descripciones redactadas por el LLM, sin clases alucinadas | 20 de 20 | 20 de 20 |

**Lectura.**
- **El LLM interpreta casi todo bien por sí solo**, y el código cierra la brecha. En 5 de 60 consignas escribió la clase con otra forma (`wall_socket`, `baseball_cap`, `jeans`, `glasses`, `mango`) y la normalización la llevó al vocabulario, dejando registrado cómo (`traza_llm.csv`). Los dos errores de la v5 eran de este tipo y los corrigen el alias en inglés y el diccionario de sinónimos.
- **Esa mejora es optimista**, porque los dos arreglos se motivaron con estas mismas preguntas. La medida que no depende de ellos es la de clases exactas.
- **Con la interpretación correcta, los dos modos dan la misma respuesta:** las ejecuta el mismo motor. Por eso la exactitud del modo LLM queda igual a la determinista, y los errores restantes son de percepción (Fase 8).
- **El LLM nunca decide la respuesta.** Las redacciones que no conservan el valor del motor se reemplazan por la frase del motor, y las descripciones se aceptan sólo si nombran clases presentes con números sustentados en el resumen.
- **Casos extra:** ambiguas, atributos, fuera de alcance y fuera del vocabulario se manejan bien; comparación y proximidad heredan los errores de conteo de la percepción, igual que en el modo determinista.

**Límites.** El benchmark tiene sólo 60 consignas con redacciones muy regulares ("¿Cuántas instancias de la categoría...?"). Con preguntas más libres, la interpretación sería el eslabón más débil. El set extra de 5.6 lo prueba sólo en parte: agrega comparaciones, color y preguntas abiertas fuera de alcance, pero las demás también siguen plantillas.

## 7. Validación cruzada de la configuración del detector (agregado en la v6)

**Problema.** En 2.9 el umbral y la cantidad de clases por caja (k) se eligieron sobre las mismas 120 imágenes en las que después se mide el F1. Ese F1 es optimista: parte de la mejora puede ser ajuste al azar de estas imágenes.

**7.1 Validación cruzada.** Partimos las 120 imágenes en 5 grupos de 24 (al azar, con semilla fija). En cada vuelta elegimos k y umbral sobre 4 grupos (96 imágenes), con el mismo criterio de 2.9 (máximo F1, paso 0,01), y medimos el F1 en el grupo que quedó afuera. La media de esas 5 mediciones estima el F1 esperable en imágenes nuevas. También mostramos qué tan sensible es el F1 al umbral.

**7.2 Experimento E9: ¿un umbral por fuente?** Las cajas de las ventanas (E4) salen de recortes ampliados, así que su confianza podría no ser comparable con la de la imagen completa. Hipótesis: un umbral distinto para cada fuente sube el F1. Se valida con las mismas 5 particiones, comparando contra un umbral único elegido de la misma forma.

Todo corre en CPU sobre las detecciones crudas que ya están en memoria (`dets_multi`, la variante ganadora de 2.9). Las anotaciones se usan sólo para medir.

In [ ]:
# --- 7.1 Validación cruzada (5 particiones por imagen) de k y del umbral ---
ids_cv = np.array(sorted(img_meta))
np.random.default_rng(SEMILLA).shuffle(ids_cv)
PARTICIONES = [set(p) for p in np.array_split(ids_cv, 5)]
UMBRALES_CV = np.round(np.arange(0.10, 0.401, 0.01), 2)
nms_por_k = {k: aplicar_nms(dets_multi[dets_multi.rango < k]) for k in (1, 2, 3)}
F1_DEV_FINAL = float(tabla_det.F1.max())     # F1 de la configuración final, medido dentro de la muestra (2.9)

def elegir_config(imagenes):
    """(k, umbral, F1) de mayor F1 de cajas en esas imágenes: el criterio de 2.9 con paso 0,01."""
    return max(((k, float(u), evaluar_cajas(d[d.score >= u], imagenes=imagenes)["F1"])
                for k, d in nms_por_k.items() for u in UMBRALES_CV), key=lambda x: x[2])

t0 = time.time()
filas_cv = []
for i, prueba in enumerate(PARTICIONES, 1):
    entren = set(ids_cv) - prueba
    k, u, f1_entren = elegir_config(entren)
    m = evaluar_cajas(nms_por_k[k][nms_por_k[k].score >= u], imagenes=prueba)
    d_fin = nms_por_k[K_MAX]
    filas_cv.append(dict(particion=i, imagenes_prueba=len(prueba), k=k, umbral=u, F1_entrenamiento=f1_entren,
                         F1_prueba=m["F1"], P_prueba=m["P"], R_prueba=m["R"],
                         F1_prueba_config_final=evaluar_cajas(d_fin[d_fin.score >= UMBRAL], imagenes=prueba)["F1"]))
cv = pd.DataFrame(filas_cv)
cv.to_csv(RESULTADOS / "det_validacion_cruzada.csv", index=False)
print(f"Validación cruzada en {time.time() - t0:.0f} s\n")
print(cv.round(4).to_string(index=False))
print(f"\nF1 en las particiones de prueba: {cv.F1_prueba.mean():.4f} ± {cv.F1_prueba.std():.4f} (media ± desvío)")
print(f"F1 de la configuración final dentro de la muestra (2.9): {F1_DEV_FINAL:.4f} -> optimismo estimado {F1_DEV_FINAL - cv.F1_prueba.mean():+.4f}")
print("Umbrales elegidos:", sorted(cv.umbral.tolist()), "| k elegidos:", cv.k.value_counts().to_dict())

# Sensibilidad: F1 sobre las 120 imágenes según el umbral, con el k final
curva_u = pd.DataFrame([dict(umbral=float(u), **evaluar_cajas(nms_por_k[K_MAX][nms_por_k[K_MAX].score >= u])) for u in UMBRALES_CV])
curva_u.to_csv(RESULTADOS / "det_curva_umbral.csv", index=False)
cerca = curva_u[(curva_u.umbral >= UMBRAL - 0.03) & (curva_u.umbral <= UMBRAL + 0.03)]
print(f"F1 con umbrales a ±0,03 del final: entre {cerca.F1.min():.4f} y {cerca.F1.max():.4f}")
ax = curva_u.plot(x="umbral", y=["P", "R", "F1"], marker=".", color=[GRIS, "#c98a2b", AZUL])
ax.axvspan(cv.umbral.min() - 0.005, cv.umbral.max() + 0.005, color=AZUL, alpha=0.12, label="umbrales elegidos en la validación cruzada")
ax.axvline(UMBRAL, color=GRIS, ls="--", label=f"umbral final ({UMBRAL:.2f})")
ax.set(title=f"Sensibilidad del F1 de cajas al umbral (k = {K_MAX}, 120 imágenes)", xlabel="Umbral de confianza", ylabel="Valor")
ax.legend(); plt.show()

In [ ]:
# --- 7.2 E9: ¿conviene un umbral distinto para las cajas de las ventanas? ---
d_k = nms_por_k[K_MAX]
U_FUENTE = [0.15, 0.18, 0.20, 0.22, 0.25, 0.28, 0.30, 0.35]

def f1_por_fuente(u_img, u_vent, imagenes=None):
    """F1 de cajas con un umbral para las cajas de la imagen completa y otro para las de las ventanas (después del NMS)."""
    sel = d_k[np.where(d_k.fuente == "ventana", d_k.score >= u_vent, d_k.score >= u_img)]
    return evaluar_cajas(sel, imagenes=imagenes)["F1"]

print("Detecciones finales por fuente (k y umbral finales):", d_k[d_k.score >= UMBRAL].fuente.value_counts().to_dict())
if d_k.fuente.nunique() > 1:
    t0, filas_e9 = time.time(), []
    for i, prueba in enumerate(PARTICIONES, 1):
        entren = set(ids_cv) - prueba
        grilla = {(a, b): f1_por_fuente(a, b, entren) for a in U_FUENTE for b in U_FUENTE}
        a, b = max(grilla, key=grilla.get)                                     # dos umbrales
        u1 = max((u for u in U_FUENTE), key=lambda u: grilla[(u, u)])          # un umbral, elegido igual
        filas_e9.append(dict(particion=i, umbral_imagen=a, umbral_ventanas=b, F1_prueba_dos_umbrales=f1_por_fuente(a, b, prueba),
                             umbral_unico=u1, F1_prueba_umbral_unico=f1_por_fuente(u1, u1, prueba)))
    e9 = pd.DataFrame(filas_e9)
    e9.to_csv(RESULTADOS / "det_umbral_por_fuente.csv", index=False)
    print(f"E9 en {time.time() - t0:.0f} s\n")
    print(e9.round(4).to_string(index=False))
    dif = e9.F1_prueba_dos_umbrales - e9.F1_prueba_umbral_unico
    print(f"\nDos umbrales: {e9.F1_prueba_dos_umbrales.mean():.4f} ± {e9.F1_prueba_dos_umbrales.std():.4f} | "
          f"umbral único: {e9.F1_prueba_umbral_unico.mean():.4f} ± {e9.F1_prueba_umbral_unico.std():.4f} | "
          f"diferencia media {dif.mean():+.4f} (mejora en {int((dif > 0).sum())} de 5 particiones, empeora en {int((dif < 0).sum())})")
else:
    print("La configuración final no usa ventanas: E9 no aplica.")
marcar_fase("7 validación cruzada")

### 7.3 Conclusiones de la Fase 7

| Medida | Valor |
|---|---|
| F1 de cajas en las particiones de prueba (media ± desvío) | **0,673 ± 0,023** |
| F1 de la configuración final dentro de la muestra (2.9) | 0,679 → optimismo estimado de 0,006 |
| Configuración elegida en cada partición | umbral 0,25 en 3 particiones y 0,22 en 2; k = 3 en 4 de 5 |
| F1 con umbrales a ±0,03 del final (0,22 a 0,28) | entre 0,671 y 0,679 |
| E9 · dos umbrales (imagen / ventanas) contra uno, ambos elegidos por validación cruzada | 0,674 contra 0,674: +0,001, mejora en 2 particiones y empeora en 1 |

**Lectura.**
- **El F1 de desarrollo es una estimación razonable.** Elegir el umbral sobre las mismas imágenes infla el F1 en sólo 0,006. El requisito de promoción (≥ 0,50) se cumple con margen también fuera de la muestra.
- **La configuración es estable:** el óptimo casi no se mueve entre particiones y el F1 cambia menos de 0,01 en un rango de ±0,03 alrededor del umbral.
- **El ruido entre imágenes domina.** El desvío entre particiones (0,023) es cuatro veces el optimismo. Por eso las diferencias de menos de ~0,01 entre variantes de 2.9 (sinónimos, ventanas, ensamble) no son concluyentes sólo con desarrollo. La configuración final se apoya además en el leaderboard, que subió de 0,6656 a 0,7022 al adoptarla (v3 → v4).
- **E9 no se adopta.** Un umbral por fuente agrega un parámetro sin una mejora que se sostenga fuera de la muestra: se mantiene el umbral único.

## 8. Trazas y propagación de errores (agregado en la v6)

La traza tiene tres niveles, todos con el mismo `RUN_ID`:

| Archivo | Una fila por | Qué registra |
|---|---|---|
| `traza_envio.csv` | imagen × módulo | parámetros, inicio, duración, estado, cantidad de salidas y error (detector, NMS + umbral, segmentador, propiedades, relaciones, atributos, validación) |
| `traza_objetos.csv` | objeto de cada SceneJSON | **linaje**: confianza, rango de la clase en su caja (k), fuente (imagen completa o ventana), caja del detector, segmentador, píxeles de la máscara, si se usó la caja de respaldo |
| `traza_qa.csv`, `traza_llm.csv` | pregunta | consulta ejecutada; en el modo LLM, la salida cruda del intérprete, los reintentos y su motivo, la normalización de clases y la redacción usada |

Con eso respondemos dos preguntas:

1. **¿En qué etapa se pierde cada objeto real?** (8.1) Para los 1790 objetos de desarrollo seguimos la cadena *detector → umbral → NMS → emparejamiento → SAM2* y asignamos cada objeto a la **primera** etapa donde se pierde:
   - **no localizado:** ninguna caja cruda (de ninguna clase, confianza ≥ 0,05) tiene IoU ≥ 0,5 con él;
   - **localizado con otra clase:** hay cajas que lo cubren, pero ninguna de su clase entre las k clases candidatas;
   - **debajo del umbral:** hay una caja de su clase, pero con confianza menor que el umbral final;
   - **eliminado por NMS:** había una caja de su clase sobre el umbral y el NMS la borró;
   - **caja usada por otro objeto:** la caja sobrevivió, pero el emparejamiento uno a uno la asignó a otro objeto real;
   - **máscara mala (SAM2):** la caja es correcta, pero la máscara tiene IoU < 0,5 con la real;
   - **OK:** caja y máscara correctas.
2. **¿De dónde viene cada respuesta incorrecta?** (8.2) Cada error de QA se lleva hasta la etapa que lo causó: interpretación del LLM, una regla derivada (zona, tamaño, relación) o la percepción, y en ese caso hasta la etapa de 8.1 de los objetos involucrados. Si los objetos que faltan se perdieron en etapas distintas, se informa la que explica más objetos y, si empatan, la más temprana de la pipeline; la columna `detalle` muestra todas.

En 8.3 reconstruimos una respuesta completa leyendo sólo las trazas, y en 8.4 usamos las trazas para evaluar las etapas (tiempos, fallas, reintentos y normalizaciones). Las anotaciones se usan sólo para medir.

In [ ]:
# --- 8.1 ¿En qué etapa se pierde cada objeto real? (desarrollo, 120 imágenes) ---
def iou_matriz(A, B):
    """IoU entre dos listas de cajas [x, y, w, h] (vectorizado)."""
    A, B = np.asarray(A, float).reshape(-1, 4), np.asarray(B, float).reshape(-1, 4)
    iw = np.clip(np.minimum((A[:, 0] + A[:, 2])[:, None], (B[:, 0] + B[:, 2])[None]) - np.maximum(A[:, 0][:, None], B[:, 0][None]), 0, None)
    ih = np.clip(np.minimum((A[:, 1] + A[:, 3])[:, None], (B[:, 1] + B[:, 3])[None]) - np.maximum(A[:, 1][:, None], B[:, 1][None]), 0, None)
    inter = iw * ih
    return inter / (A[:, 2:].prod(1)[:, None] + B[:, 2:].prod(1)[None] - inter + 1e-9)

ETAPAS = ["OK", "máscara mala (SAM2)", "caja usada por otro objeto", "eliminado por NMS", "debajo del umbral",
          "localizado con otra clase", "no localizado"]
crudas_k = dets_multi[dets_multi.rango < K_MAX]
crudas_por_img = {i: d for i, d in crudas_k.groupby("image_id")}
final_por_img = {i: d for i, d in dets_final.groupby("image_id")}
vacio = dets_final.iloc[:0]

t0, filas_prop = time.time(), []
for img_id, r in img_meta.items():
    anns = anns_por_img[img_id]
    fin, cru = final_por_img.get(img_id, vacio), crudas_por_img.get(img_id, crudas_k.iloc[:0])
    # Emparejamiento uno a uno por clase, igual que evaluar_cajas (mayor confianza primero)
    asignada = {}
    for c in {a["category_id"] for a in anns}:
        gts = [a for a in anns if a["category_id"] == c]
        usado = [False] * len(gts)
        for idx, b in fin[fin.category_id == c].sort_values("score", ascending=False).bbox.items():
            ious = [-1 if usado[j] else iou_xywh(b, g["bbox"]) for j, g in enumerate(gts)]
            j = int(np.argmax(ious))
            if ious[j] >= 0.5:
                usado[j] = True
                asignada[gts[j]["id"]] = idx
    for a in anns:
        c, fila = a["category_id"], dict(image_id=img_id, ann_id=a["id"], clase=id2name[a["category_id"]], area=a["area"])
        if a["id"] in asignada:
            d = dets_final.loc[asignada[a["id"]]]
            gt_m = mask_utils.merge(mask_utils.frPyObjects(a["segmentation"], r.height, r.width))
            iou_m = float(mask_utils.iou([a_rle_coco(d.rle_sam)], [gt_m], [0])[0][0])
            filas_prop.append(dict(fila, etapa="OK" if iou_m >= 0.5 else "máscara mala (SAM2)", iou_mascara=round(iou_m, 3),
                                   score=round(float(d.score), 4), fuente=d.get("fuente", "imagen")))
            continue
        cubre = cru[iou_matriz([a["bbox"]], np.stack(cru.bbox.values))[0] >= 0.5] if len(cru) else cru
        de_su_clase = cubre[cubre.category_id == c]
        fin_c = fin[fin.category_id == c]
        sobrevive = len(fin_c) and (iou_matriz([a["bbox"]], np.stack(fin_c.bbox.values))[0] >= 0.5).any()
        if not len(cubre):
            etapa = "no localizado"
        elif not len(de_su_clase):
            etapa = "localizado con otra clase"
        elif de_su_clase.score.max() < UMBRAL:
            etapa = "debajo del umbral"
        elif not sobrevive:
            etapa = "eliminado por NMS"
        else:
            etapa = "caja usada por otro objeto"
        filas_prop.append(dict(fila, etapa=etapa, iou_mascara=np.nan,
                               score=round(float(de_su_clase.score.max()), 4) if len(de_su_clase) else np.nan, fuente=None))
prop = pd.DataFrame(filas_prop)
prop["tamaño"] = pd.cut(prop.area, [0, 32**2, 96**2, np.inf], labels=["chico (<32²)", "mediano", "grande (≥96²)"])
prop.to_csv(RESULTADOS / "propagacion_objetos.csv", index=False)
print(f"{len(prop)} objetos reales clasificados en {time.time() - t0:.0f} s")

tabla_prop = pd.crosstab(prop.etapa, prop["tamaño"], margins=True, margins_name="total").reindex(ETAPAS + ["total"])
tabla_prop["%"] = (100 * tabla_prop["total"] / len(prop)).round(1)
tabla_prop.to_csv(RESULTADOS / "propagacion_resumen.csv")
print(tabla_prop.to_string())
cajas_ok = prop.etapa.isin(["OK", "máscara mala (SAM2)"]).sum()
print(f"\nControl: objetos con caja correcta = {cajas_ok} (TP de cajas de 2.9 con la misma configuración: {int(tabla_det.loc[tabla_det.F1.idxmax(), 'TP'])})")
bajo = prop[prop.etapa == "debajo del umbral"].score
print(f"'Debajo del umbral': confianza máxima de su clase, mediana {bajo.median():.3f} (cuartiles {bajo.quantile(0.25):.3f} - {bajo.quantile(0.75):.3f})")
print(f"IoU de máscara de los objetos con caja correcta: media {prop.iou_mascara.mean():.3f} (oráculo con cajas reales, 3.3b: {oraculo.iou.mean():.3f})")
print("\nClases que más objetos pierden, por etapa:")
print(prop[prop.etapa != "OK"].groupby(["clase", "etapa"]).size().sort_values(ascending=False).head(10).to_string())
recall_clase = prop.etapa.isin(["OK", "máscara mala (SAM2)"]).groupby(prop.clase).mean()
print("Recall de cajas de las clases señaladas en 2.6 (en la línea base, person tenía 0,03 y book 26 de 43 sin caja):",
      {c: round(float(recall_clase[c]), 2) for c in ("person", "book", "shoe") if c in recall_clase})

pct = pd.crosstab(prop["tamaño"], prop.etapa, normalize="index").reindex(columns=ETAPAS) * 100
colores_etapa = [AZUL, "#7fa7c9", "#c9c9c9", "#a0a0a0", "#c98a2b", "#d9b26f", "#b03a2e"]
ax = pct.plot.barh(stacked=True, color=colores_etapa, figsize=(9, 3.2))
ax.set(title="¿En qué etapa se pierde cada objeto real? (por tamaño)", xlabel="% de los objetos reales", ylabel="")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8); plt.show()

In [ ]:
# --- 8.2 ¿De dónde viene cada respuesta incorrecta? (benchmark, los dos modos) ---
lin_idx = linaje.set_index(["image_id", "obj_id"])
prop_idx = prop.set_index("ann_id")
preg = qs.set_index("question_id")

def detalle_percepcion(image_key, clases):
    """Para las clases de la pregunta compara el SceneJSON con las anotaciones: objetos reales que faltan (con la etapa
    de 8.1 donde se perdieron) y objetos predichos que no corresponden a ninguno real (con su linaje)."""
    esc, img_id = escena_por_id[image_key], stem2id[image_key]
    perdidos, falsos = Counter(), []
    for clase in clases:
        gts = [a for a in anns_por_img[img_id] if a["category_id"] == name2id[clase]]
        usado = [False] * len(gts)
        for o in instancias(esc, clase):
            b = [o["bbox"][0], o["bbox"][1], o["bbox"][2] - o["bbox"][0], o["bbox"][3] - o["bbox"][1]]
            ious = [-1 if usado[j] else iou_xywh(b, g["bbox"]) for j, g in enumerate(gts)]
            j = int(np.argmax(ious)) if gts else -1
            if j >= 0 and ious[j] >= 0.5:
                usado[j] = True
            else:
                l = lin_idx.loc[(image_key, o["id"])]
                falsos.append(f"{clase} {o['id']} ({l.fuente}, confianza {l.score:.2f})")
        for g, u in zip(gts, usado):
            if not u:
                perdidos[(clase, prop_idx.loc[g["id"], "etapa"])] += 1
    return perdidos, falsos

def etapa_del_error(p, fila_llm=None):
    """Etapa que explica una respuesta incorrecta, en el orden de la pipeline inversa: interpretación -> percepción -> regla."""
    if fila_llm is not None and not fila_llm.interpretacion_ok:
        return "interpretación (LLM)", f"consulta del LLM: {fila_llm.consulta_llm}"
    clases = list(p.class_en)
    perdidos, falsos = detalle_percepcion(p.image_key, clases)
    texto = "; ".join([f"{n} {c} perdido(s): {e}" for (c, e), n in perdidos.items()] + [f"sobra {f}" for f in falsos])
    if perdidos:   # la etapa que explica más objetos perdidos; si empatan, la más temprana de la pipeline
        por_etapa = Counter()
        for (_, e), n in perdidos.items():
            por_etapa[e] += n
        temprana = {e: i for i, e in enumerate(reversed(ETAPAS))}
        return f"percepción: {max(por_etapa, key=lambda e: (por_etapa[e], -temprana[e]))}", texto
    if falsos:
        return "percepción: falso positivo del detector", texto
    etapas_gt = {prop_idx.loc[a["id"], "etapa"] for c in clases for a in anns_por_img[stem2id[p.image_key]]
                 if a["category_id"] == name2id[c]}
    if "máscara mala (SAM2)" in etapas_gt:
        return "segmentación: máscara mala", "la instancia es la correcta, pero su máscara (y su centroide) no"
    return "regla derivada (zona, tamaño o relación)", "misma instancia y buena máscara; el valor cae del otro lado de un límite de la regla"

filas_eq = []
for f in ev_det[~ev_det.correcta].itertuples():
    etapa, det = etapa_del_error(preg.loc[f.question_id])
    filas_eq.append(dict(modo="determinista", question_id=f.question_id, type=f.type, clases=f.clases, pred=f.pred, ref=f.ref,
                         etapa=etapa, detalle=det))
cerr_llm = ev_llm[(ev_llm.type != "description") & ~ev_llm.correcta_llm.astype(bool)]
for f in cerr_llm.itertuples():
    p = preg.loc[f.question_id]
    etapa, det = etapa_del_error(p, f)
    filas_eq.append(dict(modo="LLM", question_id=f.question_id, type=f.type, clases=", ".join(p.class_en), pred=f.respuesta_llm,
                         ref=f.ref, etapa=etapa, detalle=det))
prop_qa = pd.DataFrame(filas_eq)
prop_qa.to_csv(RESULTADOS / "propagacion_qa.csv", index=False)
print("Respuestas incorrectas por etapa de origen (preguntas cerradas):")
print(pd.crosstab(prop_qa.etapa, prop_qa.modo, margins=True, margins_name="total").to_string())
print()
with pd.option_context("display.max_colwidth", 110):
    print(prop_qa[["modo", "question_id", "type", "pred", "ref", "etapa", "detalle"]].to_string(index=False))

In [ ]:
# --- 8.3 Reconstrucción de una respuesta leyendo sólo las trazas y el SceneJSON ---
traza_llm_df = pd.DataFrame(traza_llm)

def reconstruir(question_id):
    p = preg.loc[question_id]
    k, esc = p.image_key, escena_por_id[p.image_key]
    print(f"■ {question_id} ({p.type}) sobre {k}: {p.question}")
    print(f"  1. Corrida: run_id {esc['trace']['run_id']} | hash de la configuración de percepción {esc['trace']['config_percepcion']}")
    tr = traza[traza.image_id == k]
    print("  2. Percepción (traza_envio):", " → ".join(f"{m} {n} ({ms:.0f} ms)" for m, n, ms in zip(tr.modulo, tr.n_salidas, tr.duracion_ms)))
    det = next(x for x in esc["questions"] if x["question_id"] == question_id)
    print(f"  3. Determinista (traza_qa): consulta {json.dumps(det['query'], ensure_ascii=False)} → {det['normalized_answer']}"
          f" [{det['status']}], evidencia {det['evidence']}, confianza {det['confidence']}")
    llm = next(x for x in esc["questions_llm"] if x["question_id"] == question_id)
    t = traza_llm_df.set_index("question_id").loc[question_id]
    print(f"  4. LLM (traza_llm): salida cruda {t.salida_interprete_1!r}")
    if t.intentos > 1:
        print(f"     reintento por {t.motivo_reintento}: {t.salida_interprete_2!r}")
    print(f"     normalización {t.normalizacion} → consulta {t.consulta} → {llm['normalized_answer']} [{llm['status']}]")
    print(f"     redacción ({t.redaccion}): {llm['answer']}")
    for oid in dict.fromkeys(det["evidence"] + llm["evidence"]):
        l = lin_idx.loc[(k, oid)]
        print(f"  5. Evidencia {oid} (traza_objetos): {l.clase}, confianza {l.score:.3f}, clase n.º {l.rango_clase + 1} de su caja, "
              f"fuente {l.fuente}, caja del detector {l.bbox_detector}, máscara de {l.mascara_px} px con {l.segmentador.split('/')[-1]}")
    perdidos, falsos = detalle_percepcion(k, list(p.class_en))
    ref = ejecutar(query_determinista(p.to_dict()), escenas_gt[k])["normalized_answer"]
    print(f"  6. Contra las anotaciones (sólo evaluación): referencia {ref} | objetos reales perdidos: "
          f"{dict((f'{c}: {e}', n) for (c, e), n in perdidos.items()) or 'ninguno'} | predicciones sin objeto real: {falsos or 'ninguna'}\n")

# Un error de conteo del modo determinista y, del modo LLM, una consulta con reintento o donde el código tradujo la clase
# (alias en inglés o diccionario español); si no hay ninguna, la primera de relación
ej_det = next((q for q in prop_qa[prop_qa.modo == "determinista"].question_id if preg.loc[q].type == "count"), prop_qa.question_id.iloc[0])
con_reintento = ev_llm[(ev_llm.intentos > 1) & (ev_llm.type != "description")].question_id.tolist()
traducidas = [q for q, n in zip(traza_llm_df.question_id, traza_llm_df.normalizacion)
              if {x["como"] for x in json.loads(n)} & {"alias_en", "diccionario_es"}]
ej_llm = (con_reintento + traducidas + qs[qs.type == "relation"].question_id.tolist())[0]
for q in dict.fromkeys([ej_det, ej_llm]):
    reconstruir(q)

In [ ]:
# --- 8.4 Evaluación de las etapas a partir de las trazas ---
# Percepción (traza_envio.csv): costo y fallas de cada módulo en las 80 imágenes del envío
modulos = traza.groupby("modulo", sort=False).agg(ejecuciones=("ok", "size"), fallas=("ok", lambda s: int((~s).sum())),
                                                  ms_medio=("duracion_ms", "mean"), ms_p95=("duracion_ms", lambda s: s.quantile(0.95)),
                                                  salidas_medias=("n_salidas", "mean"))
modulos["% del tiempo"] = 100 * traza.groupby("modulo", sort=False).duracion_ms.sum() / traza.duracion_ms.sum()
modulos.to_csv(RESULTADOS / "traza_modulos.csv")
print("Percepción por módulo (80 imágenes):")
print(modulos.round(1).to_string())
n_crudas = int(traza[traza.modulo == "detector"].n_salidas.sum())
n_finales = int(traza[traza.modulo == "nms_umbral"].n_salidas.sum())
print(f"\nEmbudo del detector: {n_crudas} detecciones crudas (≥ 0,05) → {n_finales} tras NMS y umbral ({n_finales / n_crudas:.1%})"
      f" → {len(linaje)} objetos en los SceneJSON | escenas vacías por falla: {sum(len(e['objects']) == 0 for e in escenas)}")

# Linaje (traza_objetos.csv): de dónde salen los objetos que llegan al SceneJSON
print("Objetos por fuente:", linaje.fuente.value_counts(normalize=True).round(3).to_dict(),
      "| por rango de la clase en su caja:", linaje.rango_clase.value_counts().sort_index().to_dict(),
      "| máscaras de respaldo (caja):", int(linaje.mascara_respaldo.sum()))
print("Fracción de la caja cubierta por la máscara: mediana", round(float(linaje.relleno_caja.median()), 3),
      "| objetos con máscara < 20 % de su caja:", int((linaje.relleno_caja < 0.2).sum()))

# Modo LLM (traza_llm.csv): reintentos, normalización de clases y redacción
norm = Counter(n["como"] for x in traza_llm_df.normalizacion for n in json.loads(x))
print("\nIntérprete LLM | reintentos:", traza_llm_df.motivo_reintento.replace("", "ninguno").value_counts().to_dict(),
      "| normalización de clases:", dict(norm), f"| {traza_llm_df.ms_interprete.mean():.0f} ms por pregunta")
print("Redacción usada:", traza_llm_df.redaccion.value_counts().to_dict())
marcar_fase("8 trazas y propagación")

### 8.5 Conclusiones de la Fase 8

**Dónde se pierden los objetos reales (8.1, 1790 objetos de desarrollo).**

| Etapa | Objetos | % | De ellos, chicos |
|---|---|---|---|
| OK (caja y máscara correctas) | 1073 | 59,9 | 459 |
| Debajo del umbral | 453 | 25,3 | 299 |
| No localizado | 120 | 6,7 | 111 |
| Máscara mala (SAM2) | 83 | 4,6 | 64 |
| Localizado con otra clase | 31 | 1,7 | 14 |
| Caja usada por otro objeto | 16 | 0,9 | 10 |
| Eliminado por NMS | 14 | 0,8 | 10 |

- **La primera causa de pérdida es la confianza, no la localización.** Uno de cada cuatro objetos reales tiene una caja de su clase que lo cubre, pero con confianza menor que 0,25 (mediana 0,17). Bajar el umbral no sirve (7.1): entran más falsos positivos que objetos recuperados. Lo que falta es un puntaje mejor calibrado por clase; `car`, `book`, `shoe`, `chair` y `wheel` son las que más pierden así.
- **Los objetos chicos concentran las pérdidas que el umbral no explica:** 111 de los 120 no localizados y 64 de las 83 máscaras malas.
- **SAM2 casi no agrega error:** el 93 % de los objetos con caja correcta también tiene máscara correcta (IoU medio 0,80). El control de 8.1 cierra con la métrica: 1156 objetos con caja correcta, los mismos TP de 2.9.
- **NMS y emparejamiento son marginales** (30 objetos entre ambos), como anticipaba el EDA (1.6).

**De dónde vienen las respuestas incorrectas (8.2).** Los 9 errores del modo determinista vienen de la percepción, y en el modo LLM son los mismos 9, porque con la interpretación correcta el motor da la misma respuesta. Ninguno es del motor ni de una regla derivada:
- en 6 la causa principal es que objetos de la clase preguntada quedaron **debajo del umbral** (`doorknob`, `shoe`, `glove`, `strap`, `banner`, `elephant`);
- 1 es un **falso positivo**: 3 servilletas que la imagen tiene verificadas como ausentes, todas de las ventanas y con confianza apenas sobre el umbral (0,26 a 0,29);
- 2 conteos pierden objetos en dos etapas distintas (`trousers`: otra clase y umbral; `fork`: no localizado y NMS).

De los 11 objetos reales que faltan en esas respuestas, 8 quedaron debajo del umbral: la misma conclusión que a nivel de objetos. Para el QA, la palanca es la misma que para el F1.

**Las trazas alcanzan para reconstruir un resultado (8.3).** Desde el `run_id` de un SceneJSON se llega a la configuración, a los módulos de percepción de esa imagen, a la consulta ejecutada, a la salida cruda del LLM con su normalización (por ejemplo, `mango` → `handle` por el diccionario español) y al linaje de cada objeto de la evidencia: confianza, rango de clase, fuente y máscara. Las anotaciones sólo aparecen en el último paso, para explicar el error.

**Evaluación de las etapas con las trazas (8.4).** Ningún módulo falló en las 80 imágenes. El detector es el módulo más caro (≈ 2/3 del tiempo, por las 5 pasadas de OWLv2) y SAM2 le sigue (≈ 1/4). Del lado del intérprete, la normalización resolvió 6 clases que el LLM no escribió exactamente como en el vocabulario (3 por coincidencia aproximada, 1 por singular, 1 por alias en inglés y 1 por el diccionario español), sin necesidad de reintentos.

## 9. Manifiesto de la corrida y resumen de resultados (agregado en la v6)

Cierre de la traza. `manifiesto_corrida.json` registra:
- `run_id` y `config_hash`: el hash de **toda** la configuración (percepción, LLM, prompts y semilla). El SceneJSON de cada fila trae el `run_id` y el hash de la parte de percepción, así que cada fila del envío se puede atar a esta corrida;
- las versiones del entorno y el tiempo de cada fase;
- la lista de **artefactos** generados (`submission.csv`, trazas, `results/*.csv`, detecciones crudas) con su tamaño y su SHA-256.

`results/resumen_final.csv` junta en una tabla las métricas principales de cada etapa, con el archivo de donde sale cada número. Es la tabla que cita el reporte (`REPORTE.md`).

In [ ]:
# --- 9.1 Resumen de resultados: cada número sale de una variable de esta corrida y queda atado a su archivo ---
fila_det = tabla_det.loc[tabla_det.F1.idxmax()]
seg_fin = tabla_seg.loc[tabla_seg.segmentador == SAM_ID].iloc[0]
cerradas_llm = ev_llm[ev_llm.type != "description"]
aceptadas = ev_desc[ev_desc.redaccion == "llm"]
R = lambda x: round(float(x), 4)
resumen_final = pd.DataFrame([
    ("detección", "F1 de cajas en desarrollo (configuración final)", R(fila_det.F1), "det_experimentos.csv"),
    ("detección", "precisión / recall", f"{R(fila_det.P)} / {R(fila_det.R)}", "det_experimentos.csv"),
    ("detección", "cobertura de evaluación (LVIS)", R(fila_det.cobertura), "det_experimentos.csv"),
    ("detección", "F1 de cajas en validación cruzada (media)", R(cv.F1_prueba.mean()), "det_validacion_cruzada.csv"),
    ("detección", "F1 de cajas en validación cruzada (desvío)", R(cv.F1_prueba.std()), "det_validacion_cruzada.csv"),
    ("segmentación", "F1 de máscaras en desarrollo", R(seg_fin.F1), "seg_experimentos.csv"),
    ("segmentación", "IoU medio de SAM2 con cajas reales (oráculo)", R(oraculo.iou.mean()), "seg_oraculo.csv"),
    ("propagación", "objetos reales con caja y máscara correctas", R((prop.etapa == "OK").mean()), "propagacion_objetos.csv"),
    ("propagación", "objetos reales perdidos por el umbral", R((prop.etapa == "debajo del umbral").mean()), "propagacion_objetos.csv"),
    ("propagación", "objetos reales no localizados", R((prop.etapa == "no localizado").mean()), "propagacion_objetos.csv"),
    ("QA determinista", "exactitud en preguntas cerradas", R(ev_det.correcta.mean()), "qa_det_exactitud.csv"),
    ("QA determinista", "fidelidad al SceneJSON", R(fid.fiel.mean()), "qa_det_fidelidad.csv"),
    ("QA determinista", "casos extra bien manejados", R(ev_extra.correcto.mean()), "qa_extra_manejo.csv"),
    ("QA LLM", "clases escritas exactamente como en el vocabulario (sin normalizar)", R(ev_llm.clases_exactas_1.mean()), "qa_llm_respuestas.csv"),
    ("QA LLM", "interpretación correcta al primer intento", R(ev_llm.interpretacion_ok_1.mean()), "qa_llm_respuestas.csv"),
    ("QA LLM", "interpretación correcta (final)", R(ev_llm.interpretacion_ok.mean()), "qa_llm_respuestas.csv"),
    ("QA LLM", "JSON válido (final)", R(ev_llm.json_valido_final.mean()), "qa_llm_respuestas.csv"),
    ("QA LLM", "exactitud en preguntas cerradas", R(cerradas_llm.correcta_llm.astype(bool).mean()), "qa_llm_respuestas.csv"),
    ("QA LLM", "misma respuesta que el modo determinista", R(cerradas_llm.igual_det.astype(bool).mean()), "qa_llm_respuestas.csv"),
    ("QA LLM", "redacciones aceptadas por el control (cerradas)", R((cerradas_llm.redaccion == "llm").mean()), "qa_llm_respuestas.csv"),
    ("QA LLM", "descripciones redactadas por el LLM", R((ev_desc.redaccion == "llm").mean()), "qa_llm_descripciones.csv"),
    ("QA LLM", "clases alucinadas en las descripciones aceptadas", int(aceptadas.alucinadas.sum()), "qa_llm_descripciones.csv"),
    ("QA LLM", "cobertura de las 3 clases principales en la descripción", R(ev_desc.cobertura_top3.mean()), "qa_llm_descripciones.csv"),
    ("QA LLM", "casos extra bien manejados", R(ev_extra_llm.correcto.mean()), "qa_llm_extra.csv"),
    ("envío", "filas válidas", f"{sum(not problemas_de_fila(e) for e in escenas)} de {len(escenas)}", "submission.csv"),
], columns=["etapa", "metrica", "valor", "archivo"])
resumen_final.to_csv(RESULTADOS / "resumen_final.csv", index=False)
print(resumen_final.to_string(index=False))

# --- 9.2 Manifiesto: configuración completa, versiones, tiempos y artefactos ---
marcar_fase("9 resumen")
assert all(e.get("trace", {}).get("run_id") == RUN_ID for e in escenas), "hay filas del envío sin el RUN_ID de esta corrida"
CONFIG_COMPLETA = dict(semilla=SEMILLA, percepcion=CONFIG_PERCEPCION, llm=LLM_CONFIG,
                       prompts={n: huella(v) for n, v in dict(interprete=MENSAJES_INTERPRETE, verbalizar=MENSAJES_VERBALIZAR,
                                                              describir=[PROMPT_DESCRIBIR, EJEMPLO_DESCRIBIR]).items()},
                       qa=dict(conf_ausencia=CONF_AUSENCIA, alias_en=len(ALIAS_EN), reintentos_max=1))

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

artefactos = [dict(ruta=p.relative_to(SALIDA).as_posix(), bytes=p.stat().st_size, sha256=sha256(p))
              for p in sorted(SALIDA.rglob("*")) if p.is_file() and p.name != "manifiesto_corrida.json" and ".ipynb" not in p.name]
manifiesto = dict(run_id=RUN_ID, config_hash=huella(CONFIG_COMPLETA), config_percepcion_hash=TRAZA_ESCENA["config_percepcion"],
                  config=CONFIG_COMPLETA, versiones=VERSIONES, tiempos_fase_s=TIEMPOS_FASE,
                  duracion_total_min=round((time.time() - T_INICIO) / 60, 1),
                  datos=dict(competencia=ROOT.name, schema_sha256=sha256(ROOT / "scene_schema.json"),
                             imagenes_desarrollo=len(img_meta), filas_envio=len(escenas), consignas=len(qs)),
                  artefactos=artefactos)
(SALIDA / "manifiesto_corrida.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"\nmanifiesto_corrida.json | run_id {RUN_ID} | config_hash {manifiesto['config_hash']} | {len(artefactos)} artefactos "
      f"| {manifiesto['duracion_total_min']} min en total")
print("Tiempos por fase (s desde el inicio):", TIEMPOS_FASE)
print(pd.DataFrame(artefactos).assign(MB=lambda d: (d.bytes / 1e6).round(2))[["ruta", "MB", "sha256"]]
      .assign(sha256=lambda d: d.sha256.str.slice(0, 12)).to_string(index=False))

### 9.3 Dónde se cumple cada criterio de la consigna

| Criterio | Dónde está |
|---|---|
| EDA que fundamenta decisiones | 1.1 a 1.10 (tabla observación → decisión) |
| Detector open-vocabulary justificado (modelo, vocabulario, umbral, resolución, supresión de duplicados) | 2.2 a 2.9, con validación cruzada en la Fase 7 |
| F1 de detección ≥ 0,50 en desarrollo (IoU 0,5, reglas LVIS) | 2.9 (0,679) y 7.1 (0,673 ± 0,023 fuera de la muestra) |
| SAM2 con las cajas como prompt, RLE column-major | 3.1 a 3.3b |
| Propiedades, atributos y relaciones derivados de las máscaras | 3.4 y 3.6 |
| SceneJSON válido y consistente | 4.1, 4.2, 5.7 y 6.5 (schema, RLE e ids validados antes de escribir) |
| `questions` en modo determinista, con fidelidad medida | Fase 5 (5.5: exactitud y fidelidad) |
| `questions_llm`: el LLM sólo recibe el texto y arma la consulta; el motor determinista la ejecuta | Fase 6 |
| Ubicación, comparación, atributos y relaciones en los dos modos | 5.3, 5.6 y 6.4 |
| Manejo de ambiguas, no soportadas y sin información | 5.3, 5.6 y 6.4 (casos extra) |
| Traza que reconstruye la ejecución (parámetros, artefactos, errores) | 0, 4.1, 4.2, 8.3 y 9.2 (`manifiesto_corrida.json`) |
| Trazas usadas para evaluar las etapas y la propagación de errores | 8.1, 8.2 y 8.4 |
| Mejora contra una línea base con experimentos controlados | 2.7, 2.9, 3.3, 3.3b y 7.2 (E1 a E9, incluidos los que no mejoraron) |
| Casos correctos, ambiguos y fallidos | 5.5, 5.6, 6.4, 8.2 y 8.3 |
| Resultados en formatos auditables | `results/*.csv`, trazas y manifiesto; resumen en `results/resumen_final.csv` |